
# ==========Section_2: Definitions================#

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Callable, List, Tuple, Optional




@dataclass
class SuperquadricFieldParameters:
    """
    Superquadric implicit field parameters
    
    F_3d(x,y,z) = (f(x,y))^(eps2/eps1) + |z/a3|^(2/eps1) - 1
    f(x,y)      = |x/a1|^(2/eps2) + |y/a2|^(2/eps2)
    """
    a1: float
    a2: float
    a3: float
    eps1: float
    eps2: float
    sdf_eps: float = 1e-8  # Denominator stability term for normalized signed field

@dataclass
class StiffnessParameters:
    """
   Smooth stiffness function parameters
    
    k(d) = k_min + ((1 - tanh(d/d0))/2) * k_max
    """
    k_min: float
    k_max: float
    d0: float




## ==========Geometry================


In [8]:
import numpy as np
from dataclasses import dataclass
from typing import Tuple


# =========================
# Basic data structure
# =========================

@dataclass
class Pose:
    """
    Pose X = (R, p), with
        R in SO(3), shape (3, 3)
        p in R^3,  shape (3,)
    """
    R: np.ndarray
    p: np.ndarray

    def __post_init__(self):
        self.R = ensure_matrix(self.R, (3, 3))
        self.p = ensure_vector(self.p, 3)


# =========================
# Shape / type utilities
# =========================

def ensure_vector(x: np.ndarray, dim: int) -> np.ndarray:
    """
    Ensure x is a 1D vector of length `dim`.
    """
    x = np.asarray(x, dtype=float).reshape(-1)
    if x.shape[0] != dim:
        raise ValueError(f"Expected vector of length {dim}, got shape {x.shape}")
    return x


def ensure_matrix(x: np.ndarray, shape: Tuple[int, int]) -> np.ndarray:
    """
    Ensure x is a matrix of the specified shape.
    """
    x = np.asarray(x, dtype=float)
    if x.shape != shape:
        raise ValueError(f"Expected matrix of shape {shape}, got shape {x.shape}")
    return x


# =========================
# so(3) <-> R^3
# =========================

def hat(v: np.ndarray) -> np.ndarray:
    """
    hat : R^3 -> so(3)

    For v = [v1, v2, v3]^T,
    hat(v) = [[  0, -v3,  v2],
              [ v3,   0, -v1],
              [-v2, v1,   0 ]]
    """
    v = ensure_vector(v, 3)
    return np.array([
        [0.0,   -v[2],  v[1]],
        [v[2],   0.0,  -v[0]],
        [-v[1],  v[0],  0.0]
    ])


def vee(M: np.ndarray) -> np.ndarray:
    """
    vee : so(3) -> R^3
    """
    M = ensure_matrix(M, (3, 3))
    return np.array([
        M[2, 1],
        M[0, 2],
        M[1, 0]
    ])


# =========================
# SO(3) maps
# =========================

def project_to_so3(M: np.ndarray) -> np.ndarray:
    """
    Project a 3x3 matrix to SO(3) via SVD.
    """
    M = ensure_matrix(M, (3, 3))
    U, _, Vt = np.linalg.svd(M)
    R = U @ np.diag([1.0, 1.0, np.linalg.det(U @ Vt)]) @ Vt
    return R


def so3_exp(phi: np.ndarray) -> np.ndarray:
    """
    Exponential map on SO(3):
        R = exp(hat(phi))

    Uses Rodrigues' formula with small-angle handling.
    """
    phi = ensure_vector(phi, 3)
    theta = np.linalg.norm(phi)
    Phi = hat(phi)

    if theta < 1e-12:
        return np.eye(3) + Phi

    A = np.sin(theta) / theta
    B = (1.0 - np.cos(theta)) / (theta ** 2)
    return np.eye(3) + A * Phi + B * (Phi @ Phi)


def so3_log(R: np.ndarray) -> np.ndarray:
    """
    Logarithm map on SO(3):
        phi = log(R)^vee

    For numerical robustness, R is first projected to SO(3).
    Includes special handling near theta = pi.
    """
    R = ensure_matrix(R, (3, 3))
    R = project_to_so3(R)

    cos_theta = (np.trace(R) - 1.0) / 2.0
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    theta = np.arccos(cos_theta)

    # Near zero
    if theta < 1e-12:
        return 0.5 * vee(R - R.T)

    # Near pi
    if np.pi - theta < 1e-6:
        A = 0.5 * (R + np.eye(3))
        axis = np.array([
            np.sqrt(max(A[0, 0], 0.0)),
            np.sqrt(max(A[1, 1], 0.0)),
            np.sqrt(max(A[2, 2], 0.0))
        ])

        # Recover signs from off-diagonal entries
        if axis[0] > 1e-8:
            axis[1] = np.copysign(axis[1], R[0, 1] + R[1, 0])
            axis[2] = np.copysign(axis[2], R[0, 2] + R[2, 0])
        elif axis[1] > 1e-8:
            axis[2] = np.copysign(axis[2], R[1, 2] + R[2, 1])
        else:
            # fallback: axis[2] dominates or all are tiny
            pass

        norm_axis = np.linalg.norm(axis)
        if norm_axis < 1e-12:
            axis = np.array([1.0, 0.0, 0.0])
        else:
            axis = axis / norm_axis

        return theta * axis

    return vee((theta / (2.0 * np.sin(theta))) * (R - R.T))


# =========================
# Pose operations （SE（3））
# =========================

def rigid_compose(X1: Pose, X2: Pose) -> Pose:
    """
    Standard rigid pose composition:
        X1 * X2 = (R1 R2, R1 p2 + p1)
    """
    R = X1.R @ X2.R
    p = X1.R @ X2.p + X1.p
    return Pose(R=R, p=p)


def rigid_inverse(X: Pose) -> Pose:
    """
    Pose inverse:
        X^{-1} = (R^T, -R^T p)
    """
    R_inv = X.R.T
    p_inv = -R_inv @ X.p
    return Pose(R=R_inv, p=p_inv)


# =========================
# Pose operations （SO（3）X R^3）
# =========================

def dp_compose(X1: Pose, X2: Pose) -> Pose:
    """
    Direct-product group operation on SO(3) x R^3:
        (R1, p1) * (R2, p2) = (R1 R2, p1 + p2).
    """
    return Pose(R=X1.R @ X2.R, p=X1.p + X2.p)


def dp_inverse(X: Pose) -> Pose:
    """
    Direct-product inverse:
        (R, p)^{-1} = (R^T, -p).
    """
    return Pose(R=X.R.T, p=-X.p)


def rigid_compose(X1: Pose, X2: Pose) -> Pose:
    """
    Standard rigid transform composition:
        (R1, p1) o (R2, p2) = (R1 R2, R1 p2 + p1).

    Use this only for coordinate-frame transformations, not for the
    direct-product group structure used in the estimator.
    """
    return Pose(R=X1.R @ X2.R, p=X1.R @ X2.p + X1.p)


def rigid_inverse(X: Pose) -> Pose:
    """
    Standard rigid transform inverse:
        (R, p)^{-1} = (R^T, -R^T p).
    """
    R_inv = X.R.T
    return Pose(R=R_inv, p=-R_inv @ X.p)





# =========================
# Right perturbation on SO(3) x R^3
# =========================

def split_xi(xi: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Split xi in R^6 into (phi, v), where
        xi = [phi; v]
        phi in R^3, v in R^3
    """
    xi = ensure_vector(xi, 6)
    return xi[:3], xi[3:]


def pack_xi(phi: np.ndarray, v: np.ndarray) -> np.ndarray:
    """
    Pack (phi, v) into xi = [phi; v] in R^6.
    """
    phi = ensure_vector(phi, 3)
    v = ensure_vector(v, 3)
    return np.concatenate([phi, v])


def apply_right_perturbation(X: Pose, xi: np.ndarray) -> Pose:
    """
    Right perturbation under the direct-product local model:
        X ⊕ xi = (R exp(hat(phi)), p + v)

    where xi = [phi; v] in R^6.
    """
    phi, v = split_xi(xi)
    R_new = X.R @ so3_exp(phi)
    p_new = X.p + v
    return Pose(R=R_new, p=p_new)


# =========================
# Error / distance utilities
# =========================

def rotation_geodesic_error(R1: np.ndarray, R2: np.ndarray) -> float:
    """
    Geodesic distance on SO(3):
        d(R1, R2) = || log(R1^T R2) ||_2
    """
    R1 = ensure_matrix(R1, (3, 3))
    R2 = ensure_matrix(R2, (3, 3))
    return float(np.linalg.norm(so3_log(R1.T @ R2)))


def pose_rotation_error(X1: Pose, X2: Pose) -> float:
    """
    Rotation geodesic error between two poses.
    """
    return rotation_geodesic_error(X1.R, X2.R)


def pose_translation_error(X1: Pose, X2: Pose) -> float:
    """
    Euclidean translation error between two poses.
    """
    return float(np.linalg.norm(X1.p - X2.p))


# =========================
# Optional validation helper
# =========================

def is_rotation_matrix(R: np.ndarray, atol: float = 1e-6) -> bool:
    """
    Check whether R is approximately in SO(3).
    """
    R = ensure_matrix(R, (3, 3))
    should_be_I = R.T @ R
    det_R = np.linalg.det(R)
    return (
        np.allclose(should_be_I, np.eye(3), atol=atol)
        and np.isclose(det_R, 1.0, atol=atol)
    )


## ==========MFG / Distribution - related================


In [9]:
# =========================
# MFG / Matrix-Fisher-Gaussian related
# =========================

from dataclasses import dataclass
import numpy as np


@dataclass
class MFGParameters:
    """
    Matrix-Fisher-Gaussian parameters:
        Theta = {F, mu, Lambda, Gamma}

    F      : Matrix Fisher parameter matrix, shape (3,3)
    mu     : translation base mean, shape (3,)
    Lambda : translation precision matrix, shape (3,3)
    Gamma  : rotation-translation coupling matrix, shape (3,3)
    """
    F: np.ndarray
    mu: np.ndarray
    Lambda: np.ndarray
    Gamma: np.ndarray

    def __post_init__(self):
        self.F = ensure_matrix(self.F, (3, 3))
        self.mu = ensure_vector(self.mu, 3)
        self.Lambda = ensure_matrix(self.Lambda, (3, 3))
        self.Gamma = ensure_matrix(self.Gamma, (3, 3))



def mfg_svd(F: np.ndarray):
    """
    Proper SVD for the Matrix-Fisher parameter:
        F = U S V^T,  U,V in SO(3),
    where the last signed singular value may be negative.
    """
    F = ensure_matrix(F, (3, 3))

    if not np.isfinite(F).all():
        raise ValueError("F contains NaN or Inf; SVD cannot be performed.")

    U0, s0, Vt0 = np.linalg.svd(F)
    V0 = Vt0.T

    DU = np.diag([1.0, 1.0, np.linalg.det(U0)])
    DV = np.diag([1.0, 1.0, np.linalg.det(V0)])

    U = U0 @ DU
    V = V0 @ DV
    S = DU @ np.diag(s0) @ DV

    return U, S, V




def rotation_mode_from_F(F: np.ndarray) -> np.ndarray:
    """
    Recover the mode rotation from F.

    If
        F = U S V^T,
    then
        R_mode = U diag(1,1,det(UV^T)) V^T.
    """
    F = ensure_matrix(F, (3, 3))

    if not np.isfinite(F).all():
        raise ValueError("F contains NaN or Inf; SVD cannot be performed.")

    U, _, Vt = np.linalg.svd(F)
    R_mode = U @ np.diag([1.0, 1.0, np.linalg.det(U @ Vt)]) @ Vt
    return R_mode


def mfg_nu_R(R: np.ndarray, F: np.ndarray) -> np.ndarray:
    """
    The MFG rotational deviation vector nu_R defined by Definition II.7:

        F = U S V^T,
        Q = U^T R V,
        nu_R = (Q S - S Q^T)^vee.

    This is the theory-consistent definition of nu_R.
    """
    R = ensure_matrix(R, (3, 3))
    F = ensure_matrix(F, (3, 3))

    U, S, V = mfg_svd(F)
    Q = U.T @ R @ V

    A = Q @ S - S @ Q.T
    # Numerical antisymmetrization for stability
    A = 0.5 * (A - A.T)
    return vee(A)


def conditional_translation_mean(R: np.ndarray, theta: MFGParameters) -> np.ndarray:
    """
    Rotation-dependent translation mean:
        mu_c(R) = mu + Gamma nu_R.
    """
    R = ensure_matrix(R, (3, 3))
    nu_R = mfg_nu_R(R, theta.F)
    return theta.mu + theta.Gamma @ nu_R



def recover_pose_from_theta(theta: MFGParameters) -> Pose:
    """
    Recover a representative MAP-like pose from the coupled MFG parameters.

    We use the Matrix-Fisher mode for rotation and the corresponding
    conditional translation mean:
        R_hat = mode(F),
        p_hat = mu + Gamma nu_R_hat.

    At the ideal Matrix-Fisher mode, nu_R_hat = 0, hence p_hat = mu.
    """
    R_hat = rotation_mode_from_F(theta.F)
    p_hat = conditional_translation_mean(R_hat, theta)
    return Pose(R=R_hat, p=p_hat)






# ============ Section_3  Potential and Wrench Model

## part A: SDF / stiffness / 
## superquadric implicit field


In [14]:
# =========================
# Field / stiffness parameter classes
# =========================


@dataclass
class SuperquadricFieldParameters:
    """
    Superquadric implicit field parameters.

    Implicit field:
        F_3d(x,y,z) = (f(x,y))^(eps2/eps1) + |z/a3|^(2/eps1) - 1

    where
        f(x,y) = |x/a1|^(2/eps2) + |y/a2|^(2/eps2)

    Notes
    -----
    - a1, a2, a3 > 0 are scale parameters
    - eps1, eps2 > 0 are shape parameters
    - sdf_eps is a small positive regularization used in the normalized signed field
    """
    a1: float
    a2: float
    a3: float
    eps1: float
    eps2: float
    sdf_eps: float = 1e-8

    def __post_init__(self):
        self.a1 = float(self.a1)
        self.a2 = float(self.a2)
        self.a3 = float(self.a3)
        self.eps1 = float(self.eps1)
        self.eps2 = float(self.eps2)
        self.sdf_eps = float(self.sdf_eps)

        if self.a1 <= 0 or self.a2 <= 0 or self.a3 <= 0:
            raise ValueError("a1, a2, a3 must all be positive.")
        if self.eps1 <= 0 or self.eps2 <= 0:
            raise ValueError("eps1 and eps2 must both be positive.")
        if self.sdf_eps <= 0:
            raise ValueError("sdf_eps must be positive.")


@dataclass
class StiffnessParameters:
    """
    Smooth stiffness profile parameters.

    Stiffness law:
        k(d) = k_min + ((1 - tanh(d / d0)) / 2) * k_max

    Notes
    -----
    - d0 > 0 controls the transition width
    - k_min, k_max are typically nonnegative
    """
    k_min: float
    k_max: float
    d0: float

    def __post_init__(self):
        self.k_min = float(self.k_min)
        self.k_max = float(self.k_max)
        self.d0 = float(self.d0)

        if self.d0 <= 0:
            raise ValueError("d0 must be positive.")
        if self.k_min < 0 or self.k_max < 0:
            raise ValueError("k_min and k_max should be nonnegative.")

In [15]:
# =========================
# Superquadric implicit field + normalized signed field
# + stiffness profile + local contact energy structure
# =========================

from typing import Iterable

# ---------------------------------------------------------
# Basic scalar-field numerical differentiation
# ---------------------------------------------------------

def numerical_gradient_scalar_field(
    func: Callable[[np.ndarray], float],
    x: np.ndarray,
    h: float = 1e-6
) -> np.ndarray:
    """
    Central-difference gradient of a scalar field func: R^3 -> R.
    """
    x = ensure_vector(x, 3)
    grad = np.zeros(3, dtype=float)

    for i in range(3):
        e = np.zeros(3, dtype=float)
        e[i] = 1.0
        grad[i] = (func(x + h * e) - func(x - h * e)) / (2.0 * h)

    return grad


def numerical_hessian_scalar_field(
    func: Callable[[np.ndarray], float],
    x: np.ndarray,
    h: float = 1e-4
) -> np.ndarray:
    """
    Central-difference Hessian of a scalar field func: R^3 -> R.
    """
    x = ensure_vector(x, 3)
    H = np.zeros((3, 3), dtype=float)

    for i in range(3):
        for j in range(3):
            ei = np.zeros(3, dtype=float)
            ej = np.zeros(3, dtype=float)
            ei[i] = 1.0
            ej[j] = 1.0

            f_pp = func(x + h * ei + h * ej)
            f_pm = func(x + h * ei - h * ej)
            f_mp = func(x - h * ei + h * ej)
            f_mm = func(x - h * ei - h * ej)

            H[i, j] = (f_pp - f_pm - f_mp + f_mm) / (4.0 * h * h)

    # enforce symmetry numerically
    H = 0.5 * (H + H.T)
    return H


# ---------------------------------------------------------
# Superquadric implicit field F_3d
# ---------------------------------------------------------

def smooth_abs_power(x: float, power: float, eps: float = 1e-12) -> float:
    """
    Smoothed version of |x|^power:
        |x|^p  ~  (x^2 + eps)^(p/2)

    This is used for numerical stability near x = 0.
    """
    x = float(x)
    power = float(power)
    eps = float(eps)
    return (x * x + eps) ** (0.5 * power)


def superquadric_f_xy(c: np.ndarray, params: SuperquadricFieldParameters) -> float:
    """
    2D intermediate function:
        f(x,y) = |x/a1|^(2/eps2) + |y/a2|^(2/eps2)
    """
    c = ensure_vector(c, 3)
    x, y, _ = c

    px = 2.0 / params.eps2
    py = 2.0 / params.eps2

    term_x = smooth_abs_power(x / params.a1, px)
    term_y = smooth_abs_power(y / params.a2, py)
    return term_x + term_y


def superquadric_F3d(c: np.ndarray, params: SuperquadricFieldParameters) -> float:
    """
    Superquadric implicit field:
        F_3d(x,y,z) = (f(x,y))^(eps2/eps1) + |z/a3|^(2/eps1) - 1
    """
    c = ensure_vector(c, 3)
    _, _, z = c

    fxy = superquadric_f_xy(c, params)
    power_xy = params.eps2 / params.eps1
    power_z = 2.0 / params.eps1

    term_xy = (fxy + 1e-16) ** power_xy
    term_z = smooth_abs_power(z / params.a3, power_z)

    return term_xy + term_z - 1.0


# ---------------------------------------------------------
# Normalized signed field
# ---------------------------------------------------------

def normalized_signed_field(c: np.ndarray, params: SuperquadricFieldParameters) -> float:
    """
    Normalized signed field:
        F_tilde(c) = F_3d(c) / (||grad F_3d(c)|| + sdf_eps)

    This is a normalized signed field / signed-distance-like field,
    not necessarily an exact analytical SDF.
    """
    c = ensure_vector(c, 3)

    def F_func(x: np.ndarray) -> float:
        return superquadric_F3d(x, params)

    F_val = F_func(c)
    grad_F = numerical_gradient_scalar_field(F_func, c, h=1e-6)
    denom = np.linalg.norm(grad_F) + params.sdf_eps
    return F_val / denom


def normalized_signed_field_gradient(
    c: np.ndarray,
    params: SuperquadricFieldParameters
) -> np.ndarray:
    """
    Gradient of the normalized signed field:
        g_B = grad_c F_tilde(c)
    """
    c = ensure_vector(c, 3)

    def F_tilde_func(x: np.ndarray) -> float:
        return normalized_signed_field(x, params)

    return numerical_gradient_scalar_field(F_tilde_func, c, h=1e-6)


def normalized_signed_field_hessian(
    c: np.ndarray,
    params: SuperquadricFieldParameters
) -> np.ndarray:
    """
    Hessian of the normalized signed field:
        H_B = Hessian_c F_tilde(c)
    """
    c = ensure_vector(c, 3)

    def F_tilde_func(x: np.ndarray) -> float:
        return normalized_signed_field(x, params)

    return numerical_hessian_scalar_field(F_tilde_func, c, h=1e-4)


# ---------------------------------------------------------
# Stiffness profile k(d), k'(d), k''(d)
# ---------------------------------------------------------

def stiffness_k(d: float, stiffness_params: StiffnessParameters) -> float:
    """
    Smooth stiffness profile:
        k(d) = k_min + ((1 - tanh(d / d0)) / 2) * k_max
    """
    d = float(d)
    k_min = stiffness_params.k_min
    k_max = stiffness_params.k_max
    d0 = stiffness_params.d0

    u = d / d0
    return k_min + 0.5 * (1.0 - np.tanh(u)) * k_max


def stiffness_k_prime(d: float, stiffness_params: StiffnessParameters) -> float:
    """
    First derivative of k(d).
    """
    d = float(d)
    k_max = stiffness_params.k_max
    d0 = stiffness_params.d0

    u = d / d0
    t = np.tanh(u)
    sech2 = 1.0 - t * t
    return -0.5 * k_max * sech2 / d0


def stiffness_k_double_prime(d: float, stiffness_params: StiffnessParameters) -> float:
    """
    Second derivative of k(d).
    """
    d = float(d)
    k_max = stiffness_params.k_max
    d0 = stiffness_params.d0

    u = d / d0
    t = np.tanh(u)
    sech2 = 1.0 - t * t
    return k_max * sech2 * t / (d0 ** 2)


# ---------------------------------------------------------
# Local contact energy density psi(d)
# ---------------------------------------------------------

def contact_energy_density(F_tilde: float, stiffness_params: StiffnessParameters) -> float:
    """
    Local contact energy density:
        psi(d) = 0.5 * k(d) * d^2
    with d = F_tilde.
    """
    d = float(F_tilde)
    k_val = stiffness_k(d, stiffness_params)
    return 0.5 * k_val * d * d


def alpha_from_F_tilde(F_tilde: float, stiffness_params: StiffnessParameters) -> float:
    """
    alpha(d) = psi'(d) = k(d)d + 0.5 k'(d)d^2
    """
    d = float(F_tilde)

    k_val = stiffness_k(d, stiffness_params)
    k_prime_val = stiffness_k_prime(d, stiffness_params)

    return k_val * d + 0.5 * k_prime_val * (d ** 2)


def kappa_from_F_tilde(F_tilde: float, stiffness_params: StiffnessParameters) -> float:
    """
    kappa(d) = psi''(d)
             = k(d) + 2 k'(d)d + 0.5 k''(d)d^2
    """
    d = float(F_tilde)

    k_val = stiffness_k(d, stiffness_params)
    k_prime_val = stiffness_k_prime(d, stiffness_params)
    k_double_prime_val = stiffness_k_double_prime(d, stiffness_params)

    return (
        k_val
        + 2.0 * k_prime_val * d
        + 0.5 * k_double_prime_val * (d ** 2)
    )


def alpha_and_kappa_from_F_tilde(
    F_tilde: float,
    stiffness_params: StiffnessParameters
) -> Tuple[float, float]:
    """
    Return both alpha(d)=psi'(d) and kappa(d)=psi''(d).
    """
    alpha = alpha_from_F_tilde(F_tilde, stiffness_params)
    kappa = kappa_from_F_tilde(F_tilde, stiffness_params)
    return alpha, kappa


# ---------------------------------------------------------
# Local gradient / Hessian of the contact energy wrt c_B
# ---------------------------------------------------------

def contact_energy_gradient_wrt_cB(
    c_B: np.ndarray,
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters
) -> np.ndarray:
    """
    Gradient of the local contact energy wrt c_B:

        grad_{c_B} psi = alpha(F_tilde) * grad F_tilde
    """
    c_B = ensure_vector(c_B, 3)

    F_tilde = normalized_signed_field(c_B, field_params)
    g_B = normalized_signed_field_gradient(c_B, field_params)
    alpha = alpha_from_F_tilde(F_tilde, stiffness_params)

    grad = alpha * g_B

    if not np.isfinite(grad).all():
        raise ValueError("NaN/Inf encountered in contact_energy_gradient_wrt_cB.")

    return grad


def local_stiffness_matrix(
    c_B: np.ndarray,
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters
):
    """
    Compute the local geometric quantities and the Hessian-like matrix

        K_i = kappa * g_B g_B^T + alpha * H_B

    where
        g_B = grad F_tilde(c_B),
        H_B = Hessian F_tilde(c_B).

    Returns
    -------
    g_B      : gradient of F_tilde wrt c_B
    F_tilde  : normalized signed field value
    alpha    : psi'(F_tilde)
    K_i      : Hessian of the local contact energy wrt c_B
    """
    c_B = ensure_vector(c_B, 3)

    F_tilde = normalized_signed_field(c_B, field_params)
    g_B = normalized_signed_field_gradient(c_B, field_params)
    H_B = normalized_signed_field_hessian(c_B, field_params)

    alpha, kappa = alpha_and_kappa_from_F_tilde(F_tilde, stiffness_params)

    K_i = kappa * np.outer(g_B, g_B) + alpha * H_B
    K_i = 0.5 * (K_i + K_i.T)

    if (
        (not np.isfinite(F_tilde))
        or (not np.isfinite(g_B).all())
        or (not np.isfinite(H_B).all())
        or (not np.isfinite(alpha))
        or (not np.isfinite(kappa))
        or (not np.isfinite(K_i).all())
    ):
        raise ValueError(
            "NaN/Inf encountered in local_stiffness_matrix:\n"
            f"c_B={c_B}\n"
            f"F_tilde={F_tilde}\n"
            f"g_B={g_B}\n"
            f"H_B=\n{H_B}\n"
            f"alpha={alpha}, kappa={kappa}\n"
            f"K_i=\n{K_i}"
        )

    return g_B, F_tilde, alpha, K_i


def contact_energy_hessian_wrt_cB(
    c_B: np.ndarray,
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters
) -> np.ndarray:
    """
    Hessian of the local contact energy wrt c_B.
    """
    _, _, _, K_i = local_stiffness_matrix(c_B, field_params, stiffness_params)
    return K_i


# ---------------------------------------------------------
# Point transform: frame A -> frame B
# ---------------------------------------------------------

def point_A_to_B(c_A: np.ndarray, X_A: Pose, X_B: Pose) -> np.ndarray:
    """
    Transform a point c_A expressed in frame {A} into frame {B}:

        c_W = R_A c_A + p_A
        c_B = R_B^T (c_W - p_B)
    """
    c_A = ensure_vector(c_A, 3)

    c_W = X_A.R @ c_A + X_A.p
    c_B = X_B.R.T @ (c_W - X_B.p)
    return c_B


# ---------------------------------------------------------
# Single-point and multi-point interaction potential
# ---------------------------------------------------------

def single_contact_potential(
    c_A: np.ndarray,
    X_A: Pose,
    X_B: Pose,
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters
) -> float:
    """
    Single-point interaction contribution:
        psi_i = psi(F_tilde(c_B))
    """
    c_A = ensure_vector(c_A, 3)
    c_B = point_A_to_B(c_A, X_A, X_B)
    F_tilde = normalized_signed_field(c_B, field_params)
    return contact_energy_density(F_tilde, stiffness_params)


def interaction_potential(
    X_A: Pose,
    X_B: Pose,
    contact_points_A: Iterable[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters
) -> float:
    """
    Total interaction potential:
        W_int(X_A, X_B) = sum_i psi(F_tilde(c_B^(i)))

    where c_B^(i) is the i-th contact point transformed from frame {A} to frame {B}.
    """
    total = 0.0
    for c_A in contact_points_A:
        total += single_contact_potential(
            c_A=c_A,
            X_A=X_A,
            X_B=X_B,
            field_params=field_params,
            stiffness_params=stiffness_params,
        )
    return float(total)

## Part B : Potential functions 

In [16]:
# =========================
# Control potential + interaction potential + total potential
# =========================


# ---------------------------------------------------------
# Control-potential parameters
# ---------------------------------------------------------

@dataclass
class ControlPotentialParameters:
    """
    Control-potential metric parameters.

    The control potential is
        W_ctrl(X_A, X_U) = 0.5 * d^T K_c d,

    where
        d = [ log_SO3(R_U R_A^T) ;
              p_U - p_A ]  in R^6.

    We use a block-diagonal metric:
        K_c = diag(K_R, K_p),
    where K_R, K_p are 3x3 positive-definite matrices.
    """
    K_R: np.ndarray
    K_p: np.ndarray

    def __post_init__(self):
        self.K_R = ensure_matrix(self.K_R, (3, 3))
        self.K_p = ensure_matrix(self.K_p, (3, 3))

        # basic symmetry check
        if not np.allclose(self.K_R, self.K_R.T, atol=1e-8):
            raise ValueError("K_R must be symmetric.")
        if not np.allclose(self.K_p, self.K_p.T, atol=1e-8):
            raise ValueError("K_p must be symmetric.")

        # basic positive-definite check
        eig_R = np.linalg.eigvalsh(self.K_R)
        eig_p = np.linalg.eigvalsh(self.K_p)
        if np.min(eig_R) <= 0.0:
            raise ValueError("K_R must be positive definite.")
        if np.min(eig_p) <= 0.0:
            raise ValueError("K_p must be positive definite.")

    @property
    def K_c(self) -> np.ndarray:
        """
        Full 6x6 block-diagonal control metric.
        """
        return np.block([
            [self.K_R, np.zeros((3, 3))],
            [np.zeros((3, 3)), self.K_p]
        ])


# ---------------------------------------------------------
# Control displacement d(X_U, X_A) -- Eq.（III.3）
# ---------------------------------------------------------

def control_displacement_vector(X_U: Pose, X_A: Pose) -> np.ndarray:
    """
    Control displacement vector in the world-frame convention:

        d(X_U, X_A) =
            [ log_SO3(R_U R_A^T) ;
              p_U - p_A ]  in R^6.

    This matches the notation in the current paper draft.
    """
    phi = so3_log(X_U.R @ X_A.R.T)
    dp = X_U.p - X_A.p
    return pack_xi(phi, dp)


def control_rotation_error(X_U: Pose, X_A: Pose) -> np.ndarray:
    """
    Rotation part of the control displacement:
        phi = log_SO3(R_U R_A^T).
    """
    return so3_log(X_U.R @ X_A.R.T)


def control_translation_error(X_U: Pose, X_A: Pose) -> np.ndarray:
    """
    Translation part of the control displacement:
        dp = p_U - p_A.
    """
    return ensure_vector(X_U.p - X_A.p, 3)


def weighted_quadratic_form(x: np.ndarray, Q: np.ndarray) -> float:
    """
    Compute x^T Q x for vector x and symmetric matrix Q.
    """
    x = np.asarray(x, dtype=float).reshape(-1)
    Q = np.asarray(Q, dtype=float)

    if Q.shape != (x.shape[0], x.shape[0]):
        raise ValueError(
            f"Shape mismatch: x has length {x.shape[0]}, but Q has shape {Q.shape}."
        )

    return float(x.T @ Q @ x)


# ---------------------------------------------------------
# Control potential W_ctrl
# ---------------------------------------------------------

def control_potential(
    X_A: Pose,
    X_U: Pose,
    ctrl_params: ControlPotentialParameters
) -> float:
    """
    Control potential:

        W_ctrl(X_A, X_U) = 0.5 * d^T K_c d,

    where
        d = [ log_SO3(R_U R_A^T) ;
              p_U - p_A ].
    """
    d = control_displacement_vector(X_U, X_A)
    return 0.5 * weighted_quadratic_form(d, ctrl_params.K_c)


# ---------------------------------------------------------
# Interaction potential wrappers
# ---------------------------------------------------------

def interaction_potential_single_value(
    c_A: np.ndarray,
    X_A: Pose,
    X_B: Pose,
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters
) -> float:
    """
    Single-point interaction contribution:

        W_int^(i) = 0.5 * k(F_tilde_i) * (F_tilde_i)^2.
    """
    return single_contact_potential(
        c_A=c_A,
        X_A=X_A,
        X_B=X_B,
        field_params=field_params,
        stiffness_params=stiffness_params,
    )


def interaction_potential_total(
    X_A: Pose,
    X_B: Pose,
    contact_points_A: Iterable[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters
) -> float:
    """
    Total interaction potential:

        W_int(X_A, X_B)
        = sum_i 0.5 * k(F_tilde_i) * (F_tilde_i)^2
        = sum_i psi(F_tilde_i).
    """
    return interaction_potential(
        X_A=X_A,
        X_B=X_B,
        contact_points_A=contact_points_A,
        field_params=field_params,
        stiffness_params=stiffness_params,
    )


# ---------------------------------------------------------
# Total potential W = W_ctrl + W_int
# ---------------------------------------------------------

def total_potential(
    X_A: Pose,
    X_B: Pose,
    X_U: Pose,
    contact_points_A: Iterable[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters
) -> float:
    """
    Total potential:

        W(X_A, X_B, X_U)
        = W_ctrl(X_A, X_U) + W_int(X_A, X_B).
    """
    W_ctrl = control_potential(X_A=X_A, X_U=X_U, ctrl_params=ctrl_params)
    W_int = interaction_potential_total(
        X_A=X_A,
        X_B=X_B,
        contact_points_A=contact_points_A,
        field_params=field_params,
        stiffness_params=stiffness_params,
    )
    return float(W_ctrl + W_int)


def total_potential_terms(
    X_A: Pose,
    X_B: Pose,
    X_U: Pose,
    contact_points_A: Iterable[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters
) -> Tuple[float, float, float]:
    """
    Return (W_ctrl, W_int, W_total) separately.
    Useful for debugging and visualization.
    """
    W_ctrl = control_potential(X_A=X_A, X_U=X_U, ctrl_params=ctrl_params)
    W_int = interaction_potential_total(
        X_A=X_A,
        X_B=X_B,
        contact_points_A=contact_points_A,
        field_params=field_params,
        stiffness_params=stiffness_params,
    )
    W_total = W_ctrl + W_int
    return float(W_ctrl), float(W_int), float(W_total)

## Part C: inner solve X_A^{*} --- Algorithm 2 

## &

## Part D: Residual Jacobian --- Schur Compensation

In [17]:
# =========================
# Implicit quasi-static equilibrium:
# solve X_A^*, predicted wrench, and residual Jacobian J_k
# =========================

from dataclasses import dataclass
from typing import Optional, Iterable, Tuple
import numpy as np


# ---------------------------------------------------------
# Solver configuration and result containers
# ---------------------------------------------------------

@dataclass
class ImplicitEquilibriumSolverConfig:
    """
    Configuration for the inner Newton solve of the implicit equilibrium state X_A^*.

    Notes
    -----
    - fd_rot_step / fd_pos_step are finite-difference step sizes in local coordinates
    - tol_eq is the stopping tolerance for the equilibrium residual norm
    - tol_step is the stopping tolerance for the Newton step norm
    - hessian_damping is a small Tikhonov regularization added to W_AA if needed
    """
    max_iter: int = 20
    tol_eq: float = 1e-8
    tol_step: float = 1e-8
    fd_rot_step: float = 1e-6
    fd_pos_step: float = 1e-6
    hessian_damping: float = 1e-8


@dataclass
class ImplicitEquilibriumResult:
    """
    Output of the inner equilibrium solve and local reduced linearization.

    Attributes
    ----------
    X_A_init   : initial guess used by the Newton solver
    X_A_star   : converged / returned implicit equilibrium state
    y_hat      : predicted wrench at equilibrium
    J_k        : residual Jacobian wrt X_B, i.e.
                 J_k = - W_UA W_AA^{-1} W_AB
    W_AA       : local Hessian block wrt X_A
    W_UA       : local mixed block wrt (X_U, X_A)
    W_AB       : local mixed block wrt (X_A, X_B)
    F_A        : final equilibrium residual vector
    converged  : whether stopping criterion was met
    n_iter     : number of Newton iterations used
    res_norm   : final residual norm
    step_norm  : final Newton-step norm
    """
    X_A_init: Pose
    X_A_star: Pose
    y_hat: np.ndarray
    J_k: np.ndarray
    W_AA: np.ndarray
    W_UA: np.ndarray
    W_AB: np.ndarray
    F_A: np.ndarray
    converged: bool
    n_iter: int
    res_norm: float
    step_norm: float


# ---------------------------------------------------------
# Small pose helpers
# ---------------------------------------------------------

def copy_pose(X: Pose) -> Pose:
    """
    Make a safe copy of a pose.
    """
    return Pose(R=X.R.copy(), p=X.p.copy())


def pose_fd_steps(config: ImplicitEquilibriumSolverConfig) -> np.ndarray:
    """
    Finite-difference step vector in local coordinates:
        [h_rot, h_rot, h_rot, h_pos, h_pos, h_pos].
    """
    return np.array([
        config.fd_rot_step,
        config.fd_rot_step,
        config.fd_rot_step,
        config.fd_pos_step,
        config.fd_pos_step,
        config.fd_pos_step,
    ], dtype=float)


# ---------------------------------------------------------
# Numerical local differentiation wrt pose variables
# ---------------------------------------------------------

def numerical_gradient_scalar_wrt_pose(
    func,
    X: Pose,
    config: ImplicitEquilibriumSolverConfig
) -> np.ndarray:
    """
    Central-difference gradient of a scalar function wrt the local pose coordinate xi in R^6,
    under the right perturbation X ⊕ xi.
    """
    steps = pose_fd_steps(config)
    grad = np.zeros(6, dtype=float)

    for i in range(6):
        delta = np.zeros(6, dtype=float)
        delta[i] = steps[i]

        f_plus = func(apply_right_perturbation(X, delta))
        f_minus = func(apply_right_perturbation(X, -delta))
        grad[i] = (f_plus - f_minus) / (2.0 * steps[i])

    return grad


def numerical_jacobian_vector_wrt_pose(
    func,
    X: Pose,
    config: ImplicitEquilibriumSolverConfig
) -> np.ndarray:
    """
    Central-difference Jacobian of a vector-valued function wrt the local pose coordinate xi in R^6.

    If func(X) in R^m, then the returned matrix has shape (m, 6).
    """
    steps = pose_fd_steps(config)
    f0 = np.asarray(func(X), dtype=float).reshape(-1)
    m = f0.shape[0]
    J = np.zeros((m, 6), dtype=float)

    for j in range(6):
        delta = np.zeros(6, dtype=float)
        delta[j] = steps[j]

        f_plus = np.asarray(func(apply_right_perturbation(X, delta)), dtype=float).reshape(-1)
        f_minus = np.asarray(func(apply_right_perturbation(X, -delta)), dtype=float).reshape(-1)

        J[:, j] = (f_plus - f_minus) / (2.0 * steps[j])

    return J


def numerical_hessian_scalar_wrt_pose(
    func,
    X: Pose,
    config: ImplicitEquilibriumSolverConfig
) -> np.ndarray:
    """
    Hessian of a scalar function wrt the local pose coordinate xi in R^6.

    Implemented as the Jacobian of the local gradient.
    """
    def grad_func(X_curr: Pose) -> np.ndarray:
        return numerical_gradient_scalar_wrt_pose(func, X_curr, config)

    H = numerical_jacobian_vector_wrt_pose(grad_func, X, config)
    H = 0.5 * (H + H.T)
    return H


def numerical_cross_hessian_scalar_wrt_poses(
    func_row_col,
    X_row: Pose,
    X_col: Pose,
    config: ImplicitEquilibriumSolverConfig
) -> np.ndarray:
    """
    Mixed Hessian block of a scalar function with respect to two pose variables.

    The returned matrix is H_{row,col} with shape (6, 6), defined by
        [H_{row,col}]_{ij} = d^2 f / (d xi_row_i d xi_col_j).

    Here func_row_col is a scalar function of two pose arguments:
        f = func_row_col(X_row, X_col).
    """
    def grad_row_given_col(X_col_curr: Pose) -> np.ndarray:
        def scalar_in_row(X_row_curr: Pose) -> float:
            return func_row_col(X_row_curr, X_col_curr)
        return numerical_gradient_scalar_wrt_pose(scalar_in_row, X_row, config)

    H_row_col = numerical_jacobian_vector_wrt_pose(grad_row_given_col, X_col, config)
    return H_row_col


# ---------------------------------------------------------
# Equilibrium operator Phi(X_A; X_B, X_U) = grad_{X_A} W
# ---------------------------------------------------------

def equilibrium_operator(
    X_A: Pose,
    X_B: Pose,
    X_U: Pose,
    contact_points_A: Iterable[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters,
    solver_config: ImplicitEquilibriumSolverConfig
) -> np.ndarray:
    """
    Local equilibrium operator in coordinates:

        Phi(X_A; X_B, X_U)
        = d/d xi_A W(X_A ⊕ xi_A, X_B, X_U) |_{xi_A=0}  in R^6.

    This is the local-coordinate representation of grad_{X_A} W.
    """
    def scalar_func(X_A_curr: Pose) -> float:
        return total_potential(
            X_A=X_A_curr,
            X_B=X_B,
            X_U=X_U,
            contact_points_A=contact_points_A,
            field_params=field_params,
            stiffness_params=stiffness_params,
            ctrl_params=ctrl_params,
        )

    return numerical_gradient_scalar_wrt_pose(scalar_func, X_A, solver_config)


def equilibrium_hessian_AA(
    X_A: Pose,
    X_B: Pose,
    X_U: Pose,
    contact_points_A: Iterable[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters,
    solver_config: ImplicitEquilibriumSolverConfig
) -> np.ndarray:
    """
    Local Hessian block:
        W_AA = Hessian_{X_A} W(X_A, X_B, X_U)  in local coordinates.
    """
    def scalar_func(X_A_curr: Pose) -> float:
        return total_potential(
            X_A=X_A_curr,
            X_B=X_B,
            X_U=X_U,
            contact_points_A=contact_points_A,
            field_params=field_params,
            stiffness_params=stiffness_params,
            ctrl_params=ctrl_params,
        )

    H = numerical_hessian_scalar_wrt_pose(scalar_func, X_A, solver_config)
    return 0.5 * (H + H.T)


def total_potential_block_UA(
    X_A: Pose,
    X_B: Pose,
    X_U: Pose,
    contact_points_A: Iterable[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters,
    solver_config: ImplicitEquilibriumSolverConfig
) -> np.ndarray:
    """
    Mixed block:
        W_UA = d^2 W / (d xi_U d xi_A)
    evaluated at (X_A, X_B, X_U).
    """
    def scalar_func(X_U_curr: Pose, X_A_curr: Pose) -> float:
        return total_potential(
            X_A=X_A_curr,
            X_B=X_B,
            X_U=X_U_curr,
            contact_points_A=contact_points_A,
            field_params=field_params,
            stiffness_params=stiffness_params,
            ctrl_params=ctrl_params,
        )

    return numerical_cross_hessian_scalar_wrt_poses(
        func_row_col=scalar_func,
        X_row=X_U,
        X_col=X_A,
        config=solver_config,
    )


def total_potential_block_AB(
    X_A: Pose,
    X_B: Pose,
    X_U: Pose,
    contact_points_A: Iterable[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters,
    solver_config: ImplicitEquilibriumSolverConfig
) -> np.ndarray:
    """
    Mixed block:
        W_AB = d^2 W / (d xi_A d xi_B)
    evaluated at (X_A, X_B, X_U).
    """
    def scalar_func(X_A_curr: Pose, X_B_curr: Pose) -> float:
        return total_potential(
            X_A=X_A_curr,
            X_B=X_B_curr,
            X_U=X_U,
            contact_points_A=contact_points_A,
            field_params=field_params,
            stiffness_params=stiffness_params,
            ctrl_params=ctrl_params,
        )

    return numerical_cross_hessian_scalar_wrt_poses(
        func_row_col=scalar_func,
        X_row=X_A,
        X_col=X_B,
        config=solver_config,
    )


# ---------------------------------------------------------
# Predicted wrench from control potential
# ---------------------------------------------------------

def predicted_control_wrench(
    X_A: Pose,
    X_U: Pose,
    ctrl_params: ControlPotentialParameters,
    solver_config: ImplicitEquilibriumSolverConfig
) -> np.ndarray:
    """
    Predicted control wrench:

        y_hat = - d/d xi_U W_ctrl(X_A, X_U) |_{xi_U=0}  in R^6.

    This matches the control-wrench observation convention used later.
    """
    def scalar_func(X_U_curr: Pose) -> float:
        return control_potential(
            X_A=X_A,
            X_U=X_U_curr,
            ctrl_params=ctrl_params,
        )

    grad_U = numerical_gradient_scalar_wrt_pose(scalar_func, X_U, solver_config)
    return -grad_U


# ---------------------------------------------------------
# Initial guess policy
# ---------------------------------------------------------

def select_equilibrium_initial_guess(
    X_U_k: Pose,
    explicit_init: Optional[Pose] = None,
    prev_outer_same_k: Optional[Pose] = None,
    prev_sample_same_outer: Optional[Pose] = None,
) -> Pose:
    """
    Select the initial guess for the inner Newton solve.

    Priority:
    1) explicit_init            : if the caller explicitly provides one
    2) prev_outer_same_k        : use the previous outer iteration's solution for the same k
                                  (this is the natural choice in Alg2 at outer iteration t >= 1)
    3) prev_sample_same_outer   : use the previous sample's equilibrium solution in the same pass
                                  (Remark VII.1: for k >= 2, initialize by X_{A,k-1}^*)
    4) X_U_k                    : fallback / first sample initialization

    Notes
    -----
    - For the very first solve (outer t = 0, first sample), use X_U_k.
    - In Python loops, if k is 0-based, then "first sample" means k == 0.
    """
    if explicit_init is not None:
        return copy_pose(explicit_init)

    if prev_outer_same_k is not None:
        return copy_pose(prev_outer_same_k)

    if prev_sample_same_outer is not None:
        return copy_pose(prev_sample_same_outer)

    return copy_pose(X_U_k)


# ---------------------------------------------------------
# Newton inner solve for X_A^*
# ---------------------------------------------------------

def solve_XA_star(
    X_B: Pose,
    X_U_k: Pose,
    contact_points_A: Iterable[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters,
    solver_config: Optional[ImplicitEquilibriumSolverConfig] = None,
    X_A_init: Optional[Pose] = None,
    prev_outer_same_k: Optional[Pose] = None,
    prev_sample_same_outer: Optional[Pose] = None,
) -> Tuple[Pose, dict]:
    """
    Solve the implicit equilibrium state X_A^*(X_U_k, X_B) by Newton iteration.

    Returns
    -------
    X_A_star : Pose
        Returned equilibrium pose
    info : dict
        Diagnostic information including convergence status
    """
    if solver_config is None:
        solver_config = ImplicitEquilibriumSolverConfig()

    X_A0 = select_equilibrium_initial_guess(
        X_U_k=X_U_k,
        explicit_init=X_A_init,
        prev_outer_same_k=prev_outer_same_k,
        prev_sample_same_outer=prev_sample_same_outer,
    )

    X_A = copy_pose(X_A0)
    converged = False
    last_res_norm = np.inf
    last_step_norm = np.inf
    last_F_A = None
    last_W_AA = None

    for j in range(solver_config.max_iter):
        F_A = equilibrium_operator(
            X_A=X_A,
            X_B=X_B,
            X_U=X_U_k,
            contact_points_A=contact_points_A,
            field_params=field_params,
            stiffness_params=stiffness_params,
            ctrl_params=ctrl_params,
            solver_config=solver_config,
        )
        W_AA = equilibrium_hessian_AA(
            X_A=X_A,
            X_B=X_B,
            X_U=X_U_k,
            contact_points_A=contact_points_A,
            field_params=field_params,
            stiffness_params=stiffness_params,
            ctrl_params=ctrl_params,
            solver_config=solver_config,
        )

        last_F_A = F_A
        last_W_AA = W_AA
        last_res_norm = float(np.linalg.norm(F_A))

        # Practical early stop if already equilibrated
        if last_res_norm <= solver_config.tol_eq:
            last_step_norm = 0.0
            converged = True
            break

        W_AA_reg = W_AA + solver_config.hessian_damping * np.eye(6)

        try:
            delta = -np.linalg.solve(W_AA_reg, F_A)
        except np.linalg.LinAlgError:
            delta = -np.linalg.lstsq(W_AA_reg, F_A, rcond=None)[0]

        last_step_norm = float(np.linalg.norm(delta))

        X_A = apply_right_perturbation(X_A, delta)

        # Mirror the paper-style practical stopping rule
        if last_step_norm <= solver_config.tol_step:
            converged = True
            break

    # Re-evaluate at the returned point for a consistent final record
    F_A_final = equilibrium_operator(
        X_A=X_A,
        X_B=X_B,
        X_U=X_U_k,
        contact_points_A=contact_points_A,
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        solver_config=solver_config,
    )
    W_AA_final = equilibrium_hessian_AA(
        X_A=X_A,
        X_B=X_B,
        X_U=X_U_k,
        contact_points_A=contact_points_A,
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        solver_config=solver_config,
    )

    info = {
        "X_A_init": X_A0,
        "converged": bool(converged or (np.linalg.norm(F_A_final) <= solver_config.tol_eq)),
        "n_iter": int(j + 1),
        "res_norm": float(np.linalg.norm(F_A_final)),
        "step_norm": float(last_step_norm),
        "F_A": F_A_final,
        "W_AA": W_AA_final,
    }
    return X_A, info


# ---------------------------------------------------------
# Linearization blocks and Schur-complement residual Jacobian
# ---------------------------------------------------------

def equilibrium_linearization_blocks(
    X_A_star: Pose,
    X_B: Pose,
    X_U_k: Pose,
    contact_points_A: Iterable[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters,
    solver_config: Optional[ImplicitEquilibriumSolverConfig] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Evaluate the local blocks at the equilibrium triple:

        W_AA, W_UA, W_AB.

    These are later used to form the residual Jacobian
        J_k = - W_UA W_AA^{-1} W_AB.
    """
    if solver_config is None:
        solver_config = ImplicitEquilibriumSolverConfig()

    W_AA = equilibrium_hessian_AA(
        X_A=X_A_star,
        X_B=X_B,
        X_U=X_U_k,
        contact_points_A=contact_points_A,
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        solver_config=solver_config,
    )

    W_UA = total_potential_block_UA(
        X_A=X_A_star,
        X_B=X_B,
        X_U=X_U_k,
        contact_points_A=contact_points_A,
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        solver_config=solver_config,
    )

    W_AB = total_potential_block_AB(
        X_A=X_A_star,
        X_B=X_B,
        X_U=X_U_k,
        contact_points_A=contact_points_A,
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        solver_config=solver_config,
    )

    return W_AA, W_UA, W_AB


def schur_complement_residual_jacobian(
    W_AA: np.ndarray,
    W_UA: np.ndarray,
    W_AB: np.ndarray,
    solver_config: Optional[ImplicitEquilibriumSolverConfig] = None,
) -> np.ndarray:
    """
    Form the residual Jacobian

        J_k = - W_UA W_AA^{-1} W_AB.

    This matches the residual convention
        r_k = y_k - y_hat_k(X_B),
    so J_k is the Jacobian of the residual wrt the local perturbation of X_B.
    """
    W_AA = ensure_matrix(W_AA, (6, 6))
    W_UA = ensure_matrix(W_UA, (6, 6))
    W_AB = ensure_matrix(W_AB, (6, 6))

    if solver_config is None:
        solver_config = ImplicitEquilibriumSolverConfig()

    W_AA_reg = W_AA + solver_config.hessian_damping * np.eye(6)

    try:
        tmp = np.linalg.solve(W_AA_reg, W_AB)
    except np.linalg.LinAlgError:
        tmp = np.linalg.lstsq(W_AA_reg, W_AB, rcond=None)[0]

    J_k = -W_UA @ tmp
    return J_k


# ---------------------------------------------------------
# Main interface used  by Alg1 / Alg2
# ---------------------------------------------------------

def solve_implicit_equilibrium_and_linearization(
    X_B: Pose,
    X_U_k: Pose,
    contact_points_A: Iterable[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters,
    solver_config: Optional[ImplicitEquilibriumSolverConfig] = None,
    X_A_init: Optional[Pose] = None,
    prev_outer_same_k: Optional[Pose] = None,
    prev_sample_same_outer: Optional[Pose] = None,
) -> ImplicitEquilibriumResult:
    """
    Main interface for both Alg1 and Alg2.

    It performs:
    1) inner Newton solve for X_A^*
    2) predicted wrench evaluation
    3) local block evaluation
    4) residual Jacobian construction

    Returns
    -------
    ImplicitEquilibriumResult
        containing X_A_star, y_hat, J_k, and all diagnostics.
    """
    if solver_config is None:
        solver_config = ImplicitEquilibriumSolverConfig()

    X_A_star, info = solve_XA_star(
        X_B=X_B,
        X_U_k=X_U_k,
        contact_points_A=contact_points_A,
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        solver_config=solver_config,
        X_A_init=X_A_init,
        prev_outer_same_k=prev_outer_same_k,
        prev_sample_same_outer=prev_sample_same_outer,
    )

    y_hat = predicted_control_wrench(
        X_A=X_A_star,
        X_U=X_U_k,
        ctrl_params=ctrl_params,
        solver_config=solver_config,
    )

    W_AA, W_UA, W_AB = equilibrium_linearization_blocks(
        X_A_star=X_A_star,
        X_B=X_B,
        X_U_k=X_U_k,
        contact_points_A=contact_points_A,
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        solver_config=solver_config,
    )

    J_k = schur_complement_residual_jacobian(
        W_AA=W_AA,
        W_UA=W_UA,
        W_AB=W_AB,
        solver_config=solver_config,
    )

    return ImplicitEquilibriumResult(
        X_A_init=info["X_A_init"],
        X_A_star=X_A_star,
        y_hat=y_hat,
        J_k=J_k,
        W_AA=W_AA,
        W_UA=W_UA,
        W_AB=W_AB,
        F_A=info["F_A"],
        converged=info["converged"],
        n_iter=info["n_iter"],
        res_norm=info["res_norm"],
        step_norm=info["step_norm"],
    )

## Part E =====batch-level Measurement k (model evaluation)

In [18]:
# =========================
# Batch-level reduced model 
# for a given candidate X_B
# =========================

from dataclasses import dataclass
from typing import Iterable, Optional, Sequence
import numpy as np


# ---------------------------------------------------------
# Result container
# ---------------------------------------------------------

@dataclass
class BatchReducedModelResult:
    """
    Batch-level reduced model evaluation result for a fixed candidate X_B.

    Attributes
    ----------
    X_B                  : current candidate object pose
    X_A_stars            : list of implicit equilibrium states X_{A,k}^*
    y_obs                : observed wrench stack, shape (K, 6)
    y_hat                : predicted wrench stack, shape (K, 6)
    residuals            : residual stack, shape (K, 6), residual = y_obs - y_hat
    J                    : residual Jacobian stack, shape (K, 6, 6)
    J_phi                : rotation block of J, shape (K, 6, 3)
    J_v                  : translation block of J, shape (K, 6, 3)
    per_sample_costs     : 0.5 * r_k^T Sigma_w^{-1} r_k, shape (K,)
    per_sample_whitened_norms : sqrt(r_k^T Sigma_w^{-1} r_k), shape (K,)
    total_whitened_cost  : sum_k 0.5 * r_k^T Sigma_w^{-1} r_k
    whitened_rms         : sqrt(mean_k r_k^T Sigma_w^{-1} r_k)
    converged_mask       : whether the inner equilibrium solve converged for each k
    n_converged          : number of converged inner solves
    sample_results       : raw per-sample ImplicitEquilibriumResult objects
    """
    X_B: Pose
    X_A_stars: list[Pose]
    y_obs: np.ndarray
    y_hat: np.ndarray
    residuals: np.ndarray
    J: np.ndarray
    J_phi: np.ndarray
    J_v: np.ndarray
    per_sample_costs: np.ndarray
    per_sample_whitened_norms: np.ndarray
    total_whitened_cost: float
    whitened_rms: float
    converged_mask: np.ndarray
    n_converged: int
    sample_results: list
    rho: float
    rho_sq: float
    


# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------

def prepare_measurement_precision(
    Sigma_w: Optional[np.ndarray] = None,
    Sigma_w_inv: Optional[np.ndarray] = None,
) -> np.ndarray:
    """
    Prepare the 6x6 measurement precision matrix Sigma_w^{-1}.

    Priority:
    1) use Sigma_w_inv directly if provided
    2) otherwise invert Sigma_w if provided
    3) otherwise use identity
    """
    if Sigma_w_inv is not None:
        P = ensure_matrix(Sigma_w_inv, (6, 6))
        return P

    if Sigma_w is not None:
        S = ensure_matrix(Sigma_w, (6, 6))
        try:
            return np.linalg.inv(S)
        except np.linalg.LinAlgError:
            return np.linalg.pinv(S)

    return np.eye(6)


def whitened_quadratic_cost(r: np.ndarray, Sigma_w_inv: np.ndarray) -> float:
    """
    0.5 * r^T Sigma_w^{-1} r
    """
    r = ensure_vector(r, 6)
    Sigma_w_inv = ensure_matrix(Sigma_w_inv, (6, 6))
    return 0.5 * float(r.T @ Sigma_w_inv @ r)


def whitened_norm_sq(r: np.ndarray, Sigma_w_inv: np.ndarray) -> float:
    """
    r^T Sigma_w^{-1} r
    """
    r = ensure_vector(r, 6)
    Sigma_w_inv = ensure_matrix(Sigma_w_inv, (6, 6))
    return float(r.T @ Sigma_w_inv @ r)


def split_residual_jacobian_blocks(J_k: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Split J_k in R^{6x6} into
        J_k = [J_phi, J_v],
    where both blocks are 6x3.
    """
    J_k = ensure_matrix(J_k, (6, 6))
    return J_k[:, :3], J_k[:, 3:]


def materialize_contact_points(
    contact_points_A: Iterable[np.ndarray]
) -> list[np.ndarray]:
    """
    Convert contact points to a reusable list of 3D vectors.
    This avoids issues if the input is a one-pass generator.
    """
    return [ensure_vector(c, 3).copy() for c in contact_points_A]


def get_contact_points_for_sample(
    k: int,
    contact_points_A_shared: Optional[Iterable[np.ndarray]] = None,
    contact_points_A_per_sample: Optional[Sequence[Iterable[np.ndarray]]] = None,
) -> list[np.ndarray]:
    """
    Select contact points for sample k.

    Either:
    - provide contact_points_A_shared, reused for all k
    - or provide contact_points_A_per_sample[k]
    """
    if contact_points_A_per_sample is not None:
        if len(contact_points_A_per_sample) <= k:
            raise IndexError(
                f"contact_points_A_per_sample has length {len(contact_points_A_per_sample)}, "
                f"but tried to access index k={k}."
            )
        return materialize_contact_points(contact_points_A_per_sample[k])

    if contact_points_A_shared is not None:
        return materialize_contact_points(contact_points_A_shared)

    raise ValueError(
        "You must provide either contact_points_A_shared or contact_points_A_per_sample."
    )


# ---------------------------------------------------------
########  Main batch-level 
# ---------------------------------------------------------

def evaluate_batch_reduced_model(
    X_B: Pose,
    X_U_list: Sequence[Pose],
    y_list: Sequence[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters,
    *,
    contact_points_A_shared: Optional[Iterable[np.ndarray]] = None,
    contact_points_A_per_sample: Optional[Sequence[Iterable[np.ndarray]]] = None,
    solver_config: Optional[ImplicitEquilibriumSolverConfig] = None,
    Sigma_w: Optional[np.ndarray] = None,
    Sigma_w_inv: Optional[np.ndarray] = None,
    explicit_XA_inits: Optional[Sequence[Optional[Pose]]] = None,
    prev_outer_XA_stars: Optional[Sequence[Optional[Pose]]] = None,
) -> BatchReducedModelResult:
    """
    Batch-level reduced-model oracle for a fixed candidate X_B.

    For each sample k:
      1) solve the implicit equilibrium state X_{A,k}^*
      2) compute predicted wrench y_hat_k(X_B)
      3) compute residual Jacobian J_k
      4) form residual r_k = y_k - y_hat_k
      5) accumulate whitened costs

    Parameters
    ----------
    X_B : Pose
        Current candidate object pose.
    X_U_list : sequence of Pose
        Control/reference poses {X_{U,k}}.
    y_list : sequence of ndarray
        Observed wrench measurements {y_k}, each shape (6,).
    field_params, stiffness_params, ctrl_params
        Model parameters.
    contact_points_A_shared : iterable of ndarray, optional
        Same contact points used for every k.
    contact_points_A_per_sample : sequence of iterable, optional
        Per-sample contact points, if they vary with k.
    solver_config : ImplicitEquilibriumSolverConfig, optional
        Inner Newton solver config.
    Sigma_w, Sigma_w_inv : ndarray, optional
        Measurement covariance or precision matrix for whitened cost.
    explicit_XA_inits : optional sequence of Pose / None
        Explicit initial guesses for each k.
    prev_outer_XA_stars : optional sequence of Pose / None
        Warm starts from the previous outer iteration for the same k.
        This is the natural warm start used in Alg2.

    Returns
    -------
    BatchReducedModelResult
    """
    if solver_config is None:
        solver_config = ImplicitEquilibriumSolverConfig()

    K = len(X_U_list)
    if len(y_list) != K:
        raise ValueError(
            f"Length mismatch: len(X_U_list)={K}, but len(y_list)={len(y_list)}."
        )

    if explicit_XA_inits is not None and len(explicit_XA_inits) != K:
        raise ValueError(
            f"Length mismatch: explicit_XA_inits has length {len(explicit_XA_inits)}, "
            f"expected {K}."
        )

    if prev_outer_XA_stars is not None and len(prev_outer_XA_stars) != K:
        raise ValueError(
            f"Length mismatch: prev_outer_XA_stars has length {len(prev_outer_XA_stars)}, "
            f"expected {K}."
        )

    P_w = prepare_measurement_precision(Sigma_w=Sigma_w, Sigma_w_inv=Sigma_w_inv)

    X_A_stars: list[Pose] = []
    y_obs_list: list[np.ndarray] = []
    y_hat_list: list[np.ndarray] = []
    residual_list: list[np.ndarray] = []
    J_list: list[np.ndarray] = []
    J_phi_list: list[np.ndarray] = []
    J_v_list: list[np.ndarray] = []
    per_sample_costs: list[float] = []
    per_sample_whitened_norms: list[float] = []
    converged_mask: list[bool] = []
    sample_results: list = []

    # Same-outer-pass warm start chain:
    # if prev_outer_XA_stars[k] is not provided, we use X_{A,k-1}^*.
    prev_sample_same_outer: Optional[Pose] = None

    for k in range(K):
        X_U_k = X_U_list[k]
        y_k = ensure_vector(y_list[k], 6)

        contact_points_k = get_contact_points_for_sample(
            k=k,
            contact_points_A_shared=contact_points_A_shared,
            contact_points_A_per_sample=contact_points_A_per_sample,
        )

        X_A_init_k = None if explicit_XA_inits is None else explicit_XA_inits[k]
        prev_outer_same_k = None if prev_outer_XA_stars is None else prev_outer_XA_stars[k]

        sample_res = solve_implicit_equilibrium_and_linearization(
            X_B=X_B,
            X_U_k=X_U_k,
            contact_points_A=contact_points_k,
            field_params=field_params,
            stiffness_params=stiffness_params,
            ctrl_params=ctrl_params,
            solver_config=solver_config,
            X_A_init=X_A_init_k,
            prev_outer_same_k=prev_outer_same_k,
            prev_sample_same_outer=prev_sample_same_outer,
        )

        r_k = y_k - sample_res.y_hat
        cost_k = whitened_quadratic_cost(r_k, P_w)
        norm_sq_k = whitened_norm_sq(r_k, P_w)
        wn_k = float(np.sqrt(max(norm_sq_k, 0.0)))

        J_phi_k, J_v_k = split_residual_jacobian_blocks(sample_res.J_k)

        X_A_stars.append(sample_res.X_A_star)
        y_obs_list.append(y_k.copy())
        y_hat_list.append(sample_res.y_hat.copy())
        residual_list.append(r_k.copy())
        J_list.append(sample_res.J_k.copy())
        J_phi_list.append(J_phi_k.copy())
        J_v_list.append(J_v_k.copy())
        per_sample_costs.append(cost_k)
        per_sample_whitened_norms.append(wn_k)
        converged_mask.append(bool(sample_res.converged))
        sample_results.append(sample_res)

        # same-pass sequential warm start
        prev_sample_same_outer = sample_res.X_A_star

    y_obs_stack = np.vstack(y_obs_list)
    y_hat_stack = np.vstack(y_hat_list)
    residual_stack = np.vstack(residual_list)
    J_stack = np.stack(J_list, axis=0)
    J_phi_stack = np.stack(J_phi_list, axis=0)
    J_v_stack = np.stack(J_v_list, axis=0)
    per_sample_costs_arr = np.asarray(per_sample_costs, dtype=float)
    per_sample_whitened_norms_arr = np.asarray(per_sample_whitened_norms, dtype=float)
    converged_mask_arr = np.asarray(converged_mask, dtype=bool)

    total_whitened_cost = float(np.sum(per_sample_costs_arr))
    whitened_rms = float(np.sqrt(np.mean(per_sample_whitened_norms_arr ** 2)))
    n_converged = int(np.sum(converged_mask_arr))
    # New: residual merit used by Algorithm 1 and Algorithm 2
    rho_sq = float(np.sum(per_sample_whitened_norms_arr ** 2))
    rho = float(np.sqrt(max(rho_sq, 0.0)))

    return BatchReducedModelResult(
        X_B=copy_pose(X_B),
        X_A_stars=X_A_stars,
        y_obs=y_obs_stack,
        y_hat=y_hat_stack,
        residuals=residual_stack,
        J=J_stack,
        J_phi=J_phi_stack,
        J_v=J_v_stack,
        per_sample_costs=per_sample_costs_arr,
        per_sample_whitened_norms=per_sample_whitened_norms_arr,
        total_whitened_cost=total_whitened_cost,
        whitened_rms=whitened_rms,
        converged_mask=converged_mask_arr,
        n_converged=n_converged,
        sample_results=sample_results,
        rho=rho,
        rho_sq=rho_sq,
    )




@dataclass
class DataInformationResult:
    """
    Data-induced local quantities used by Algorithm 1 and Algorithm 2.
    """
    rho: float
    g_data: np.ndarray
    H_data: np.ndarray
    I_rot: np.ndarray
    s_rot: float
    H_blocks: dict


def compute_data_information(
    batch: BatchReducedModelResult,
    Sigma_w: Optional[np.ndarray] = None,
    Sigma_w_inv: Optional[np.ndarray] = None,
    schur_damping: float = 1e-10,
) -> DataInformationResult:
    """
    Compute the local data gradient, Gauss-Newton information matrix,
    residual merit, and conditional rotational information score.

    Residual convention:
        r_k = y_k - y_hat_k,
        r_k(X_bar oplus xi) = r_k(X_bar) + J_k xi + O(||xi||^2).

    Therefore:
        g_data = sum_k J_k^T Sigma_w^{-1} r_k,
        H_data = sum_k J_k^T Sigma_w^{-1} J_k.
    """
    P_w = prepare_measurement_precision(Sigma_w=Sigma_w, Sigma_w_inv=Sigma_w_inv)

    g_data = np.zeros(6, dtype=float)
    H_data = np.zeros((6, 6), dtype=float)
    rho_sq = 0.0

    K = batch.residuals.shape[0]

    for k in range(K):
        r_k = ensure_vector(batch.residuals[k], 6)
        J_k = ensure_matrix(batch.J[k], (6, 6))

        g_data += J_k.T @ P_w @ r_k
        H_data += J_k.T @ P_w @ J_k
        rho_sq += float(r_k.T @ P_w @ r_k)

    H_data = 0.5 * (H_data + H_data.T)
    rho = float(np.sqrt(max(rho_sq, 0.0)))

    H_phiphi = H_data[:3, :3]
    H_phiv = H_data[:3, 3:]
    H_vphi = H_data[3:, :3]
    H_vv = H_data[3:, 3:]

    H_vv_sym = 0.5 * (H_vv + H_vv.T)

    # Robust Schur complement for conditional rotational information.
    try:
        H_vv_reg = H_vv_sym + schur_damping * np.eye(3)
        tmp = np.linalg.solve(H_vv_reg, H_vphi)
    except np.linalg.LinAlgError:
        tmp = np.linalg.pinv(H_vv_sym) @ H_vphi

    I_rot = H_phiphi - H_phiv @ tmp
    I_rot = 0.5 * (I_rot + I_rot.T)

    eig_I_rot = np.linalg.eigvalsh(I_rot)
    s_rot = float(np.min(eig_I_rot))

    H_blocks = {
        "H_phiphi": H_phiphi,
        "H_phiv": H_phiv,
        "H_vphi": H_vphi,
        "H_vv": H_vv,
    }

    return DataInformationResult(
        rho=rho,
        g_data=g_data,
        H_data=H_data,
        I_rot=I_rot,
        s_rot=s_rot,
        H_blocks=H_blocks,
    )


# ===============Algorithm I

## 完全闭式版
### 无论在什么样的初值设定下，alg2 最后的表现都没有超过initial的

In [9]:
# =========================
# Algorithm 1
# Single-Pass Closed-Form MFG Posterior Update
# =========================

from dataclasses import dataclass
from typing import Optional, Sequence
import numpy as np


# ---------------------------------------------------------
# Result container
# ---------------------------------------------------------

@dataclass
class Algorithm1Result:
    """
    Output of Algorithm 1:
    Single-pass closed-form MFG posterior update under a fixed nominal pose.

    Main outputs correspond to the paper algorithm:

        (Theta_post, X_A_star_batch, rho, s_rot, g_data, H_data)

    where

        rho    : whitened residual merit
        s_rot  : lambda_min(I_rot)
        g_data : data-induced local gradient in R^6
        H_data : data-induced Gauss--Newton curvature in R^{6x6}
    """

    # Main algorithm outputs
    Theta_post: "MFGParameters"
    chi_A_star: list
    rho: float
    s_rot: float
    I_rot: np.ndarray
    g_data: np.ndarray
    H_data: np.ndarray

    # Raw batch reduced-model result
    batch_result: "BatchReducedModelResult"

    # Nominal MFG rotational quantities
    # nu_bar: np.ndarray
    # J_nu: np.ndarray
    # e_bar: np.ndarray
    #（重新替换）
    # Prior-induced nominal MFG rotational quantities
    # These are used in the local prior expansion.
    nu_bar: np.ndarray
    J_nu: np.ndarray
    e_bar: np.ndarray

    # Posterior-induced self-consistent MFG coupling quantities
    # These are computed from F_post in the revised Theorem 4.3 closure.
    nu_bar_post: np.ndarray
    J_nu_post: np.ndarray




    # Prior gradient blocks
    g_prior_phi: np.ndarray
    g_prior_v: np.ndarray
    g_prior: np.ndarray

    # Prior curvature blocks
    H_prior_phiphi: np.ndarray
    H_prior_phiv: np.ndarray
    H_prior_vphi: np.ndarray
    H_prior_vv: np.ndarray
    H_prior: np.ndarray

    # Data gradient blocks
    g_data_phi: np.ndarray
    g_data_v: np.ndarray

    # Data curvature blocks
    H_data_phiphi: np.ndarray
    H_data_phiv: np.ndarray
    H_data_vphi: np.ndarray
    H_data_vv: np.ndarray

    # Total local posterior gradient blocks
    g_phi: np.ndarray
    g_v: np.ndarray
    g: np.ndarray

    # Total local posterior curvature blocks
    H_phiphi: np.ndarray
    H_phiv: np.ndarray
    H_vphi: np.ndarray
    H_vv: np.ndarray
    H: np.ndarray

    # Closed-form posterior auxiliary quantities
    M_post: np.ndarray
    S_post: np.ndarray
    a_post: np.ndarray

    # Diagnostics
    rho_sq: float
    rho_per_sample: np.ndarray
    eig_H_data: np.ndarray
    eig_I_rot: np.ndarray


# ---------------------------------------------------------
# Small linear-algebra helpers
# ---------------------------------------------------------

def sym3(A: np.ndarray) -> np.ndarray:
    """
    Symmetric part of a 3x3 matrix:
        sym(A) = 0.5 * (A + A^T).
    """
    A = ensure_matrix(A, (3, 3))
    return 0.5 * (A + A.T)


def symmetrize(A: np.ndarray) -> np.ndarray:
    """
    Symmetrize a square matrix.
    """
    A = np.asarray(A, dtype=float)
    return 0.5 * (A + A.T)


def symmetrize_with_jitter(A: np.ndarray, jitter: float = 1e-9) -> np.ndarray:
    """
    Symmetrize a matrix and add diagonal jitter if needed to make it numerically
    positive definite.

    This is used for precision matrices such as Lambda_post. It is not the PSD
    projection of the Matrix-Fisher curvature.
    """
    A = symmetrize(A)
    eigvals = np.linalg.eigvalsh(A)
    min_eig = float(np.min(eigvals))

    if min_eig <= jitter:
        A = A + (jitter - min_eig) * np.eye(A.shape[0])

    return A


def robust_inverse_3x3(A: np.ndarray, damping: float = 1e-12) -> np.ndarray:
    """
    Robust inverse / pseudo-inverse for a 3x3 matrix.
    """
    A = ensure_matrix(A, (3, 3))

    try:
        return np.linalg.inv(A)
    except np.linalg.LinAlgError:
        return np.linalg.pinv(A + damping * np.eye(3))


def robust_solve(A: np.ndarray, B: np.ndarray, damping: float = 1e-12) -> np.ndarray:
    """
    Robust linear solve A X = B.
    Falls back to least squares / pseudo-inverse if A is singular.
    """
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)

    try:
        return np.linalg.solve(A, B)
    except np.linalg.LinAlgError:
        A_reg = A + damping * np.eye(A.shape[0])
        return np.linalg.pinv(A_reg) @ B


def split_gradient_blocks(g: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Split g in R^6 into
        g = [g_phi; g_v].
    """
    g = ensure_vector(g, 6)
    return g[:3], g[3:]


def split_hessian_blocks_full(
    H: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Split H in R^{6x6} according to xi = [phi; v]:

        H = [[H_phiphi, H_phiv],
             [H_vphi,   H_vv  ]].
    """
    H = ensure_matrix(H, (6, 6))
    H_phiphi = H[:3, :3]
    H_phiv = H[:3, 3:]
    H_vphi = H[3:, :3]
    H_vv = H[3:, 3:]
    return H_phiphi, H_phiv, H_vphi, H_vv


# ---------------------------------------------------------
# Prior negative log-density and nominal prior quantities
# ---------------------------------------------------------

def mfg_negative_log_prior(
    X_B: "Pose",
    Theta_prior: "MFGParameters",
) -> float:
    """
    Negative log prior up to an additive constant:

        E_prior(R_B, p_B)
        = - tr(F^T R_B)
          + 0.5 ||p_B - (mu + Gamma nu_R)||_Lambda^2.
    """
    R_B = ensure_matrix(X_B.R, (3, 3))
    p_B = ensure_vector(X_B.p, 3)

    mu_c = conditional_translation_mean(R_B, Theta_prior)
    e = p_B - mu_c

    E_rot = -float(np.trace(Theta_prior.F.T @ R_B))
    E_trans = 0.5 * float(e.T @ Theta_prior.Lambda @ e)

    return E_rot + E_trans

### 新增一个通用函数 (revised 15,May)
def nu_bar_and_J_nu_from_F(
    F: np.ndarray,
    X_B_bar: "Pose",
    solver_config: "ImplicitEquilibriumSolverConfig",
) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute the MFG rotational deviation coordinate and its local Jacobian
    at the nominal rotation R_bar for an arbitrary Matrix--Fisher parameter F:

        nu_bar(F)
            = nu_R(R_bar; F),

        J_nu(F)
            = d / d phi nu_R(R_bar exp(hat(phi)); F) |_{phi=0}.

    The local perturbation convention is

        R_B(phi) = R_bar exp(hat(phi)).

    This function is used in two places:
        1. Prior local expansion, with F = F_prior;
        2. Self-consistent posterior closure, with F = F_post.
    """
    F = ensure_matrix(F, (3, 3))
    R_B_bar = ensure_matrix(X_B_bar.R, (3, 3))

    nu_bar = mfg_nu_R(R_B_bar, F)

    h = float(solver_config.fd_rot_step)
    J_nu = np.zeros((3, 3), dtype=float)

    for j in range(3):
        delta = np.zeros(6, dtype=float)
        delta[j] = h

        X_plus = apply_right_perturbation(X_B_bar, delta)
        X_minus = apply_right_perturbation(X_B_bar, -delta)

        nu_plus = mfg_nu_R(X_plus.R, F)
        nu_minus = mfg_nu_R(X_minus.R, F)

        J_nu[:, j] = (nu_plus - nu_minus) / (2.0 * h)

    return nu_bar, J_nu


def nominal_nu_bar_and_J_nu(
    Theta_prior: "MFGParameters",
    X_B_bar: "Pose",
    solver_config: "ImplicitEquilibriumSolverConfig",
) -> tuple[np.ndarray, np.ndarray]:
    """
    Backward-compatible wrapper for the prior-induced nominal quantities:

        nu_bar_prior = nu_R(R_bar; F_prior),
        J_nu_prior   = d nu_R / d phi at R_bar induced by F_prior.

    This function is kept unchanged at the interface level so that
    prior_quantities_at_X_B_bar(...) and the current Algorithm 1 code
    remain compatible.
    """
    return nu_bar_and_J_nu_from_F(
        F=Theta_prior.F,
        X_B_bar=X_B_bar,
        solver_config=solver_config,
    )



# def nominal_nu_bar_and_J_nu(
#     Theta_prior: "MFGParameters",
#     X_B_bar: "Pose",
#     solver_config: "ImplicitEquilibriumSolverConfig",
# ) -> tuple[np.ndarray, np.ndarray]:
#     """
#     Compute

#         nu_bar = nu_R evaluated at R_bar,
#         J_nu   = d nu_R / d phi evaluated at R_bar,

#     where the local right perturbation is

#         R_B(phi) = R_bar exp(hat(phi)).
#     """
#     R_B_bar = ensure_matrix(X_B_bar.R, (3, 3))
#     nu_bar = mfg_nu_R(R_B_bar, Theta_prior.F)

#     h = float(solver_config.fd_rot_step)
#     J_nu = np.zeros((3, 3), dtype=float)

#     for j in range(3):
#         delta = np.zeros(6, dtype=float)
#         delta[j] = h

#         X_plus = apply_right_perturbation(X_B_bar, delta)
#         X_minus = apply_right_perturbation(X_B_bar, -delta)

#         nu_plus = mfg_nu_R(X_plus.R, Theta_prior.F)
#         nu_minus = mfg_nu_R(X_minus.R, Theta_prior.F)

#         J_nu[:, j] = (nu_plus - nu_minus) / (2.0 * h)

#     return nu_bar, J_nu


def prior_quantities_at_X_B_bar(
    Theta_prior: "MFGParameters",
    X_B_bar: "Pose",
    solver_config: "ImplicitEquilibriumSolverConfig",
    use_psd_mf_curvature: bool = False,
) -> dict:
    """
    Compute the local prior gradient and curvature at the nominal pose X_B_bar.

    The prior energy is

        E_prior(R_B,p_B)
        = -tr(F^T R_B)
          + 0.5 ||p_B - (mu + Gamma nu_R)||_Lambda^2.

    Under xi = [phi; v], the local expansion is

        E_prior(X_bar oplus xi)
        =
        E_prior(X_bar)
        + g_prior^T xi
        + 0.5 xi^T H_prior xi
        + higher-order terms.

    By default, the Matrix-Fisher curvature is not PSD-projected, so that the
    code follows the exact second-order expansion used in the paper.
    """
    F = ensure_matrix(Theta_prior.F, (3, 3))
    mu = ensure_vector(Theta_prior.mu, 3)
    Lambda = ensure_matrix(Theta_prior.Lambda, (3, 3))
    Gamma = ensure_matrix(Theta_prior.Gamma, (3, 3))

    R_B_bar = ensure_matrix(X_B_bar.R, (3, 3))
    p_B_bar = ensure_vector(X_B_bar.p, 3)

    # nu_bar and J_nu
    nu_bar, J_nu = nominal_nu_bar_and_J_nu(
        Theta_prior=Theta_prior,
        X_B_bar=X_B_bar,
        solver_config=solver_config,
    )

    # e_bar = p_bar - (mu + Gamma nu_bar)
    e_bar = p_B_bar - (mu + Gamma @ nu_bar)

    # -------------------------
    # Gradient blocks
    # -------------------------
    g_prior_v = Lambda @ e_bar

    # Matrix-Fisher rotation gradient:
    # g_phi^(MF) = - (R_bar^T F - F^T R_bar)^vee
    g_prior_phi_mf = -vee(R_B_bar.T @ F - F.T @ R_B_bar)

    # Coupling-induced rotation gradient:
    # g_phi^(cpl) = - J_nu^T Gamma^T Lambda e_bar
    g_prior_phi_cpl = -J_nu.T @ Gamma.T @ Lambda @ e_bar

    g_prior_phi = g_prior_phi_mf + g_prior_phi_cpl

    # -------------------------
    # Curvature blocks
    # -------------------------
    H_prior_vv = symmetrize(Lambda)

    # H_phi_v = - J_nu^T Gamma^T Lambda
    H_prior_phiv = -J_nu.T @ Gamma.T @ Lambda
    H_prior_vphi = H_prior_phiv.T

    # Exact Matrix-Fisher local curvature:
    # H_phi_phi^(MF) = tr(F^T R_bar) I - sym(F^T R_bar)
    A = F.T @ R_B_bar
    H_prior_phiphi_mf = np.trace(A) * np.eye(3) - sym3(A)

    # Optional numerical PSD projection.
    # Keep False for the current theoretical version of the paper.
    if use_psd_mf_curvature:
        evals, evecs = np.linalg.eigh(symmetrize(H_prior_phiphi_mf))
        evals = np.maximum(evals, 0.0)
        H_prior_phiphi_mf = evecs @ np.diag(evals) @ evecs.T

    # Coupling term:
    # H_phi_phi^(cpl) = J_nu^T Gamma^T Lambda Gamma J_nu
    H_prior_phiphi_cpl = J_nu.T @ Gamma.T @ Lambda @ Gamma @ J_nu

    H_prior_phiphi = H_prior_phiphi_mf + H_prior_phiphi_cpl
    H_prior_phiphi = symmetrize(H_prior_phiphi)

    # Assemble full prior gradient and curvature
    g_prior = np.concatenate([g_prior_phi, g_prior_v])

    H_prior = np.block([
        [H_prior_phiphi, H_prior_phiv],
        [H_prior_vphi,   H_prior_vv],
    ])
    H_prior = symmetrize(H_prior)

    return {
        "nu_bar": nu_bar,
        "J_nu": J_nu,
        "e_bar": e_bar,

        "g_prior": g_prior,
        "H_prior": H_prior,

        "g_prior_phi": g_prior_phi,
        "g_prior_v": g_prior_v,

        "H_prior_phiphi": H_prior_phiphi,
        "H_prior_phiv": H_prior_phiv,
        "H_prior_vphi": H_prior_vphi,
        "H_prior_vv": H_prior_vv,

        # Diagnostics
        "g_prior_phi_mf": g_prior_phi_mf,
        "g_prior_phi_cpl": g_prior_phi_cpl,
        "H_prior_phiphi_mf": H_prior_phiphi_mf,
        "H_prior_phiphi_cpl": H_prior_phiphi_cpl,
    }


# ---------------------------------------------------------
# Data information and conditional rotational information
# ---------------------------------------------------------

def conditional_rotational_information_from_Hdata(
    H_data_phiphi: np.ndarray,
    H_data_phiv: np.ndarray,
    H_data_vphi: np.ndarray,
    H_data_vv: np.ndarray,
    damping: float = 1e-10,
) -> tuple[np.ndarray, float, np.ndarray]:
    """
    Compute the conditional rotational information matrix

        I_rot = H_phiphi - H_phiv H_vv^{-1} H_vphi.

    This is the Schur-complement rotational information after translational
    perturbations have been optimized out.
    """
    H_data_phiphi = ensure_matrix(H_data_phiphi, (3, 3))
    H_data_phiv = ensure_matrix(H_data_phiv, (3, 3))
    H_data_vphi = ensure_matrix(H_data_vphi, (3, 3))
    H_data_vv = ensure_matrix(H_data_vv, (3, 3))

    H_vv_sym = symmetrize(H_data_vv)
    H_vv_reg = H_vv_sym + damping * np.eye(3)

    try:
        tmp = np.linalg.solve(H_vv_reg, H_data_vphi)
    except np.linalg.LinAlgError:
        tmp = np.linalg.pinv(H_vv_sym) @ H_data_vphi

    I_rot = H_data_phiphi - H_data_phiv @ tmp
    I_rot = symmetrize(I_rot)

    eig_I_rot = np.linalg.eigvalsh(I_rot)
    s_rot = float(np.min(eig_I_rot))

    return I_rot, s_rot, eig_I_rot


def compute_data_information_from_batch(
    batch_result: "BatchReducedModelResult",
    P_w: np.ndarray,
) -> dict:
    """
    Compute data gradient, Gauss--Newton data curvature, residual merit,
    and conditional rotational information from a batch reduced-model result.

    Residual convention:
        r_k = y_k - y_hat_k,

    and local expansion:
        r_k(X_bar oplus xi) = r_k(X_bar) + J_k xi + O(||xi||^2).

    Therefore:
        g_data = sum_k J_k^T Sigma_w^{-1} r_k,
        H_data = sum_k J_k^T Sigma_w^{-1} J_k.
    """
    P_w = ensure_matrix(P_w, (6, 6))

    K = batch_result.residuals.shape[0]

    g_data_phi = np.zeros(3, dtype=float)
    g_data_v = np.zeros(3, dtype=float)

    H_data_phiphi = np.zeros((3, 3), dtype=float)
    H_data_phiv = np.zeros((3, 3), dtype=float)
    H_data_vphi = np.zeros((3, 3), dtype=float)
    H_data_vv = np.zeros((3, 3), dtype=float)

    rho_sq = 0.0
    rho_per_sample = np.zeros(K, dtype=float)

    for k in range(K):
        r_k = ensure_vector(batch_result.residuals[k], 6)
        J_k_phi = ensure_matrix(batch_result.J_phi[k], (6, 3))
        J_k_v = ensure_matrix(batch_result.J_v[k], (6, 3))

        # Data gradient blocks
        g_data_phi += J_k_phi.T @ P_w @ r_k
        g_data_v += J_k_v.T @ P_w @ r_k

        # Data Gauss--Newton curvature blocks
        H_data_phiphi += J_k_phi.T @ P_w @ J_k_phi
        H_data_phiv += J_k_phi.T @ P_w @ J_k_v
        H_data_vphi += J_k_v.T @ P_w @ J_k_phi
        H_data_vv += J_k_v.T @ P_w @ J_k_v

        rho_k_sq = float(r_k.T @ P_w @ r_k)
        rho_per_sample[k] = np.sqrt(max(rho_k_sq, 0.0))
        rho_sq += rho_k_sq

    rho = float(np.sqrt(max(rho_sq, 0.0)))

    g_data = np.concatenate([g_data_phi, g_data_v])

    H_data = np.block([
        [H_data_phiphi, H_data_phiv],
        [H_data_vphi,   H_data_vv],
    ])
    H_data = symmetrize(H_data)

    # Re-split after symmetrization for consistency
    H_data_phiphi, H_data_phiv, H_data_vphi, H_data_vv = split_hessian_blocks_full(H_data)

    I_rot, s_rot, eig_I_rot = conditional_rotational_information_from_Hdata(
        H_data_phiphi=H_data_phiphi,
        H_data_phiv=H_data_phiv,
        H_data_vphi=H_data_vphi,
        H_data_vv=H_data_vv,
    )

    eig_H_data = np.linalg.eigvalsh(H_data)

    return {
        "rho": rho,
        "rho_sq": rho_sq,
        "rho_per_sample": rho_per_sample,

        "g_data": g_data,
        "H_data": H_data,

        "g_data_phi": g_data_phi,
        "g_data_v": g_data_v,

        "H_data_phiphi": H_data_phiphi,
        "H_data_phiv": H_data_phiv,
        "H_data_vphi": H_data_vphi,
        "H_data_vv": H_data_vv,

        "I_rot": I_rot,
        "s_rot": s_rot,
        "eig_I_rot": eig_I_rot,
        "eig_H_data": eig_H_data,
    }


# ---------------------------------------------------------
# Closed-form posterior update （revised for coupled MFG posteriors : 15, May）
# ---------------------------------------------------------

## 重写
def closed_form_mfg_posterior_update(
    Theta_prior: "MFGParameters",
    X_B_bar: "Pose",
    nu_bar: np.ndarray,
    J_nu: np.ndarray,
    g_phi: np.ndarray,
    g_v: np.ndarray,
    H_phiphi: np.ndarray,
    H_vphi: np.ndarray,
    H_vv: np.ndarray,
    *,
    solver_config: Optional["ImplicitEquilibriumSolverConfig"] = None,
    precision_jitter: float = 1e-9,
) -> dict:
    """
    Self-consistent closed-form coupled MFG posterior closure.

    Given the local quadratic posterior model

        Phi(xi) = const + g^T xi + 0.5 xi^T H xi,

    with xi = [phi; v], this constructs

        Theta_post = (F_post, mu_post, Lambda_post, Gamma_post)

    in the following explicit order:

        1. Lambda_post from H_vv;
        2. Schur-complement rotational terms
               a_post, M_post;
        3. F_post from the remaining rotational gradient and curvature;
        4. Posterior-induced
               nu_bar_post, J_nu_post
           computed from F_post;
        5. Gamma_post and mu_post from the self-consistent
           posterior coupling coordinate.

    Notes
    -----
    The arguments Theta_prior, nu_bar, and J_nu are retained only to preserve
    compatibility with the existing Algorithm 1 call structure. They are not
    used in the self-consistent posterior closure itself.
    """
    if solver_config is None:
        solver_config = ImplicitEquilibriumSolverConfig()

    R_B_bar = ensure_matrix(X_B_bar.R, (3, 3))
    p_B_bar = ensure_vector(X_B_bar.p, 3)

    # Retained for backward compatibility / validation of the old interface.
    _ = Theta_prior
    _ = ensure_vector(nu_bar, 3)
    _ = ensure_matrix(J_nu, (3, 3))

    g_phi = ensure_vector(g_phi, 3)
    g_v = ensure_vector(g_v, 3)

    H_phiphi = ensure_matrix(H_phiphi, (3, 3))
    H_vphi = ensure_matrix(H_vphi, (3, 3))
    H_vv = ensure_matrix(H_vv, (3, 3))

    # -----------------------------------------------------
    # 1. Translational posterior precision
    # -----------------------------------------------------
    Lambda_post = symmetrize_with_jitter(
        H_vv,
        jitter=precision_jitter,
    )

    # Since the assembled posterior Hessian is symmetrized upstream,
    # H_phiv = H_vphi^T.
    H_phiv = H_vphi.T

    # Reusable linear solves
    Lambda_inv_gv = robust_solve(
        Lambda_post,
        g_v,
        damping=precision_jitter,
    )

    Lambda_inv_Hvphi = robust_solve(
        Lambda_post,
        H_vphi,
        damping=precision_jitter,
    )

    # -----------------------------------------------------
    # 2. Schur-complement rotational matching terms
    # -----------------------------------------------------
    # a_post = g_phi - H_phiv H_vv^{-1} g_v
    a_post = g_phi - H_phiv @ Lambda_inv_gv

    # M_post = H_phiphi - H_phiv H_vv^{-1} H_vphi
    M_post = H_phiphi - H_phiv @ Lambda_inv_Hvphi
    M_post = symmetrize(M_post)

    # NOTE:
    # In the revised theorem this quantity may be denoted C_post
    # to avoid conflict with the SVD singular-value matrix.
    # The code keeps the legacy name S_post for compatibility.
    S_post = 0.5 * np.trace(M_post) * np.eye(3) - M_post

    # -----------------------------------------------------
    # 3. Closed-form Matrix--Fisher posterior parameter
    # -----------------------------------------------------
    F_post = R_B_bar @ (S_post - 0.5 * hat(a_post))

    # -----------------------------------------------------
    # 4. Posterior-induced coupling coordinate
    # -----------------------------------------------------
    nu_bar_post, J_nu_post = nu_bar_and_J_nu_from_F(
        F=F_post,
        X_B_bar=X_B_bar,
        solver_config=solver_config,
    )

    J_nu_post_inv = robust_inverse_3x3(J_nu_post)

    # -----------------------------------------------------
    # 5. Coupling matrix and base translational mean
    # -----------------------------------------------------
    # Gamma_post = - H_vv^{-1} H_vphi J_{nu,post}^{-1}
    Gamma_post = -robust_solve(
        Lambda_post,
        H_vphi @ J_nu_post_inv,
        damping=precision_jitter,
    )

    # mu_post = p_bar - H_vv^{-1} g_v - Gamma_post nu_bar_post
    mu_post = (
        p_B_bar
        - Lambda_inv_gv
        - Gamma_post @ nu_bar_post
    )

    Theta_post = MFGParameters(
        F=F_post,
        mu=mu_post,
        Lambda=Lambda_post,
        Gamma=Gamma_post,
    )

    return {
        "Theta_post": Theta_post,

        "Lambda_post": Lambda_post,
        "Gamma_post": Gamma_post,
        "mu_post": mu_post,

        "a_post": a_post,
        "M_post": M_post,
        "S_post": S_post,
        "F_post": F_post,

        # New posterior-induced self-consistent coupling quantities
        "nu_bar_post": nu_bar_post,
        "J_nu_post": J_nu_post,
    }

# def closed_form_mfg_posterior_update(
#     Theta_prior: "MFGParameters",
#     X_B_bar: "Pose",
#     nu_bar: np.ndarray,
#     J_nu: np.ndarray,
#     g_phi: np.ndarray,
#     g_v: np.ndarray,
#     H_phiphi: np.ndarray,
#     H_vphi: np.ndarray,
#     H_vv: np.ndarray,
#     *,
#     precision_jitter: float = 1e-9,
# ) -> dict:
#     """
#     Closed-form coupled MFG posterior update.

#     Given the local quadratic posterior model

#         Phi(xi) = const + g^T xi + 0.5 xi^T H xi,

#     with xi = [phi; v], this constructs MFG parameters

#         Theta_post = (F_post, mu_post, Lambda_post, Gamma_post)

#     matching the local second-order information structure.
#     """
#     R_B_bar = ensure_matrix(X_B_bar.R, (3, 3))
#     p_B_bar = ensure_vector(X_B_bar.p, 3)

#     nu_bar = ensure_vector(nu_bar, 3)
#     J_nu = ensure_matrix(J_nu, (3, 3))

#     g_phi = ensure_vector(g_phi, 3)
#     g_v = ensure_vector(g_v, 3)

#     H_phiphi = ensure_matrix(H_phiphi, (3, 3))
#     H_vphi = ensure_matrix(H_vphi, (3, 3))
#     H_vv = ensure_matrix(H_vv, (3, 3))

#     # Translation precision of the posterior
#     Lambda_post = symmetrize_with_jitter(H_vv, jitter=precision_jitter)

#     # Coupling matrix
#     J_nu_inv = robust_inverse_3x3(J_nu)
#     Gamma_post = -robust_solve(
#         Lambda_post,
#         H_vphi @ J_nu_inv,
#         damping=precision_jitter,
#     )

#     # Base translation mean
#     mu_post = (
#         p_B_bar
#         - robust_solve(Lambda_post, g_v, damping=precision_jitter)
#         - Gamma_post @ nu_bar
#     )

#     # Rotational first-order matching term
#     a_post = g_phi + J_nu.T @ Gamma_post.T @ g_v

#     # Rotational second-order matching term
#     M_post = (
#         H_phiphi
#         - J_nu.T @ Gamma_post.T @ Lambda_post @ Gamma_post @ J_nu
#     )
#     M_post = symmetrize(M_post)

#     S_post = 0.5 * np.trace(M_post) * np.eye(3) - M_post

#     F_post = R_B_bar @ (S_post - 0.5 * hat(a_post))

#     Theta_post = MFGParameters(
#         F=F_post,
#         mu=mu_post,
#         Lambda=Lambda_post,
#         Gamma=Gamma_post,
#     )

#     return {
#         "Theta_post": Theta_post,
#         "Lambda_post": Lambda_post,
#         "Gamma_post": Gamma_post,
#         "mu_post": mu_post,
#         "a_post": a_post,
#         "M_post": M_post,
#         "S_post": S_post,
#         "F_post": F_post,
#     }


# ---------------------------------------------------------
# Main Algorithm 1 interface
# ---------------------------------------------------------

def algorithm1_single_pass_local_mfg_posterior_update(
    Theta_prior: "MFGParameters",
    X_B_bar: "Pose",
    U: Sequence["Pose"],
    Y: Sequence[np.ndarray],
    field_params: "SuperquadricFieldParameters",
    stiffness_params: "StiffnessParameters",
    ctrl_params: "ControlPotentialParameters",
    *,
    Sigma_w: Optional[np.ndarray] = None,
    Sigma_w_inv: Optional[np.ndarray] = None,
    contact_points_A_shared=None,
    contact_points_A_per_sample=None,
    solver_config: Optional["ImplicitEquilibriumSolverConfig"] = None,
    chi_A_0: Optional[Sequence[Optional["Pose"]]] = None,
    prev_outer_chi_A_star: Optional[Sequence[Optional["Pose"]]] = None,
    use_psd_mf_curvature: bool = False,
) -> Algorithm1Result:
    """
    Algorithm 1:
    Single-pass closed-form MFG posterior update under a fixed nominal pose.

    Inputs
    ------
    Theta_prior : MFGParameters
        Prior parameters Theta_prior = (F, mu, Lambda, Gamma).
    X_B_bar : Pose
        Nominal object pose.
    U : sequence of Pose
        Control batch {X_{U,k}}.
    Y : sequence of ndarray
        Measurement batch {y_k}, each y_k in R^6.
    Sigma_w or Sigma_w_inv : ndarray
        Wrench noise covariance or precision.
    chi_A_0 : optional sequence of Pose
        Equilibrium warm starts.
    prev_outer_chi_A_star : optional sequence of Pose
        Previous outer-iteration equilibrium states.

    Returns
    -------
    Algorithm1Result
        Contains
            Theta_post,
            chi_A_star,
            rho,
            s_rot,
            g_data,
            H_data,
        plus block-level diagnostics.
    """
    if solver_config is None:
        solver_config = ImplicitEquilibriumSolverConfig()

    K = len(U)
    if len(Y) != K:
        raise ValueError(f"Length mismatch: len(U)={K}, but len(Y)={len(Y)}.")

    # -----------------------------------------------------
    # Step 1: prior quantities at the nominal pose
    # -----------------------------------------------------
    prior_q = prior_quantities_at_X_B_bar(
        Theta_prior=Theta_prior,
        X_B_bar=X_B_bar,
        solver_config=solver_config,
        use_psd_mf_curvature=use_psd_mf_curvature,
    )

    nu_bar = prior_q["nu_bar"]
    J_nu = prior_q["J_nu"]
    e_bar = prior_q["e_bar"]

    g_prior = prior_q["g_prior"]
    H_prior = prior_q["H_prior"]

    g_prior_phi = prior_q["g_prior_phi"]
    g_prior_v = prior_q["g_prior_v"]

    H_prior_phiphi = prior_q["H_prior_phiphi"]
    H_prior_phiv = prior_q["H_prior_phiv"]
    H_prior_vphi = prior_q["H_prior_vphi"]
    H_prior_vv = prior_q["H_prior_vv"]

    # -----------------------------------------------------
    # Step 2-7: evaluate reduced residuals and Jacobians
    # -----------------------------------------------------
    batch_result = evaluate_batch_reduced_model(
        X_B=X_B_bar,
        X_U_list=U,
        y_list=Y,
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        contact_points_A_shared=contact_points_A_shared,
        contact_points_A_per_sample=contact_points_A_per_sample,
        solver_config=solver_config,
        Sigma_w=Sigma_w,
        Sigma_w_inv=Sigma_w_inv,
        explicit_XA_inits=chi_A_0,
        prev_outer_XA_stars=prev_outer_chi_A_star,
    )

    # -----------------------------------------------------
    # Step 8: data gradient, data curvature, residual merit,
    # and rotational information score
    # -----------------------------------------------------
    P_w = prepare_measurement_precision(
        Sigma_w=Sigma_w,
        Sigma_w_inv=Sigma_w_inv,
    )

    data_q = compute_data_information_from_batch(
        batch_result=batch_result,
        P_w=P_w,
    )

    rho = data_q["rho"]
    rho_sq = data_q["rho_sq"]
    rho_per_sample = data_q["rho_per_sample"]

    g_data = data_q["g_data"]
    H_data = data_q["H_data"]

    g_data_phi = data_q["g_data_phi"]
    g_data_v = data_q["g_data_v"]

    H_data_phiphi = data_q["H_data_phiphi"]
    H_data_phiv = data_q["H_data_phiv"]
    H_data_vphi = data_q["H_data_vphi"]
    H_data_vv = data_q["H_data_vv"]

    I_rot = data_q["I_rot"]
    s_rot = data_q["s_rot"]
    eig_I_rot = data_q["eig_I_rot"]
    eig_H_data = data_q["eig_H_data"]

    # -----------------------------------------------------
    # Step 9: local posterior information pair
    # -----------------------------------------------------
    g = g_prior + g_data
    H = H_prior + H_data
    H = symmetrize(H)

    g_phi, g_v = split_gradient_blocks(g)
    H_phiphi, H_phiv, H_vphi, H_vv = split_hessian_blocks_full(H)

    # -----------------------------------------------------
    # Step 10: closed-form coupled MFG posterior update
    # -----------------------------------------------------
    # post_q = closed_form_mfg_posterior_update(
    #     Theta_prior=Theta_prior,
    #     X_B_bar=X_B_bar,
    #     nu_bar=nu_bar,
    #     J_nu=J_nu,
    #     g_phi=g_phi,
    #     g_v=g_v,
    #     H_phiphi=H_phiphi,
    #     H_vphi=H_vphi,
    #     H_vv=H_vv,
    # )

    post_q = closed_form_mfg_posterior_update(
        Theta_prior=Theta_prior,
        X_B_bar=X_B_bar,
        nu_bar=nu_bar,
        J_nu=J_nu,
        g_phi=g_phi,
        g_v=g_v,
        H_phiphi=H_phiphi,
        H_vphi=H_vphi,
        H_vv=H_vv,
        solver_config=solver_config,
    )

    Theta_post = post_q["Theta_post"]

    # -----------------------------------------------------
    # Return
    # -----------------------------------------------------
    return Algorithm1Result(
        Theta_post=Theta_post,
        chi_A_star=batch_result.X_A_stars,

        rho=rho,
        s_rot=s_rot,
        I_rot=I_rot,
        g_data=g_data,
        H_data=H_data,

        batch_result=batch_result,

        # nu_bar=nu_bar,
        # J_nu=J_nu,
        # e_bar=e_bar, （被重新替换）
        nu_bar=nu_bar,
        J_nu=J_nu,
        e_bar=e_bar,

        nu_bar_post=post_q["nu_bar_post"],
        J_nu_post=post_q["J_nu_post"],

        g_prior_phi=g_prior_phi,
        g_prior_v=g_prior_v,
        g_prior=g_prior,

        H_prior_phiphi=H_prior_phiphi,
        H_prior_phiv=H_prior_phiv,
        H_prior_vphi=H_prior_vphi,
        H_prior_vv=H_prior_vv,
        H_prior=H_prior,

        g_data_phi=g_data_phi,
        g_data_v=g_data_v,

        H_data_phiphi=H_data_phiphi,
        H_data_phiv=H_data_phiv,
        H_data_vphi=H_data_vphi,
        H_data_vv=H_data_vv,

        g_phi=g_phi,
        g_v=g_v,
        g=g,

        H_phiphi=H_phiphi,
        H_phiv=H_phiv,
        H_vphi=H_vphi,
        H_vv=H_vv,
        H=H,

        M_post=post_q["M_post"],
        S_post=post_q["S_post"],
        a_post=post_q["a_post"],

        rho_sq=rho_sq,
        rho_per_sample=rho_per_sample,
        eig_H_data=eig_H_data,
        eig_I_rot=eig_I_rot,
    )


# ---------------------------------------------------------

## 冻结版 （当前用的）

In [19]:
# =========================
# Algorithm 1
# Single-Pass Closed-Form MFG Posterior Update
# =========================

from dataclasses import dataclass
from typing import Optional, Sequence
import numpy as np


# ---------------------------------------------------------
# Result container
# ---------------------------------------------------------

@dataclass
class Algorithm1Result:
    """
    Output of Algorithm 1:
    Single-pass closed-form MFG posterior update under a fixed nominal pose.

    Main outputs correspond to the paper algorithm:

        (Theta_post, X_A_star_batch, rho, s_rot, g_data, H_data)

    where

        rho    : whitened residual merit
        s_rot  : lambda_min(I_rot)
        g_data : data-induced local gradient in R^6
        H_data : data-induced Gauss--Newton curvature in R^{6x6}
    """

    # Main algorithm outputs
    Theta_post: "MFGParameters"
    chi_A_star: list
    rho: float
    s_rot: float
    I_rot: np.ndarray
    g_data: np.ndarray
    H_data: np.ndarray

    # Raw batch reduced-model result
    batch_result: "BatchReducedModelResult"

    # Nominal MFG rotational quantities
    nu_bar: np.ndarray
    J_nu: np.ndarray
    e_bar: np.ndarray

    # Prior gradient blocks
    g_prior_phi: np.ndarray
    g_prior_v: np.ndarray
    g_prior: np.ndarray

    # Prior curvature blocks
    H_prior_phiphi: np.ndarray
    H_prior_phiv: np.ndarray
    H_prior_vphi: np.ndarray
    H_prior_vv: np.ndarray
    H_prior: np.ndarray

    # Data gradient blocks
    g_data_phi: np.ndarray
    g_data_v: np.ndarray

    # Data curvature blocks
    H_data_phiphi: np.ndarray
    H_data_phiv: np.ndarray
    H_data_vphi: np.ndarray
    H_data_vv: np.ndarray

    # Total local posterior gradient blocks
    g_phi: np.ndarray
    g_v: np.ndarray
    g: np.ndarray

    # Total local posterior curvature blocks
    H_phiphi: np.ndarray
    H_phiv: np.ndarray
    H_vphi: np.ndarray
    H_vv: np.ndarray
    H: np.ndarray

    # Closed-form posterior auxiliary quantities
    M_post: np.ndarray
    S_post: np.ndarray
    a_post: np.ndarray

    # Diagnostics
    rho_sq: float
    rho_per_sample: np.ndarray
    eig_H_data: np.ndarray
    eig_I_rot: np.ndarray


# ---------------------------------------------------------
# Small linear-algebra helpers
# ---------------------------------------------------------

def sym3(A: np.ndarray) -> np.ndarray:
    """
    Symmetric part of a 3x3 matrix:
        sym(A) = 0.5 * (A + A^T).
    """
    A = ensure_matrix(A, (3, 3))
    return 0.5 * (A + A.T)


def symmetrize(A: np.ndarray) -> np.ndarray:
    """
    Symmetrize a square matrix.
    """
    A = np.asarray(A, dtype=float)
    return 0.5 * (A + A.T)


def symmetrize_with_jitter(A: np.ndarray, jitter: float = 1e-9) -> np.ndarray:
    """
    Symmetrize a matrix and add diagonal jitter if needed to make it numerically
    positive definite.

    This is used for precision matrices such as Lambda_post. It is not the PSD
    projection of the Matrix-Fisher curvature.
    """
    A = symmetrize(A)
    eigvals = np.linalg.eigvalsh(A)
    min_eig = float(np.min(eigvals))

    if min_eig <= jitter:
        A = A + (jitter - min_eig) * np.eye(A.shape[0])

    return A


def robust_inverse_3x3(A: np.ndarray, damping: float = 1e-12) -> np.ndarray:
    """
    Robust inverse / pseudo-inverse for a 3x3 matrix.
    """
    A = ensure_matrix(A, (3, 3))

    try:
        return np.linalg.inv(A)
    except np.linalg.LinAlgError:
        return np.linalg.pinv(A + damping * np.eye(3))


def robust_solve(A: np.ndarray, B: np.ndarray, damping: float = 1e-12) -> np.ndarray:
    """
    Robust linear solve A X = B.
    Falls back to least squares / pseudo-inverse if A is singular.
    """
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)

    try:
        return np.linalg.solve(A, B)
    except np.linalg.LinAlgError:
        A_reg = A + damping * np.eye(A.shape[0])
        return np.linalg.pinv(A_reg) @ B


def split_gradient_blocks(g: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Split g in R^6 into
        g = [g_phi; g_v].
    """
    g = ensure_vector(g, 6)
    return g[:3], g[3:]


def split_hessian_blocks_full(
    H: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Split H in R^{6x6} according to xi = [phi; v]:

        H = [[H_phiphi, H_phiv],
             [H_vphi,   H_vv  ]].
    """
    H = ensure_matrix(H, (6, 6))
    H_phiphi = H[:3, :3]
    H_phiv = H[:3, 3:]
    H_vphi = H[3:, :3]
    H_vv = H[3:, 3:]
    return H_phiphi, H_phiv, H_vphi, H_vv


# ---------------------------------------------------------
# Prior negative log-density and nominal prior quantities
# ---------------------------------------------------------

def mfg_negative_log_prior(
    X_B: "Pose",
    Theta_prior: "MFGParameters",
) -> float:
    """
    Negative log prior up to an additive constant:

        E_prior(R_B, p_B)
        = - tr(F^T R_B)
          + 0.5 ||p_B - (mu + Gamma nu_R)||_Lambda^2.
    """
    R_B = ensure_matrix(X_B.R, (3, 3))
    p_B = ensure_vector(X_B.p, 3)

    mu_c = conditional_translation_mean(R_B, Theta_prior)
    e = p_B - mu_c

    E_rot = -float(np.trace(Theta_prior.F.T @ R_B))
    E_trans = 0.5 * float(e.T @ Theta_prior.Lambda @ e)

    return E_rot + E_trans


def nominal_nu_bar_and_J_nu(
    Theta_prior: "MFGParameters",
    X_B_bar: "Pose",
    solver_config: "ImplicitEquilibriumSolverConfig",
) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute

        nu_bar = nu_R evaluated at R_bar,
        J_nu   = d nu_R / d phi evaluated at R_bar,

    where the local right perturbation is

        R_B(phi) = R_bar exp(hat(phi)).
    """
    R_B_bar = ensure_matrix(X_B_bar.R, (3, 3))
    nu_bar = mfg_nu_R(R_B_bar, Theta_prior.F)

    h = float(solver_config.fd_rot_step)
    J_nu = np.zeros((3, 3), dtype=float)

    for j in range(3):
        delta = np.zeros(6, dtype=float)
        delta[j] = h

        X_plus = apply_right_perturbation(X_B_bar, delta)
        X_minus = apply_right_perturbation(X_B_bar, -delta)

        nu_plus = mfg_nu_R(X_plus.R, Theta_prior.F)
        nu_minus = mfg_nu_R(X_minus.R, Theta_prior.F)

        J_nu[:, j] = (nu_plus - nu_minus) / (2.0 * h)

    return nu_bar, J_nu


def prior_quantities_at_X_B_bar(
    Theta_prior: "MFGParameters",
    X_B_bar: "Pose",
    solver_config: "ImplicitEquilibriumSolverConfig",
    use_psd_mf_curvature: bool = False,
) -> dict:
    """
    Compute the local prior gradient and curvature at the nominal pose X_B_bar.

    The prior energy is

        E_prior(R_B,p_B)
        = -tr(F^T R_B)
          + 0.5 ||p_B - (mu + Gamma nu_R)||_Lambda^2.

    Under xi = [phi; v], the local expansion is

        E_prior(X_bar oplus xi)
        =
        E_prior(X_bar)
        + g_prior^T xi
        + 0.5 xi^T H_prior xi
        + higher-order terms.

    By default, the Matrix-Fisher curvature is not PSD-projected, so that the
    code follows the exact second-order expansion used in the paper.
    """
    F = ensure_matrix(Theta_prior.F, (3, 3))
    mu = ensure_vector(Theta_prior.mu, 3)
    Lambda = ensure_matrix(Theta_prior.Lambda, (3, 3))
    Gamma = ensure_matrix(Theta_prior.Gamma, (3, 3))

    R_B_bar = ensure_matrix(X_B_bar.R, (3, 3))
    p_B_bar = ensure_vector(X_B_bar.p, 3)

    # nu_bar and J_nu
    nu_bar, J_nu = nominal_nu_bar_and_J_nu(
        Theta_prior=Theta_prior,
        X_B_bar=X_B_bar,
        solver_config=solver_config,
    )

    # e_bar = p_bar - (mu + Gamma nu_bar)
    e_bar = p_B_bar - (mu + Gamma @ nu_bar)

    # -------------------------
    # Gradient blocks
    # -------------------------
    g_prior_v = Lambda @ e_bar

    # Matrix-Fisher rotation gradient:
    # g_phi^(MF) = - (R_bar^T F - F^T R_bar)^vee
    g_prior_phi_mf = -vee(R_B_bar.T @ F - F.T @ R_B_bar)

    # Coupling-induced rotation gradient:
    # g_phi^(cpl) = - J_nu^T Gamma^T Lambda e_bar
    g_prior_phi_cpl = -J_nu.T @ Gamma.T @ Lambda @ e_bar

    g_prior_phi = g_prior_phi_mf + g_prior_phi_cpl

    # -------------------------
    # Curvature blocks
    # -------------------------
    H_prior_vv = symmetrize(Lambda)

    # H_phi_v = - J_nu^T Gamma^T Lambda
    H_prior_phiv = -J_nu.T @ Gamma.T @ Lambda
    H_prior_vphi = H_prior_phiv.T

    # Exact Matrix-Fisher local curvature:
    # H_phi_phi^(MF) = tr(F^T R_bar) I - sym(F^T R_bar)
    A = F.T @ R_B_bar
    H_prior_phiphi_mf = np.trace(A) * np.eye(3) - sym3(A)

    # Optional numerical PSD projection.
    # Keep False for the current theoretical version of the paper.
    if use_psd_mf_curvature:
        evals, evecs = np.linalg.eigh(symmetrize(H_prior_phiphi_mf))
        evals = np.maximum(evals, 0.0)
        H_prior_phiphi_mf = evecs @ np.diag(evals) @ evecs.T

    # Coupling term:
    # H_phi_phi^(cpl) = J_nu^T Gamma^T Lambda Gamma J_nu
    H_prior_phiphi_cpl = J_nu.T @ Gamma.T @ Lambda @ Gamma @ J_nu

    H_prior_phiphi = H_prior_phiphi_mf + H_prior_phiphi_cpl
    H_prior_phiphi = symmetrize(H_prior_phiphi)

    # Assemble full prior gradient and curvature
    g_prior = np.concatenate([g_prior_phi, g_prior_v])

    H_prior = np.block([
        [H_prior_phiphi, H_prior_phiv],
        [H_prior_vphi,   H_prior_vv],
    ])
    H_prior = symmetrize(H_prior)

    return {
        "nu_bar": nu_bar,
        "J_nu": J_nu,
        "e_bar": e_bar,

        "g_prior": g_prior,
        "H_prior": H_prior,

        "g_prior_phi": g_prior_phi,
        "g_prior_v": g_prior_v,

        "H_prior_phiphi": H_prior_phiphi,
        "H_prior_phiv": H_prior_phiv,
        "H_prior_vphi": H_prior_vphi,
        "H_prior_vv": H_prior_vv,

        # Diagnostics
        "g_prior_phi_mf": g_prior_phi_mf,
        "g_prior_phi_cpl": g_prior_phi_cpl,
        "H_prior_phiphi_mf": H_prior_phiphi_mf,
        "H_prior_phiphi_cpl": H_prior_phiphi_cpl,
    }


# ---------------------------------------------------------
# Data information and conditional rotational information
# ---------------------------------------------------------

def conditional_rotational_information_from_Hdata(
    H_data_phiphi: np.ndarray,
    H_data_phiv: np.ndarray,
    H_data_vphi: np.ndarray,
    H_data_vv: np.ndarray,
    damping: float = 1e-10,
) -> tuple[np.ndarray, float, np.ndarray]:
    """
    Compute the conditional rotational information matrix

        I_rot = H_phiphi - H_phiv H_vv^{-1} H_vphi.

    This is the Schur-complement rotational information after translational
    perturbations have been optimized out.
    """
    H_data_phiphi = ensure_matrix(H_data_phiphi, (3, 3))
    H_data_phiv = ensure_matrix(H_data_phiv, (3, 3))
    H_data_vphi = ensure_matrix(H_data_vphi, (3, 3))
    H_data_vv = ensure_matrix(H_data_vv, (3, 3))

    H_vv_sym = symmetrize(H_data_vv)
    H_vv_reg = H_vv_sym + damping * np.eye(3)

    try:
        tmp = np.linalg.solve(H_vv_reg, H_data_vphi)
    except np.linalg.LinAlgError:
        tmp = np.linalg.pinv(H_vv_sym) @ H_data_vphi

    I_rot = H_data_phiphi - H_data_phiv @ tmp
    I_rot = symmetrize(I_rot)

    eig_I_rot = np.linalg.eigvalsh(I_rot)
    s_rot = float(np.min(eig_I_rot))

    return I_rot, s_rot, eig_I_rot


def compute_data_information_from_batch(
    batch_result: "BatchReducedModelResult",
    P_w: np.ndarray,
) -> dict:
    """
    Compute data gradient, Gauss--Newton data curvature, residual merit,
    and conditional rotational information from a batch reduced-model result.

    Residual convention:
        r_k = y_k - y_hat_k,

    and local expansion:
        r_k(X_bar oplus xi) = r_k(X_bar) + J_k xi + O(||xi||^2).

    Therefore:
        g_data = sum_k J_k^T Sigma_w^{-1} r_k,
        H_data = sum_k J_k^T Sigma_w^{-1} J_k.
    """
    P_w = ensure_matrix(P_w, (6, 6))

    K = batch_result.residuals.shape[0]

    g_data_phi = np.zeros(3, dtype=float)
    g_data_v = np.zeros(3, dtype=float)

    H_data_phiphi = np.zeros((3, 3), dtype=float)
    H_data_phiv = np.zeros((3, 3), dtype=float)
    H_data_vphi = np.zeros((3, 3), dtype=float)
    H_data_vv = np.zeros((3, 3), dtype=float)

    rho_sq = 0.0
    rho_per_sample = np.zeros(K, dtype=float)

    for k in range(K):
        r_k = ensure_vector(batch_result.residuals[k], 6)
        J_k_phi = ensure_matrix(batch_result.J_phi[k], (6, 3))
        J_k_v = ensure_matrix(batch_result.J_v[k], (6, 3))

        # Data gradient blocks
        g_data_phi += J_k_phi.T @ P_w @ r_k
        g_data_v += J_k_v.T @ P_w @ r_k

        # Data Gauss--Newton curvature blocks
        H_data_phiphi += J_k_phi.T @ P_w @ J_k_phi
        H_data_phiv += J_k_phi.T @ P_w @ J_k_v
        H_data_vphi += J_k_v.T @ P_w @ J_k_phi
        H_data_vv += J_k_v.T @ P_w @ J_k_v

        rho_k_sq = float(r_k.T @ P_w @ r_k)
        rho_per_sample[k] = np.sqrt(max(rho_k_sq, 0.0))
        rho_sq += rho_k_sq

    rho = float(np.sqrt(max(rho_sq, 0.0)))

    g_data = np.concatenate([g_data_phi, g_data_v])

    H_data = np.block([
        [H_data_phiphi, H_data_phiv],
        [H_data_vphi,   H_data_vv],
    ])
    H_data = symmetrize(H_data)

    # Re-split after symmetrization for consistency
    H_data_phiphi, H_data_phiv, H_data_vphi, H_data_vv = split_hessian_blocks_full(H_data)

    I_rot, s_rot, eig_I_rot = conditional_rotational_information_from_Hdata(
        H_data_phiphi=H_data_phiphi,
        H_data_phiv=H_data_phiv,
        H_data_vphi=H_data_vphi,
        H_data_vv=H_data_vv,
    )

    eig_H_data = np.linalg.eigvalsh(H_data)

    return {
        "rho": rho,
        "rho_sq": rho_sq,
        "rho_per_sample": rho_per_sample,

        "g_data": g_data,
        "H_data": H_data,

        "g_data_phi": g_data_phi,
        "g_data_v": g_data_v,

        "H_data_phiphi": H_data_phiphi,
        "H_data_phiv": H_data_phiv,
        "H_data_vphi": H_data_vphi,
        "H_data_vv": H_data_vv,

        "I_rot": I_rot,
        "s_rot": s_rot,
        "eig_I_rot": eig_I_rot,
        "eig_H_data": eig_H_data,
    }


# ---------------------------------------------------------
# Closed-form posterior update
# ---------------------------------------------------------

def closed_form_mfg_posterior_update(
    Theta_prior: "MFGParameters",
    X_B_bar: "Pose",
    nu_bar: np.ndarray,
    J_nu: np.ndarray,
    g_phi: np.ndarray,
    g_v: np.ndarray,
    H_phiphi: np.ndarray,
    H_vphi: np.ndarray,
    H_vv: np.ndarray,
    *,
    precision_jitter: float = 1e-9,
) -> dict:
    """
    Closed-form coupled MFG posterior update.

    Given the local quadratic posterior model

        Phi(xi) = const + g^T xi + 0.5 xi^T H xi,

    with xi = [phi; v], this constructs MFG parameters

        Theta_post = (F_post, mu_post, Lambda_post, Gamma_post)

    matching the local second-order information structure.
    """
    R_B_bar = ensure_matrix(X_B_bar.R, (3, 3))
    p_B_bar = ensure_vector(X_B_bar.p, 3)

    nu_bar = ensure_vector(nu_bar, 3)
    J_nu = ensure_matrix(J_nu, (3, 3))

    g_phi = ensure_vector(g_phi, 3)
    g_v = ensure_vector(g_v, 3)

    H_phiphi = ensure_matrix(H_phiphi, (3, 3))
    H_vphi = ensure_matrix(H_vphi, (3, 3))
    H_vv = ensure_matrix(H_vv, (3, 3))

    # Translation precision of the posterior
    Lambda_post = symmetrize_with_jitter(H_vv, jitter=precision_jitter)

    # Coupling matrix
    J_nu_inv = robust_inverse_3x3(J_nu)
    Gamma_post = -robust_solve(
        Lambda_post,
        H_vphi @ J_nu_inv,
        damping=precision_jitter,
    )

    # Base translation mean
    mu_post = (
        p_B_bar
        - robust_solve(Lambda_post, g_v, damping=precision_jitter)
        - Gamma_post @ nu_bar
    )

    # Rotational first-order matching term
    a_post = g_phi + J_nu.T @ Gamma_post.T @ g_v

    # Rotational second-order matching term
    M_post = (
        H_phiphi
        - J_nu.T @ Gamma_post.T @ Lambda_post @ Gamma_post @ J_nu
    )
    M_post = symmetrize(M_post)

    S_post = 0.5 * np.trace(M_post) * np.eye(3) - M_post

    F_post = R_B_bar @ (S_post - 0.5 * hat(a_post))

    Theta_post = MFGParameters(
        F=F_post,
        mu=mu_post,
        Lambda=Lambda_post,
        Gamma=Gamma_post,
    )

    return {
        "Theta_post": Theta_post,
        "Lambda_post": Lambda_post,
        "Gamma_post": Gamma_post,
        "mu_post": mu_post,
        "a_post": a_post,
        "M_post": M_post,
        "S_post": S_post,
        "F_post": F_post,
    }




# ---------------------------------------------------------
# Main Algorithm 1 interface
# ---------------------------------------------------------

def algorithm1_single_pass_local_mfg_posterior_update(
    Theta_prior: "MFGParameters",
    X_B_bar: "Pose",
    U: Sequence["Pose"],
    Y: Sequence[np.ndarray],
    field_params: "SuperquadricFieldParameters",
    stiffness_params: "StiffnessParameters",
    ctrl_params: "ControlPotentialParameters",
    *,
    Sigma_w: Optional[np.ndarray] = None,
    Sigma_w_inv: Optional[np.ndarray] = None,
    contact_points_A_shared=None,
    contact_points_A_per_sample=None,
    solver_config: Optional["ImplicitEquilibriumSolverConfig"] = None,
    chi_A_0: Optional[Sequence[Optional["Pose"]]] = None,
    prev_outer_chi_A_star: Optional[Sequence[Optional["Pose"]]] = None,
    use_psd_mf_curvature: bool = False,
    # Optional standalone Alg1 posterior-candidate diagnostic
    run_candidate_diagnostic: bool = False,
    candidate_diagnostic_verbose: bool = True,
) -> Algorithm1Result:
    """
    Algorithm 1:
    Single-pass closed-form MFG posterior update under a fixed nominal pose.

    Inputs
    ------
    Theta_prior : MFGParameters
        Prior parameters Theta_prior = (F, mu, Lambda, Gamma).
    X_B_bar : Pose
        Nominal object pose.
    U : sequence of Pose
        Control batch {X_{U,k}}.
    Y : sequence of ndarray
        Measurement batch {y_k}, each y_k in R^6.
    Sigma_w or Sigma_w_inv : ndarray
        Wrench noise covariance or precision.
    chi_A_0 : optional sequence of Pose
        Equilibrium warm starts.
    prev_outer_chi_A_star : optional sequence of Pose
        Previous outer-iteration equilibrium states.

    Returns
    -------
    Algorithm1Result
        Contains
            Theta_post,
            chi_A_star,
            rho,
            s_rot,
            g_data,
            H_data,
        plus block-level diagnostics.
    """
    if solver_config is None:
        solver_config = ImplicitEquilibriumSolverConfig()

    K = len(U)
    if len(Y) != K:
        raise ValueError(f"Length mismatch: len(U)={K}, but len(Y)={len(Y)}.")

    # -----------------------------------------------------
    # Step 1: prior quantities at the nominal pose
    # -----------------------------------------------------
    prior_q = prior_quantities_at_X_B_bar(
        Theta_prior=Theta_prior,
        X_B_bar=X_B_bar,
        solver_config=solver_config,
        use_psd_mf_curvature=use_psd_mf_curvature,
    )

    nu_bar = prior_q["nu_bar"]
    J_nu = prior_q["J_nu"]
    e_bar = prior_q["e_bar"]

    g_prior = prior_q["g_prior"]
    H_prior = prior_q["H_prior"]

    g_prior_phi = prior_q["g_prior_phi"]
    g_prior_v = prior_q["g_prior_v"]

    H_prior_phiphi = prior_q["H_prior_phiphi"]
    H_prior_phiv = prior_q["H_prior_phiv"]
    H_prior_vphi = prior_q["H_prior_vphi"]
    H_prior_vv = prior_q["H_prior_vv"]

    # -----------------------------------------------------
    # Step 2-7: evaluate reduced residuals and Jacobians
    # -----------------------------------------------------
    batch_result = evaluate_batch_reduced_model(
        X_B=X_B_bar,
        X_U_list=U,
        y_list=Y,
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        contact_points_A_shared=contact_points_A_shared,
        contact_points_A_per_sample=contact_points_A_per_sample,
        solver_config=solver_config,
        Sigma_w=Sigma_w,
        Sigma_w_inv=Sigma_w_inv,
        explicit_XA_inits=chi_A_0,
        prev_outer_XA_stars=prev_outer_chi_A_star,
    )

    # -----------------------------------------------------
    # Step 8: data gradient, data curvature, residual merit,
    # and rotational information score
    # -----------------------------------------------------
    P_w = prepare_measurement_precision(
        Sigma_w=Sigma_w,
        Sigma_w_inv=Sigma_w_inv,
    )

    data_q = compute_data_information_from_batch(
        batch_result=batch_result,
        P_w=P_w,
    )

    rho = data_q["rho"]
    rho_sq = data_q["rho_sq"]
    rho_per_sample = data_q["rho_per_sample"]

    g_data = data_q["g_data"]
    H_data = data_q["H_data"]

    g_data_phi = data_q["g_data_phi"]
    g_data_v = data_q["g_data_v"]

    H_data_phiphi = data_q["H_data_phiphi"]
    H_data_phiv = data_q["H_data_phiv"]
    H_data_vphi = data_q["H_data_vphi"]
    H_data_vv = data_q["H_data_vv"]

    I_rot = data_q["I_rot"]
    s_rot = data_q["s_rot"]
    eig_I_rot = data_q["eig_I_rot"]
    eig_H_data = data_q["eig_H_data"]

    # -----------------------------------------------------
    # Step 9: local posterior information pair
    # -----------------------------------------------------
    g = g_prior + g_data
    H = H_prior + H_data
    H = symmetrize(H)

    g_phi, g_v = split_gradient_blocks(g)
    H_phiphi, H_phiv, H_vphi, H_vv = split_hessian_blocks_full(H)

    # -----------------------------------------------------
    # Step 10: closed-form coupled MFG posterior update
    # -----------------------------------------------------
    post_q = closed_form_mfg_posterior_update(
        Theta_prior=Theta_prior,
        X_B_bar=X_B_bar,
        nu_bar=nu_bar,
        J_nu=J_nu,
        g_phi=g_phi,
        g_v=g_v,
        H_phiphi=H_phiphi,
        H_vphi=H_vphi,
        H_vv=H_vv,
    )

    Theta_post = post_q["Theta_post"]


    


    # -----------------------------------------------------
    # Return
    # -----------------------------------------------------
    return Algorithm1Result(
        Theta_post=Theta_post,
        chi_A_star=batch_result.X_A_stars,

        rho=rho,
        s_rot=s_rot,
        I_rot=I_rot,
        g_data=g_data,
        H_data=H_data,

        batch_result=batch_result,

        nu_bar=nu_bar,
        J_nu=J_nu,
        e_bar=e_bar,

        g_prior_phi=g_prior_phi,
        g_prior_v=g_prior_v,
        g_prior=g_prior,

        H_prior_phiphi=H_prior_phiphi,
        H_prior_phiv=H_prior_phiv,
        H_prior_vphi=H_prior_vphi,
        H_prior_vv=H_prior_vv,
        H_prior=H_prior,

        g_data_phi=g_data_phi,
        g_data_v=g_data_v,

        H_data_phiphi=H_data_phiphi,
        H_data_phiv=H_data_phiv,
        H_data_vphi=H_data_vphi,
        H_data_vv=H_data_vv,

        g_phi=g_phi,
        g_v=g_v,
        g=g,

        H_phiphi=H_phiphi,
        H_phiv=H_phiv,
        H_vphi=H_vphi,
        H_vv=H_vv,
        H=H,

        M_post=post_q["M_post"],
        S_post=post_q["S_post"],
        a_post=post_q["a_post"],

        rho_sq=rho_sq,
        rho_per_sample=rho_per_sample,
        eig_H_data=eig_H_data,
        eig_I_rot=eig_I_rot,

    )


# ---------------------------------------------------------

# Main algorithm II Safeguard

## 纯 residual 版

In [20]:
# =========================
# Algorithm 2:
# Safeguarded Multi-Pass MFG Posterior Refinement
#
# Code structure aligned with:
#   Table III : RefinePosterior
#   Table V   : BranchPool
#   Table VI  : FallbackSearch
# =========================

from dataclasses import dataclass
from typing import Optional, Sequence
import numpy as np


# =========================================================
# Candidate and result containers
# =========================================================

@dataclass
class SearchBranchCandidate:
    """
    One accepted candidate produced by SearchBranch / BranchPool / FallbackSearch.
    """
    X_B_cand: Pose
    chi_A_cand: list[Pose]
    rho_cand: float
    batch_result_cand: BatchReducedModelResult
    branch_name: str
    alpha: float
    health_metrics: Optional[dict] = None


@dataclass
class Algorithm2IterationRecord:
    """
    Diagnostics for one outer iteration of Algorithm 2.
    """
    t: int

    X_B_bar_before: Pose
    X_full: Pose
    X_trans: Pose
    X_rot: Pose

    rho_before: float
    s_rot_before: float

    A_rot: list[float]
    n_candidates: int
    branch_pool_sizes: dict
    fallback_used: bool
    n_fallback_candidates: int

    selected_branch: Optional[str]
    selected_alpha: Optional[float]
    X_B_cand: Optional[Pose]
    rho_cand: Optional[float]

    rho_after: Optional[float]
    s_rot_after: Optional[float]
    delta_R: Optional[float]
    delta_p: Optional[float]
    delta_rho: Optional[float]

    accepted: bool
    updated_best: bool
    stop_reason: Optional[str] = None

    alg1_result_after: Optional[Algorithm1Result] = None
    health_metrics_after: Optional[dict] = None


@dataclass
class Algorithm2Result:
    """
    Output of safeguarded multi-pass MFG posterior refinement.
    """
    X_B_star: Pose
    Theta_star: MFGParameters
    chi_A_star_star: list[Pose]
    rho_star: float
    s_rot_star: float

    X_B_bar_final: Pose
    Theta_post_final: MFGParameters
    chi_A_star_final: list[Pose]
    rho_final: float
    s_rot_final: float

    result_init: Algorithm1Result
    result_final: Algorithm1Result
    history: list[Algorithm2IterationRecord]

    n_outer_iterations: int
    converged: bool
    stop_reason: str


# =========================================================
# Basic pose utilities
# =========================================================

def local_pose_difference(X_from: Pose, X_to: Pose) -> np.ndarray:
    """
    Direct-product local pose difference from X_from to X_to:

        xi = [phi; v],
        phi = log(R_from^T R_to),
        v   = p_to - p_from.
    """
    phi = so3_log(X_from.R.T @ X_to.R)
    v = X_to.p - X_from.p
    return pack_xi(phi, v)


def relax_pose(X_start: Pose, X_goal: Pose, alpha: float) -> Pose:
    """
    Relax from X_start to X_goal using the direct-product right retraction:

        X(alpha) = X_start oplus alpha * xi_goal.
    """
    alpha = float(alpha)
    if not (0.0 < alpha <= 1.0):
        raise ValueError(f"alpha must satisfy 0 < alpha <= 1, got {alpha}.")

    xi_goal = local_pose_difference(X_start, X_goal)
    return apply_right_perturbation(X_start, alpha * xi_goal)


def build_translation_only_candidate(X_B_bar: Pose, X_full: Pose) -> Pose:
    """
    Translation-only candidate:
        X_trans = (R_current, p_full).
    """
    return Pose(R=X_B_bar.R.copy(), p=X_full.p.copy())


def build_rotation_only_candidate(X_B_bar: Pose, X_full: Pose) -> Pose:
    """
    Rotation-only candidate:
        X_rot = (R_full, p_current).
    """
    return Pose(R=X_full.R.copy(), p=X_B_bar.p.copy())


def validate_relaxed_step_set(A: Sequence[float]) -> list[float]:
    """
    Validate and sort a relaxed step set A subset (0,1].
    """
    A_valid = sorted(set(float(a) for a in A))
    if len(A_valid) == 0:
        raise ValueError("The relaxed step set must be non-empty.")
    for a in A_valid:
        if not (0.0 < a <= 1.0):
            raise ValueError(f"Each alpha must satisfy 0 < alpha <= 1, got {a}.")
    return A_valid


def reduced_rotation_step_set(
    A: Sequence[float],
    reduction_factor: float = 0.5,
) -> list[float]:
    """
    Construct a conservative rotation step set when s_rot is small.

    If A = [0.25, 0.5, 0.75, 1.0] and reduction_factor=0.5,
    this returns [0.125, 0.25, 0.375, 0.5].
    """
    A = validate_relaxed_step_set(A)
    A_rot = sorted(set(max(1e-8, reduction_factor * float(a)) for a in A))
    A_rot = [a for a in A_rot if 0.0 < a <= 1.0]

    if len(A_rot) == 0:
        A_rot = [min(A)]

    return A_rot


def rotation_geodesic_error(R1: np.ndarray, R2: np.ndarray) -> float:
    """
    SO(3) geodesic distance ||log(R1^T R2)||.
    """
    return float(np.linalg.norm(so3_log(R1.T @ R2)))


# =========================================================
# Residual evaluation and health diagnostics
# =========================================================

def whitened_residual_norm_from_batch_result(
    batch_result: BatchReducedModelResult,
) -> float:
    """
    Compute rho = sqrt(sum_k r_k^T Sigma_w^{-1} r_k).

    Assumes batch_result.per_sample_whitened_norms contains
    sqrt(r_k^T Sigma_w^{-1} r_k).
    """
    wn = np.asarray(batch_result.per_sample_whitened_norms, dtype=float)
    return float(np.sqrt(np.sum(wn ** 2)))


def eval_residual_candidate(
    X_B_eval: Pose,
    U: Sequence[Pose],
    Y: Sequence[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters,
    *,
    Sigma_w: Optional[np.ndarray] = None,
    Sigma_w_inv: Optional[np.ndarray] = None,
    contact_points_A_shared=None,
    contact_points_A_per_sample=None,
    solver_config: Optional[ImplicitEquilibriumSolverConfig] = None,
    chi_A_warm: Optional[Sequence[Optional[Pose]]] = None,
) -> tuple[list[Pose], float, BatchReducedModelResult]:
    """
    Evaluate candidate residual merit:

        (X_A_star, rho, batch_result)
        <- EvalRes(X_B_eval, U, Y, Sigma_w, X_A_warm).
    """
    if solver_config is None:
        solver_config = ImplicitEquilibriumSolverConfig()

    batch_result = evaluate_batch_reduced_model(
        X_B=X_B_eval,
        X_U_list=U,
        y_list=Y,
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        contact_points_A_shared=contact_points_A_shared,
        contact_points_A_per_sample=contact_points_A_per_sample,
        solver_config=solver_config,
        Sigma_w=Sigma_w,
        Sigma_w_inv=Sigma_w_inv,
        prev_outer_XA_stars=chi_A_warm,
    )

    chi_A_star = batch_result.X_A_stars
    rho = whitened_residual_norm_from_batch_result(batch_result)

    return chi_A_star, rho, batch_result


def compute_batch_result_health_metrics(
    batch_result: BatchReducedModelResult,
) -> dict:
    """
    Batch-level numerical health diagnostics.
    """
    K = int(batch_result.y_obs.shape[0])

    J_phi_norms = np.array(
        [np.linalg.norm(batch_result.J_phi[k], ord="fro") for k in range(K)],
        dtype=float,
    )
    J_v_norms = np.array(
        [np.linalg.norm(batch_result.J_v[k], ord="fro") for k in range(K)],
        dtype=float,
    )

    WAA_conds = []
    WAA_min_eigs = []
    eq_res_norms = []
    eq_step_norms = []
    all_finite = True

    for sr in batch_result.sample_results:
        WAA = sr.W_AA
        WAA_sym = 0.5 * (WAA + WAA.T)
        eigs = np.linalg.eigvalsh(WAA_sym)

        WAA_conds.append(float(np.linalg.cond(WAA)))
        WAA_min_eigs.append(float(np.min(eigs)))
        eq_res_norms.append(float(sr.res_norm))
        eq_step_norms.append(float(sr.step_norm))

        all_finite = (
            all_finite
            and np.isfinite(sr.W_AA).all()
            and np.isfinite(sr.W_UA).all()
            and np.isfinite(sr.W_AB).all()
            and np.isfinite(sr.J_k).all()
            and np.isfinite(sr.y_hat).all()
            and np.isfinite(sr.F_A).all()
        )

    return {
        "K": K,
        "n_converged": int(batch_result.n_converged),
        "all_inner_converged": bool(batch_result.n_converged == K),
        "all_finite": bool(all_finite and np.isfinite(batch_result.residuals).all()),
        "max_J_phi_norm": float(np.max(J_phi_norms)) if K > 0 else 0.0,
        "max_J_v_norm": float(np.max(J_v_norms)) if K > 0 else 0.0,
        "max_cond_WAA": float(np.max(WAA_conds)) if K > 0 else 0.0,
        "min_min_eig_WAA": float(np.min(WAA_min_eigs)) if K > 0 else 0.0,
        "max_eq_res_norm": float(np.max(eq_res_norms)) if K > 0 else 0.0,
        "max_eq_step_norm": float(np.max(eq_step_norms)) if K > 0 else 0.0,
    }


def is_batch_result_healthy(
    batch_result: BatchReducedModelResult,
    *,
    require_all_inner_converged: bool = True,
    require_psd_WAA: bool = True,
    max_cond_WAA: float = 1e5,
    min_min_eig_WAA: float = 1e-8,
    max_J_phi_norm: float = 1e4,
    max_J_v_norm: float = 1e6,
    max_eq_res_norm: float = 1e-4,
    max_eq_step_norm: float = 1e-2,
) -> tuple[bool, dict]:
    """
    Decide whether a batch residual/Jacobian evaluation is numerically healthy.
    """
    m = compute_batch_result_health_metrics(batch_result)

    healthy = True
    healthy = healthy and m["all_finite"]

    if require_all_inner_converged:
        healthy = healthy and m["all_inner_converged"]

    if require_psd_WAA:
        healthy = healthy and (m["min_min_eig_WAA"] >= min_min_eig_WAA)

    healthy = healthy and (m["max_cond_WAA"] <= max_cond_WAA)
    healthy = healthy and (m["max_J_phi_norm"] <= max_J_phi_norm)
    healthy = healthy and (m["max_J_v_norm"] <= max_J_v_norm)
    healthy = healthy and (m["max_eq_res_norm"] <= max_eq_res_norm)
    healthy = healthy and (m["max_eq_step_norm"] <= max_eq_step_norm)

    return bool(healthy), m


# =========================================================
# Table V auxiliary: SearchBranch
# =========================================================

def search_branch_safeguarded(
    X_B_bar: Pose,
    chi_A_star: Sequence[Pose],
    X_B_branch: Pose,
    U: Sequence[Pose],
    Y: Sequence[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters,
    *,
    Sigma_w: Optional[np.ndarray] = None,
    Sigma_w_inv: Optional[np.ndarray] = None,
    contact_points_A_shared=None,
    contact_points_A_per_sample=None,
    A: Sequence[float] = (0.25, 0.5, 0.75, 1.0),
    rho: float,
    eps_acc: float,
    solver_config: Optional[ImplicitEquilibriumSolverConfig] = None,
    branch_name: str = "full",
    require_all_inner_converged: bool = True,
    require_psd_WAA: bool = True,
    max_cond_WAA: float = 1e5,
    min_min_eig_WAA: float = 1e-8,
    max_J_phi_norm: float = 1e4,
    max_J_v_norm: float = 1e6,
    max_eq_res_norm: float = 1e-4,
    max_eq_step_norm: float = 1e-2,
) -> list[SearchBranchCandidate]:
    """
    SearchBranch:
    safeguarded relaxed search within one branch.
    """
    if solver_config is None:
        solver_config = ImplicitEquilibriumSolverConfig()

    A = validate_relaxed_step_set(A)
    C_b: list[SearchBranchCandidate] = []

    chi_A_prev = list(chi_A_star)

    for alpha in A:
        X_eval = relax_pose(X_B_bar, X_B_branch, alpha)

        chi_A_eval, rho_eval, batch_eval = eval_residual_candidate(
            X_B_eval=X_eval,
            U=U,
            Y=Y,
            field_params=field_params,
            stiffness_params=stiffness_params,
            ctrl_params=ctrl_params,
            Sigma_w=Sigma_w,
            Sigma_w_inv=Sigma_w_inv,
            contact_points_A_shared=contact_points_A_shared,
            contact_points_A_per_sample=contact_points_A_per_sample,
            solver_config=solver_config,
            chi_A_warm=chi_A_prev,
        )

        healthy, health = is_batch_result_healthy(
            batch_eval,
            require_all_inner_converged=require_all_inner_converged,
            require_psd_WAA=require_psd_WAA,
            max_cond_WAA=max_cond_WAA,
            min_min_eig_WAA=min_min_eig_WAA,
            max_J_phi_norm=max_J_phi_norm,
            max_J_v_norm=max_J_v_norm,
            max_eq_res_norm=max_eq_res_norm,
            max_eq_step_norm=max_eq_step_norm,
        )

        if (rho_eval <= rho - eps_acc) and healthy:
            C_b.append(
                SearchBranchCandidate(
                    X_B_cand=copy_pose(X_eval),
                    chi_A_cand=chi_A_eval,
                    rho_cand=float(rho_eval),
                    batch_result_cand=batch_eval,
                    branch_name=branch_name,
                    alpha=float(alpha),
                    health_metrics=health,
                )
            )

        # Warm-start continuity for the next relaxed step
        chi_A_prev = chi_A_eval

    return C_b


# =========================================================
# Table V: BranchPool
# =========================================================

def branch_pool(
    X_B_bar: Pose,
    chi_A_star: Sequence[Pose],
    X_full: Pose,
    X_trans: Pose,
    X_rot: Pose,
    U: Sequence[Pose],
    Y: Sequence[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters,
    *,
    Sigma_w: Optional[np.ndarray] = None,
    Sigma_w_inv: Optional[np.ndarray] = None,
    contact_points_A_shared=None,
    contact_points_A_per_sample=None,
    A: Sequence[float],
    A_rot: Sequence[float],
    rho: float,
    eps_acc: float,
    solver_config: Optional[ImplicitEquilibriumSolverConfig] = None,
    require_all_inner_converged: bool = True,
    require_psd_WAA: bool = True,
    max_cond_WAA: float = 1e5,
    min_min_eig_WAA: float = 1e-8,
    max_J_phi_norm: float = 1e4,
    max_J_v_norm: float = 1e6,
    max_eq_res_norm: float = 1e-4,
    max_eq_step_norm: float = 1e-2,
) -> tuple[list[SearchBranchCandidate], dict]:
    """
    Table V:
    Posterior-induced candidate pool.

    Returns
    -------
    C : list[SearchBranchCandidate]
        Union of full, translation-only, and rotation-only branch candidates.
    sizes : dict
        Number of accepted candidates in each branch.
    """
    C_full = search_branch_safeguarded(
        X_B_bar=X_B_bar,
        chi_A_star=chi_A_star,
        X_B_branch=X_full,
        U=U,
        Y=Y,
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        Sigma_w=Sigma_w,
        Sigma_w_inv=Sigma_w_inv,
        contact_points_A_shared=contact_points_A_shared,
        contact_points_A_per_sample=contact_points_A_per_sample,
        A=A,
        rho=rho,
        eps_acc=eps_acc,
        solver_config=solver_config,
        branch_name="full",
        require_all_inner_converged=require_all_inner_converged,
        require_psd_WAA=require_psd_WAA,
        max_cond_WAA=max_cond_WAA,
        min_min_eig_WAA=min_min_eig_WAA,
        max_J_phi_norm=max_J_phi_norm,
        max_J_v_norm=max_J_v_norm,
        max_eq_res_norm=max_eq_res_norm,
        max_eq_step_norm=max_eq_step_norm,
    )

    C_trans = search_branch_safeguarded(
        X_B_bar=X_B_bar,
        chi_A_star=chi_A_star,
        X_B_branch=X_trans,
        U=U,
        Y=Y,
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        Sigma_w=Sigma_w,
        Sigma_w_inv=Sigma_w_inv,
        contact_points_A_shared=contact_points_A_shared,
        contact_points_A_per_sample=contact_points_A_per_sample,
        A=A,
        rho=rho,
        eps_acc=eps_acc,
        solver_config=solver_config,
        branch_name="trans",
        require_all_inner_converged=require_all_inner_converged,
        require_psd_WAA=require_psd_WAA,
        max_cond_WAA=max_cond_WAA,
        min_min_eig_WAA=min_min_eig_WAA,
        max_J_phi_norm=max_J_phi_norm,
        max_J_v_norm=max_J_v_norm,
        max_eq_res_norm=max_eq_res_norm,
        max_eq_step_norm=max_eq_step_norm,
    )

    C_rot = search_branch_safeguarded(
        X_B_bar=X_B_bar,
        chi_A_star=chi_A_star,
        X_B_branch=X_rot,
        U=U,
        Y=Y,
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        Sigma_w=Sigma_w,
        Sigma_w_inv=Sigma_w_inv,
        contact_points_A_shared=contact_points_A_shared,
        contact_points_A_per_sample=contact_points_A_per_sample,
        A=A_rot,
        rho=rho,
        eps_acc=eps_acc,
        solver_config=solver_config,
        branch_name="rot",
        require_all_inner_converged=require_all_inner_converged,
        require_psd_WAA=require_psd_WAA,
        max_cond_WAA=max_cond_WAA,
        min_min_eig_WAA=min_min_eig_WAA,
        max_J_phi_norm=max_J_phi_norm,
        max_J_v_norm=max_J_v_norm,
        max_eq_res_norm=max_eq_res_norm,
        max_eq_step_norm=max_eq_step_norm,
    )

    C = list(C_full) + list(C_trans) + list(C_rot)

    sizes = {
        "full": len(C_full),
        "trans": len(C_trans),
        "rot": len(C_rot),
    }

    return C, sizes


# =========================================================
# Table VI: FallbackSearch
# =========================================================

def fallback_search(
    X_B_bar: Pose,
    chi_A_star: Sequence[Pose],
    g_data: np.ndarray,
    H_data: np.ndarray,
    U: Sequence[Pose],
    Y: Sequence[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters,
    *,
    Sigma_w: Optional[np.ndarray] = None,
    Sigma_w_inv: Optional[np.ndarray] = None,
    contact_points_A_shared=None,
    contact_points_A_per_sample=None,
    A: Sequence[float],
    rho: float,
    lambda_fb: float = 1e-6,
    solver_config: Optional[ImplicitEquilibriumSolverConfig] = None,
    require_all_inner_converged: bool = True,
    require_psd_WAA: bool = True,
    max_cond_WAA: float = 1e5,
    min_min_eig_WAA: float = 1e-8,
    max_J_phi_norm: float = 1e4,
    max_J_v_norm: float = 1e6,
    max_eq_res_norm: float = 1e-4,
    max_eq_step_norm: float = 1e-2,
) -> list[SearchBranchCandidate]:
    """
    Table VI:
    residual-descent fallback search.

    Direction:
        d_fb = -(H_data + lambda_fb I)^{-1} g_data.
    """
    if solver_config is None:
        solver_config = ImplicitEquilibriumSolverConfig()

    A = validate_relaxed_step_set(A)
    g_data = ensure_vector(g_data, 6)
    H_data = ensure_matrix(H_data, (6, 6))
    H_data = 0.5 * (H_data + H_data.T)

    H_reg = H_data + float(lambda_fb) * np.eye(6)

    try:
        d_fb = -np.linalg.solve(H_reg, g_data)
    except np.linalg.LinAlgError:
        d_fb = -np.linalg.pinv(H_reg) @ g_data

    C_fb: list[SearchBranchCandidate] = []
    chi_A_prev = list(chi_A_star)

    for alpha in A:
        X_fb = apply_right_perturbation(X_B_bar, alpha * d_fb)

        chi_A_eval, rho_eval, batch_eval = eval_residual_candidate(
            X_B_eval=X_fb,
            U=U,
            Y=Y,
            field_params=field_params,
            stiffness_params=stiffness_params,
            ctrl_params=ctrl_params,
            Sigma_w=Sigma_w,
            Sigma_w_inv=Sigma_w_inv,
            contact_points_A_shared=contact_points_A_shared,
            contact_points_A_per_sample=contact_points_A_per_sample,
            solver_config=solver_config,
            chi_A_warm=chi_A_prev,
        )

        healthy, health = is_batch_result_healthy(
            batch_eval,
            require_all_inner_converged=require_all_inner_converged,
            require_psd_WAA=require_psd_WAA,
            max_cond_WAA=max_cond_WAA,
            min_min_eig_WAA=min_min_eig_WAA,
            max_J_phi_norm=max_J_phi_norm,
            max_J_v_norm=max_J_v_norm,
            max_eq_res_norm=max_eq_res_norm,
            max_eq_step_norm=max_eq_step_norm,
        )

        if (rho_eval < rho) and healthy:
            C_fb.append(
                SearchBranchCandidate(
                    X_B_cand=copy_pose(X_fb),
                    chi_A_cand=chi_A_eval,
                    rho_cand=float(rho_eval),
                    batch_result_cand=batch_eval,
                    branch_name="fallback",
                    alpha=float(alpha),
                    health_metrics=health,
                )
            )

        chi_A_prev = chi_A_eval

    return C_fb


# =========================================================
# Posterior-level health checks
# =========================================================

def compute_theta_health_metrics(theta: MFGParameters) -> dict:
    """
    Diagnostics for MFG parameters.
    """
    F = np.asarray(theta.F, dtype=float)
    mu = np.asarray(theta.mu, dtype=float)
    Lambda = np.asarray(theta.Lambda, dtype=float)
    Gamma = np.asarray(theta.Gamma, dtype=float)

    Lambda_sym = 0.5 * (Lambda + Lambda.T)
    lambda_eigs = np.linalg.eigvalsh(Lambda_sym)

    all_finite = (
        np.isfinite(F).all()
        and np.isfinite(mu).all()
        and np.isfinite(Lambda).all()
        and np.isfinite(Gamma).all()
    )

    try:
        R_hat = rotation_mode_from_F(F)
        nu_hat = mfg_nu_R(R_hat, F)
        mode_nu_norm = float(np.linalg.norm(nu_hat))
    except Exception:
        mode_nu_norm = np.inf

    return {
        "all_finite": bool(all_finite),
        "norm_F_post": float(np.linalg.norm(F, ord="fro")),
        "norm_mu_post": float(np.linalg.norm(mu)),
        "norm_Gamma_post": float(np.linalg.norm(Gamma, ord="fro")),
        "min_eig_Lambda_post": float(np.min(lambda_eigs)),
        "max_eig_Lambda_post": float(np.max(lambda_eigs)),
        "cond_Lambda_post": float(np.linalg.cond(Lambda_sym)),
        "mode_nu_norm": mode_nu_norm,
    }


# =========================================================
# Posterior-level health checks
# Batch safeguards are now passed consistently
# =========================================================

def compute_algorithm1_result_health_metrics(
    res: Algorithm1Result,
    *,
    # batch-level safeguards
    require_all_inner_converged: bool = True,
    require_psd_WAA: bool = True,
    max_cond_WAA: float = 1e5,
    min_min_eig_WAA: float = 1e-8,
    max_J_phi_norm: float = 1e4,
    max_J_v_norm: float = 1e6,
    max_eq_res_norm: float = 1e-4,
    max_eq_step_norm: float = 1e-2,
) -> dict:
    """
    Diagnostics for an Algorithm 1 result.

    Important:
    The batch-level health checker must use the same safeguard thresholds
    as the outer Algorithm 2 call. Otherwise Stage-2 posterior refresh may
    silently revert to stricter default thresholds.
    """
    theta_metrics = compute_theta_health_metrics(res.Theta_post)

    batch_healthy, batch_metrics = is_batch_result_healthy(
        res.batch_result,
        require_all_inner_converged=require_all_inner_converged,
        require_psd_WAA=require_psd_WAA,
        max_cond_WAA=max_cond_WAA,
        min_min_eig_WAA=min_min_eig_WAA,
        max_J_phi_norm=max_J_phi_norm,
        max_J_v_norm=max_J_v_norm,
        max_eq_res_norm=max_eq_res_norm,
        max_eq_step_norm=max_eq_step_norm,
    )

    metrics = {
        **theta_metrics,
        "all_finite_result": bool(
            np.isfinite(res.g).all()
            and np.isfinite(res.H).all()
            and np.isfinite(res.g_data).all()
            and np.isfinite(res.H_data).all()
            and np.isfinite(res.I_rot).all()
            and np.isfinite(res.M_post).all()
            and np.isfinite(res.S_post).all()
            and np.isfinite(res.a_post).all()
            and np.isfinite(res.rho)
            and np.isfinite(res.s_rot)
        ),
        "norm_H": float(np.linalg.norm(res.H, ord="fro")),
        "norm_H_data": float(np.linalg.norm(res.H_data, ord="fro")),
        "norm_M_post": float(np.linalg.norm(res.M_post, ord="fro")),
        "norm_a_post": float(np.linalg.norm(res.a_post)),
        "rho": float(res.rho),
        "s_rot": float(res.s_rot),
        "batch_is_healthy": bool(batch_healthy),
        "batch_metrics": batch_metrics,
    }

    return metrics


def is_algorithm1_result_healthy(
    res: Algorithm1Result,
    *,
    # batch-level safeguards
    require_all_inner_converged: bool = True,
    require_psd_WAA: bool = True,
    max_cond_WAA: float = 1e5,
    min_min_eig_WAA: float = 1e-8,
    max_J_phi_norm: float = 1e4,
    max_J_v_norm: float = 1e6,
    max_eq_res_norm: float = 1e-4,
    max_eq_step_norm: float = 1e-2,

    # posterior-level safeguards
    min_eig_Lambda_post: float = 1e-10,
    max_cond_Lambda_post: float = 1e8,
    max_norm_H: float = 1e10,
    max_norm_M_post: float = 1e10,
    max_norm_a_post: float = 1e8,
    max_norm_F_post: float = 1e8,
    max_mode_nu_norm: float = 1e3,
) -> tuple[bool, dict]:
    """
    Posterior-level safeguard.

    The batch-level numerical health conditions and posterior-level MFG
    parameter conditions are checked together.
    """
    m = compute_algorithm1_result_health_metrics(
        res,
        require_all_inner_converged=require_all_inner_converged,
        require_psd_WAA=require_psd_WAA,
        max_cond_WAA=max_cond_WAA,
        min_min_eig_WAA=min_min_eig_WAA,
        max_J_phi_norm=max_J_phi_norm,
        max_J_v_norm=max_J_v_norm,
        max_eq_res_norm=max_eq_res_norm,
        max_eq_step_norm=max_eq_step_norm,
    )

    healthy = True
    healthy = healthy and m["all_finite"]
    healthy = healthy and m["all_finite_result"]
    healthy = healthy and m["batch_is_healthy"]

    healthy = healthy and (m["min_eig_Lambda_post"] > min_eig_Lambda_post)
    healthy = healthy and (m["cond_Lambda_post"] < max_cond_Lambda_post)
    healthy = healthy and (m["norm_H"] < max_norm_H)
    healthy = healthy and (m["norm_M_post"] < max_norm_M_post)
    healthy = healthy and (m["norm_a_post"] < max_norm_a_post)
    healthy = healthy and (m["norm_F_post"] < max_norm_F_post)
    healthy = healthy and (m["mode_nu_norm"] < max_mode_nu_norm)

    return bool(healthy), m






# =========================================================
# Table III: Main refinement algorithm
# =========================================================
# Residual-merit safeguarded multi-pass refinement
# Fixed: posterior health checker inherits relaxed batch safeguards
# =========================================================

def algorithm2_safeguarded_multi_pass_mfg_posterior_refinement(
    X_B_bar_0: Pose,
    Theta_0: MFGParameters,
    U: Sequence[Pose],
    Y: Sequence[np.ndarray],
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters,
    *,
    Sigma_w: Optional[np.ndarray] = None,
    Sigma_w_inv: Optional[np.ndarray] = None,
    contact_points_A_shared=None,
    contact_points_A_per_sample=None,
    # T_max: int = 15,
    # A: Sequence[float] = (0.25, 0.5, 0.75, 1.0),
    # eps_acc: float = 1e-6,
    # eps_R: float = 1e-6,
    # eps_p: float = 1e-6,
    # eps_info: float = 1e-8,
    # eps_g: float = 1e-8,
    # lambda_fb: float = 1e-6,
    # A_rot_reduction_factor: float = 0.5,

    T_max: int = 25,
    A: Sequence[float] = (0.25, 0.10, 0.05, 0.02, 0.01, 0.005),
    eps_acc: float = 1e-6,
    eps_R: float = 1e-6,
    eps_p: float = 1e-6,
    eps_info: float = 1e-8,
    eps_g: float = 1e-8,
    lambda_fb: float = 1e-4,
    A_rot_reduction_factor: float = 0.25,

    # New stopping rule: residual plateau detection
    rho_rel_tol: float = 5e-4,
    rho_abs_tol: float = 1e-4,
    plateau_patience: int = 3,
    min_outer: int = 4,

    solver_config: Optional[ImplicitEquilibriumSolverConfig] = None,
    chi_A_0: Optional[Sequence[Optional[Pose]]] = None,

    # batch-level safeguards
    require_all_inner_converged: bool = True,
    require_psd_WAA: bool = True,
    max_cond_WAA: float = 1e5,
    min_min_eig_WAA: float = 1e-8,
    max_J_phi_norm: float = 1e4,
    max_J_v_norm: float = 1e6,
    max_eq_res_norm: float = 1e-4,
    max_eq_step_norm: float = 1e-2,

    # posterior-level safeguards
    min_eig_Lambda_post: float = 1e-10,
    max_cond_Lambda_post: float = 1e8,
    max_norm_H: float = 1e10,
    max_norm_M_post: float = 1e10,
    max_norm_a_post: float = 1e8,
    max_norm_F_post: float = 1e8,
    max_mode_nu_norm: float = 1e3,
) -> Algorithm2Result:
    """
    Residual-merit safeguarded multi-pass MFG posterior refinement.

    Merit:
        rho(X)

    The fixed initial prior Theta_0 is reused in every Algorithm 1 refresh.
    """

    if solver_config is None:
        solver_config = ImplicitEquilibriumSolverConfig()

    A = validate_relaxed_step_set(A)

    # -----------------------------------------------------
    # Initial single-pass update
    # -----------------------------------------------------
    result_init = algorithm1_single_pass_local_mfg_posterior_update(
        Theta_prior=Theta_0,
        X_B_bar=X_B_bar_0,
        U=U,
        Y=Y,
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        Sigma_w=Sigma_w,
        Sigma_w_inv=Sigma_w_inv,
        contact_points_A_shared=contact_points_A_shared,
        contact_points_A_per_sample=contact_points_A_per_sample,
        solver_config=solver_config,
        chi_A_0=chi_A_0,
    )
    result_t = result_init

    init_healthy, init_health = is_algorithm1_result_healthy(
        result_t,

        require_all_inner_converged=require_all_inner_converged,
        require_psd_WAA=require_psd_WAA,
        max_cond_WAA=max_cond_WAA,
        min_min_eig_WAA=min_min_eig_WAA,
        max_J_phi_norm=max_J_phi_norm,
        max_J_v_norm=max_J_v_norm,
        max_eq_res_norm=max_eq_res_norm,
        max_eq_step_norm=max_eq_step_norm,

        min_eig_Lambda_post=min_eig_Lambda_post,
        max_cond_Lambda_post=max_cond_Lambda_post,
        max_norm_H=max_norm_H,
        max_norm_M_post=max_norm_M_post,
        max_norm_a_post=max_norm_a_post,
        max_norm_F_post=max_norm_F_post,
        max_mode_nu_norm=max_mode_nu_norm,
    )

    def _print_health_metrics(health, title="Algorithm 1 health metrics"):
        print("\n" + "=" * 80)
        print(title)
        print("=" * 80)
        for k, v in health.items():
            if isinstance(v, float):
                print(f"{k:30s}: {v:.6e}")
            else:
                print(f"{k:30s}: {v}")
        print("=" * 80)

    def _is_hard_unhealthy(health, eig_eps=1e-12):
        if ("all_finite" in health) and (not bool(health["all_finite"])):
            return True, "all_finite=False"

        if ("all_finite_result" in health) and (not bool(health["all_finite_result"])):
            return True, "all_finite_result=False"

        if "min_eig_Lambda_post" in health:
            if health["min_eig_Lambda_post"] <= eig_eps:
                return True, (
                    f"min_eig_Lambda_post={health['min_eig_Lambda_post']:.6e} "
                    f"<= {eig_eps:.1e}"
                )

        return False, "soft-threshold violation only"

    if not init_healthy:
        _print_health_metrics(
            init_health,
            title="Warning: initial single-pass violates health checker"
        )

        hard_bad, hard_reason = _is_hard_unhealthy(init_health)

        if hard_bad:
            raise ValueError(
                "Initial single-pass result has a hard numerical failure. "
                f"Reason: {hard_reason}. "
                f"Health metrics: {init_health}"
            )
        else:
            print(
                "[Warning] Initial single-pass violates only soft health thresholds. "
                "Continue Algorithm 2 with trust-region safeguards."
            )

    # -----------------------------------------------------
    # Initialize current state
    # -----------------------------------------------------
    X_B_bar_t = copy_pose(X_B_bar_0)
    Theta_post_t = result_t.Theta_post
    chi_A_star_t = list(result_t.chi_A_star)
    rho_t = float(result_t.rho)
    s_rot_t = float(result_t.s_rot)

    # -----------------------------------------------------
    # Initialize best-so-far state under residual merit
    # -----------------------------------------------------
    X_B_star = copy_pose(X_B_bar_0)
    Theta_star = result_t.Theta_post
    chi_A_star_star = list(result_t.chi_A_star)
    rho_star = float(result_t.rho)
    s_rot_star = float(result_t.s_rot)
    result_star = result_t

    history: list[Algorithm2IterationRecord] = []

    converged = False
    stop_reason = "reached T_max"
    plateau_count = 0
    # -----------------------------------------------------
    # Outer refinement loop
    # -----------------------------------------------------
    for t in range(T_max):
        X_B_before = copy_pose(X_B_bar_t)
        rho_before = float(rho_t)
        s_rot_before = float(s_rot_t)

        # Posterior-induced candidates
        X_full = recover_pose_from_theta(Theta_post_t)
        X_trans = build_translation_only_candidate(X_B_bar_t, X_full)
        X_rot = build_rotation_only_candidate(X_B_bar_t, X_full)

        # Information-conditioned rotation step set
        if s_rot_t < eps_info:
            A_rot = reduced_rotation_step_set(
                A,
                reduction_factor=A_rot_reduction_factor,
            )
        else:
            A_rot = list(A)

        # -------------------------------------------------
        # Candidate pool under residual decrease
        # -------------------------------------------------
        C_t, branch_sizes = branch_pool(
            X_B_bar=X_B_bar_t,
            chi_A_star=chi_A_star_t,
            X_full=X_full,
            X_trans=X_trans,
            X_rot=X_rot,
            U=U,
            Y=Y,
            field_params=field_params,
            stiffness_params=stiffness_params,
            ctrl_params=ctrl_params,
            Sigma_w=Sigma_w,
            Sigma_w_inv=Sigma_w_inv,
            contact_points_A_shared=contact_points_A_shared,
            contact_points_A_per_sample=contact_points_A_per_sample,
            A=A,
            A_rot=A_rot,
            rho=rho_t,
            eps_acc=eps_acc,
            solver_config=solver_config,

            require_all_inner_converged=require_all_inner_converged,
            require_psd_WAA=require_psd_WAA,
            max_cond_WAA=max_cond_WAA,
            min_min_eig_WAA=min_min_eig_WAA,
            max_J_phi_norm=max_J_phi_norm,
            max_J_v_norm=max_J_v_norm,
            max_eq_res_norm=max_eq_res_norm,
            max_eq_step_norm=max_eq_step_norm,
        )

        # -------------------------------------------------
        # Fallback if branch pool is empty
        # -------------------------------------------------
        fallback_used = False
        n_fallback = 0

        if len(C_t) == 0 and np.linalg.norm(result_t.g_data) > eps_g:
            C_fb = fallback_search(
                X_B_bar=X_B_bar_t,
                chi_A_star=chi_A_star_t,
                g_data=result_t.g_data,
                H_data=result_t.H_data,
                U=U,
                Y=Y,
                field_params=field_params,
                stiffness_params=stiffness_params,
                ctrl_params=ctrl_params,
                Sigma_w=Sigma_w,
                Sigma_w_inv=Sigma_w_inv,
                contact_points_A_shared=contact_points_A_shared,
                contact_points_A_per_sample=contact_points_A_per_sample,
                A=A,
                rho=rho_t,
                lambda_fb=lambda_fb,
                solver_config=solver_config,

                require_all_inner_converged=require_all_inner_converged,
                require_psd_WAA=require_psd_WAA,
                max_cond_WAA=max_cond_WAA,
                min_min_eig_WAA=min_min_eig_WAA,
                max_J_phi_norm=max_J_phi_norm,
                max_J_v_norm=max_J_v_norm,
                max_eq_res_norm=max_eq_res_norm,
                max_eq_step_norm=max_eq_step_norm,
            )
            C_t = list(C_t) + list(C_fb)
            fallback_used = True
            n_fallback = len(C_fb)

        # -------------------------------------------------
        # Stop if no candidate passed Stage 1
        # -------------------------------------------------
        if len(C_t) == 0:
            record = Algorithm2IterationRecord(
                t=t,
                X_B_bar_before=X_B_before,
                X_full=X_full,
                X_trans=X_trans,
                X_rot=X_rot,
                rho_before=rho_before,
                s_rot_before=s_rot_before,
                A_rot=A_rot,
                n_candidates=0,
                branch_pool_sizes=branch_sizes,
                fallback_used=fallback_used,
                n_fallback_candidates=n_fallback,
                selected_branch=None,
                selected_alpha=None,
                X_B_cand=None,
                rho_cand=None,
                rho_after=None,
                s_rot_after=None,
                delta_R=None,
                delta_p=None,
                delta_rho=None,
                accepted=False,
                updated_best=False,
                stop_reason="empty candidate pool",
                alg1_result_after=None,
                health_metrics_after=None,
            )
            history.append(record)

            converged = True
            stop_reason = "empty candidate pool"
            break

        # -------------------------------------------------
        # Sort candidates by residual merit
        # -------------------------------------------------
        C_t_sorted = sorted(C_t, key=lambda c: c.rho_cand)

        accepted = False
        selected_cand = None
        result_next = None
        health_next = None

        # -------------------------------------------------
        # Stage-2 posterior refresh
        # -------------------------------------------------
        for cand in C_t_sorted:
            result_try = algorithm1_single_pass_local_mfg_posterior_update(
                Theta_prior=Theta_0,
                X_B_bar=cand.X_B_cand,
                U=U,
                Y=Y,
                field_params=field_params,
                stiffness_params=stiffness_params,
                ctrl_params=ctrl_params,
                Sigma_w=Sigma_w,
                Sigma_w_inv=Sigma_w_inv,
                contact_points_A_shared=contact_points_A_shared,
                contact_points_A_per_sample=contact_points_A_per_sample,
                solver_config=solver_config,
                chi_A_0=cand.chi_A_cand,
            )

            healthy_try, health_try = is_algorithm1_result_healthy(
                result_try,

                require_all_inner_converged=require_all_inner_converged,
                require_psd_WAA=require_psd_WAA,
                max_cond_WAA=max_cond_WAA,
                min_min_eig_WAA=min_min_eig_WAA,
                max_J_phi_norm=max_J_phi_norm,
                max_J_v_norm=max_J_v_norm,
                max_eq_res_norm=max_eq_res_norm,
                max_eq_step_norm=max_eq_step_norm,

                min_eig_Lambda_post=min_eig_Lambda_post,
                max_cond_Lambda_post=max_cond_Lambda_post,
                max_norm_H=max_norm_H,
                max_norm_M_post=max_norm_M_post,
                max_norm_a_post=max_norm_a_post,
                max_norm_F_post=max_norm_F_post,
                max_mode_nu_norm=max_mode_nu_norm,
            )

            # Require residual decrease after posterior recomputation
            if healthy_try and (result_try.rho <= rho_t - eps_acc):
                accepted = True
                selected_cand = cand
                result_next = result_try
                health_next = health_try
                break

        # -------------------------------------------------
        # Stop if no candidate survives Stage 2
        # -------------------------------------------------
        if not accepted:
            record = Algorithm2IterationRecord(
                t=t,
                X_B_bar_before=X_B_before,
                X_full=X_full,
                X_trans=X_trans,
                X_rot=X_rot,
                rho_before=rho_before,
                s_rot_before=s_rot_before,
                A_rot=A_rot,
                n_candidates=len(C_t),
                branch_pool_sizes=branch_sizes,
                fallback_used=fallback_used,
                n_fallback_candidates=n_fallback,
                selected_branch=None,
                selected_alpha=None,
                X_B_cand=None,
                rho_cand=None,
                rho_after=None,
                s_rot_after=None,
                delta_R=None,
                delta_p=None,
                delta_rho=None,
                accepted=False,
                updated_best=False,
                stop_reason="no candidate survived posterior health check",
                alg1_result_after=None,
                health_metrics_after=None,
            )
            history.append(record)

            converged = True
            stop_reason = "no candidate survived posterior health check"
            break

        # -------------------------------------------------
        # Accept current nominal pose
        # -------------------------------------------------
        X_next = copy_pose(selected_cand.X_B_cand)
        delta_R = rotation_geodesic_error(X_B_bar_t.R, X_next.R)
        delta_p = float(np.linalg.norm(X_next.p - X_B_bar_t.p))

        rho_next = float(result_next.rho)
        s_rot_next = float(result_next.s_rot)
        delta_rho = float(rho_t - rho_next)

        X_B_bar_t = copy_pose(X_next)
        Theta_post_t = result_next.Theta_post
        chi_A_star_t = list(result_next.chi_A_star)
        rho_t = rho_next
        s_rot_t = s_rot_next
        result_t = result_next

        # -------------------------------------------------
        # Best-so-far under residual merit
        # -------------------------------------------------
        updated_best = False
        if rho_t < rho_star:
            X_B_star = copy_pose(X_B_bar_t)
            Theta_star = result_t.Theta_post
            chi_A_star_star = list(result_t.chi_A_star)
            rho_star = float(rho_t)
            s_rot_star = float(s_rot_t)
            result_star = result_t
            updated_best = True

        # -------------------------------------------------
        # Save outer-iteration record
        # -------------------------------------------------
        record = Algorithm2IterationRecord(
            t=t,
            X_B_bar_before=X_B_before,
            X_full=X_full,
            X_trans=X_trans,
            X_rot=X_rot,
            rho_before=rho_before,
            s_rot_before=s_rot_before,
            A_rot=A_rot,
            n_candidates=len(C_t),
            branch_pool_sizes=branch_sizes,
            fallback_used=fallback_used,
            n_fallback_candidates=n_fallback,
            selected_branch=selected_cand.branch_name,
            selected_alpha=selected_cand.alpha,
            X_B_cand=copy_pose(X_next),
            rho_cand=float(selected_cand.rho_cand),
            rho_after=rho_next,
            s_rot_after=s_rot_next,
            delta_R=delta_R,
            delta_p=delta_p,
            delta_rho=delta_rho,
            accepted=True,
            updated_best=updated_best,
            stop_reason=None,
            alg1_result_after=result_next,
            health_metrics_after=health_next,
        )
        history.append(record)

        # -------------------------------------------------
        # Stop by small pose update and residual improvement
        # -------------------------------------------------
        # if (delta_R <= eps_R) and (delta_p <= eps_p) and (delta_rho <= eps_acc):
        #     converged = True
        #     stop_reason = "pose change and residual improvement tolerances satisfied"
        #     break

        # Stop by residual plateau
        # -------------------------------------------------
        abs_improve = max(float(delta_rho), 0.0)
        rel_improve = abs_improve / max(1.0, abs(float(rho_before)))

        small_residual_gain = (
            abs_improve <= rho_abs_tol
            or rel_improve <= rho_rel_tol
        )

        if (t + 1) >= min_outer:
            if small_residual_gain:
                plateau_count += 1
            else:
                plateau_count = 0

            if plateau_count >= plateau_patience:
                converged = True
                stop_reason = (
                    "residual plateau detected: "
                    f"relative improvement <= {rho_rel_tol:.1e} "
                    f"for {plateau_patience} consecutive accepted passes"
                )

                # Mark the final accepted record with the stopping reason.
                history[-1].stop_reason = stop_reason
                break

        # Optional hard stop if both pose and residual update are numerically tiny
        if (delta_R <= eps_R) and (delta_p <= eps_p) and (delta_rho <= eps_acc):
            converged = True
            stop_reason = "pose change and residual improvement tolerances satisfied"
            history[-1].stop_reason = stop_reason
            break



    return Algorithm2Result(
        X_B_star=X_B_star,
        Theta_star=Theta_star,
        chi_A_star_star=chi_A_star_star,
        rho_star=rho_star,
        s_rot_star=s_rot_star,

        X_B_bar_final=X_B_bar_t,
        Theta_post_final=Theta_post_t,
        chi_A_star_final=chi_A_star_t,
        rho_final=rho_t,
        s_rot_final=s_rot_t,

        result_init=result_init,
        result_final=result_t,
        history=history,

        n_outer_iterations=len(history),
        converged=converged,
        stop_reason=stop_reason,
    )

# Demo / Example /Generated Data

## ## R_z() R_y() R_x()

In [21]:
# =========================
# Synthetic batch data for implicit-equilibrium wrench measurements
# Designed for:
#   Experiment 1: sparse wrench measurements and algorithmic behavior
#   Experiment 2: measurement geometry and local rotational information
# =========================

from dataclasses import dataclass, replace
from typing import Optional, Sequence
import numpy as np


# =========================================================
# Rotation helpers
# =========================================================

def make_rotation_x(theta: float) -> np.ndarray:
    c = np.cos(theta)
    s = np.sin(theta)
    return np.array([
        [1.0, 0.0, 0.0],
        [0.0, c, -s],
        [0.0, s,  c],
    ], dtype=float)


def make_rotation_y(theta: float) -> np.ndarray:
    c = np.cos(theta)
    s = np.sin(theta)
    return np.array([
        [ c, 0.0, s],
        [0.0, 1.0, 0.0],
        [-s, 0.0, c],
    ], dtype=float)


def make_rotation_z(theta: float) -> np.ndarray:
    c = np.cos(theta)
    s = np.sin(theta)
    return np.array([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0],
    ], dtype=float)


def sample_gaussian_noise(
    cov: np.ndarray,
    rng: Optional[np.random.Generator] = None,
) -> np.ndarray:
    cov = ensure_matrix(cov, (6, 6))
    if rng is None:
        rng = np.random.default_rng()
    return rng.multivariate_normal(mean=np.zeros(6), cov=cov)


def so3_pairwise_angle(R1: np.ndarray, R2: np.ndarray) -> float:
    """
    Pairwise SO(3) angular distance.
    """
    R_rel = R1.T @ R2
    c = 0.5 * (np.trace(R_rel) - 1.0)
    c = float(np.clip(c, -1.0, 1.0))
    return float(np.arccos(c))


# =========================================================
# Synthetic data containers
# =========================================================

@dataclass
class SyntheticGeometrySummary:
    """
    Summary of probing/contact geometry.
    """
    design_name: str
    K: int

    position_span_xyz: np.ndarray
    position_span_norm: float
    min_pairwise_position_dist: float
    mean_pairwise_position_dist: float

    min_pairwise_angular_dist: float
    mean_pairwise_angular_dist: float
    max_pairwise_angular_dist: float

    contact_point_span_xyz: np.ndarray
    contact_point_span_norm: float
    contact_patch_radius: float


@dataclass
class SyntheticImplicitBatchData:
    """
    Synthetic dataset for the implicit-equilibrium wrench observation model.
    """
    X_B_true: Pose
    U: list[Pose]

    Y_clean: np.ndarray
    Y: np.ndarray

    chi_A_true: list[Pose]
    generation_results: list

    Sigma_w: np.ndarray
    sigma_tau: float
    sigma_force: float

    field_params: SuperquadricFieldParameters
    stiffness_params: StiffnessParameters
    ctrl_params: ControlPotentialParameters
    solver_config: ImplicitEquilibriumSolverConfig

    contact_points_A: np.ndarray

    X_B_bar_0: Pose
    Theta_0: MFGParameters

    geometry_design: str
    geometry_summary: SyntheticGeometrySummary
    design_params: dict


# =========================================================
# Contact patch geometry
# =========================================================

def build_contact_points_A(
    patch_type: str = "small_asymmetric_3d",
    patch_scale: float = 0.010,
) -> np.ndarray:
    """
    Contact patch points expressed in frame {A}.

    This controls contact-patch geometry, i.e., local lever arms and
    torque sensitivity.

    patch_type
    ----------
    "symmetric_cross":
        Useful as a simple baseline.
    "small_asymmetric_3d":
        Recommended default. Breaks symmetry mildly and improves rotational
        excitation without changing the object shape.
    "planar_patch":
        Mostly planar contact patch. Useful for testing weak out-of-plane
        rotational sensitivity.
    """
    s = float(patch_scale)

    if patch_type == "symmetric_cross":
        P = np.array([
            [ 0.0,  0.0,  0.0],
            [ 1.0,  0.0,  0.0],
            [-1.0,  0.0,  0.0],
            [ 0.0,  1.0,  0.0],
            [ 0.0, -1.0,  0.0],
            [ 0.0,  0.0,  1.0],
        ], dtype=float) * s

    elif patch_type == "small_asymmetric_3d":
        P = np.array([
            [ 0.00,  0.00,  0.00],
            [ 1.00,  0.10,  0.00],
            [-0.85,  0.25,  0.10],
            [ 0.15,  0.95, -0.05],
            [ 0.05, -0.90,  0.12],
            [ 0.35, -0.25,  0.80],
            [-0.25,  0.30, -0.55],
        ], dtype=float) * s

    elif patch_type == "planar_patch":
        P = np.array([
            [ 0.00,  0.00, 0.0],
            [ 1.00,  0.00, 0.0],
            [-1.00,  0.00, 0.0],
            [ 0.00,  1.00, 0.0],
            [ 0.00, -1.00, 0.0],
            [ 0.65,  0.65, 0.0],
            [-0.65, -0.65, 0.0],
        ], dtype=float) * s

    else:
        raise ValueError(f"Unknown patch_type: {patch_type}")

    return P


# =========================================================
# Control  geometry
# =========================================================

def get_control_geometry_preset(
    design: str,
    num_meas: int,
) -> dict:
    """
    Presets for probing/control geometry.

    These parameters control the measurement geometry, not the object shape.

    Main controls
    -------------
    yaw_halfspan_deg, pitch_halfspan_deg, roll_halfspan_deg:
        Directional richness of control poses.

    pos_scale_xyz:
        Spatial richness of control positions.

    phase_mix:
        Prevents all poses from lying on an overly regular one-dimensional path.
    """
    if design == "sparse_baseline":
        return dict(
            design=design,
            num_meas=num_meas,
            yaw_halfspan_deg=6.0,
            pitch_halfspan_deg=3.0,
            roll_halfspan_deg=0.0,
            pos_scale_xyz=np.array([0.0040, 0.0030, 0.0025]),
            phase_mix=0.8,
        )

    if design == "same_region_dense":
        # More samples, but almost the same local region.
        # This tests whether simply increasing K helps.
        return dict(
            design=design,
            num_meas=num_meas,
            yaw_halfspan_deg=6.0,
            pitch_halfspan_deg=3.0,
            roll_halfspan_deg=0.0,
            pos_scale_xyz=np.array([0.0040, 0.0030, 0.0025]),
            phase_mix=1.2,
        )

    if design == "direction_rich":
        # Stronger orientation diversity, similar position span.
        return dict(
            design=design,
            num_meas=num_meas,
            yaw_halfspan_deg=16.0,
            pitch_halfspan_deg=10.0,
            roll_halfspan_deg=8.0,
            pos_scale_xyz=np.array([0.0040, 0.0030, 0.0025]),
            phase_mix=1.1,
        )

    if design == "spatial_rich":
        # Stronger position diversity, similar orientation diversity.
        return dict(
            design=design,
            num_meas=num_meas,
            yaw_halfspan_deg=6.0,
            pitch_halfspan_deg=3.0,
            roll_halfspan_deg=0.0,
            pos_scale_xyz=np.array([0.0100, 0.0080, 0.0060]),
            phase_mix=1.1,
        )

    if design == "joint_rich":
        # Both spatially and directionally rich.
        return dict(
            design=design,
            num_meas=num_meas,
            yaw_halfspan_deg=16.0,
            pitch_halfspan_deg=10.0,
            roll_halfspan_deg=8.0,
            pos_scale_xyz=np.array([0.0100, 0.0080, 0.0060]),
            phase_mix=1.3,
        )

    raise ValueError(f"Unknown control geometry design: {design}")

    

def build_control_batch_by_geometry_design(
    X_B_true: Pose,
    design: str = "sparse_baseline",
    num_meas: int = 8,
    design_params_override: Optional[dict] = None,
) -> tuple[list[Pose], dict]:
    """
    Build control batch U = {X_{U,k}} around X_B_true.

    This is the main place where measurement geometry is controlled.

    Parameters
    ----------
    design:
        Preset name, e.g., "sparse_baseline", "direction_rich",
        "spatial_rich".

    design_params_override:
        Optional dictionary used to override the preset geometry parameters.
        This is useful for candidate sweeps in Experiment 2.

        Example:
            dict(
                yaw_halfspan_deg=12.0,
                pitch_halfspan_deg=8.0,
                roll_halfspan_deg=4.0,
                pos_scale_xyz=np.array([0.004, 0.003, 0.0025]),
                phase_mix=1.1,
            )

    Notes
    -----
    The override is allowed to change only the geometry-shaping parameters.
    The design name and number of measurements are kept consistent with the
    function arguments.
    """
    params = get_control_geometry_preset(design, num_meas)

    if design_params_override is not None:
        allowed_keys = {
            "yaw_halfspan_deg",
            "pitch_halfspan_deg",
            "roll_halfspan_deg",
            "pos_scale_xyz",
            "phase_mix",
        }

        unknown_keys = set(design_params_override.keys()) - allowed_keys
        if len(unknown_keys) > 0:
            raise ValueError(
                "Unknown keys in design_params_override: "
                f"{sorted(list(unknown_keys))}. "
                f"Allowed keys are: {sorted(list(allowed_keys))}."
            )

        params.update(design_params_override)

    # Keep these two bookkeeping fields consistent.
    params["design"] = design
    params["num_meas"] = int(num_meas)

    K = int(params["num_meas"])
    yaw_h = np.deg2rad(params["yaw_halfspan_deg"])
    pitch_h = np.deg2rad(params["pitch_halfspan_deg"])
    roll_h = np.deg2rad(params["roll_halfspan_deg"])
    pos_scale_xyz = np.asarray(params["pos_scale_xyz"], dtype=float)
    phase_mix = float(params["phase_mix"])

    U = []

    for k in range(K):
        if K == 1:
            s = 0.0
        else:
            s = -1.0 + 2.0 * k / (K - 1)

        # Directional probing geometry
        yaw = yaw_h * s
        pitch = pitch_h * np.sin(phase_mix * 0.8 * k)
        roll = roll_h * np.cos(phase_mix * 0.6 * k + 0.3)

        R_offset = (
            make_rotation_z(yaw)
            @ make_rotation_y(pitch)
            @ make_rotation_x(roll)
        )
        R_U_k = X_B_true.R @ R_offset

        # Spatial probing geometry
        dp = np.array([
            pos_scale_xyz[0] * np.cos(phase_mix * 0.7 * k),
            pos_scale_xyz[1] * np.sin(phase_mix * 0.9 * k),
            pos_scale_xyz[2] * np.cos(phase_mix * 1.1 * k + 0.2),
        ], dtype=float)

        p_U_k = X_B_true.p + dp

        U.append(Pose(R=R_U_k, p=p_U_k))

    return U, params

# =========================================================
# Geometry summary
# =========================================================

def summarize_synthetic_geometry(
    U: Sequence[Pose],
    contact_points_A: np.ndarray,
    design_name: str,
) -> SyntheticGeometrySummary:
    """
    Compute geometry summaries that can be reported in the paper.
    """
    K = len(U)
    positions = np.vstack([X.p for X in U])
    rotations = [X.R for X in U]

    pos_span_xyz = positions.max(axis=0) - positions.min(axis=0)
    pos_span_norm = float(np.linalg.norm(pos_span_xyz))

    pair_pos = []
    pair_ang = []

    for i in range(K):
        for j in range(i + 1, K):
            pair_pos.append(float(np.linalg.norm(positions[i] - positions[j])))
            pair_ang.append(so3_pairwise_angle(rotations[i], rotations[j]))

    if len(pair_pos) == 0:
        min_pair_pos = 0.0
        mean_pair_pos = 0.0
        min_pair_ang = 0.0
        mean_pair_ang = 0.0
        max_pair_ang = 0.0
    else:
        pair_pos = np.asarray(pair_pos)
        pair_ang = np.asarray(pair_ang)
        min_pair_pos = float(np.min(pair_pos))
        mean_pair_pos = float(np.mean(pair_pos))
        min_pair_ang = float(np.min(pair_ang))
        mean_pair_ang = float(np.mean(pair_ang))
        max_pair_ang = float(np.max(pair_ang))

    P = np.asarray(contact_points_A, dtype=float)
    contact_span_xyz = P.max(axis=0) - P.min(axis=0)
    contact_span_norm = float(np.linalg.norm(contact_span_xyz))
    contact_radius = float(np.max(np.linalg.norm(P - P.mean(axis=0), axis=1)))

    return SyntheticGeometrySummary(
        design_name=design_name,
        K=K,
        position_span_xyz=pos_span_xyz,
        position_span_norm=pos_span_norm,
        min_pairwise_position_dist=min_pair_pos,
        mean_pairwise_position_dist=mean_pair_pos,
        min_pairwise_angular_dist=min_pair_ang,
        mean_pairwise_angular_dist=mean_pair_ang,
        max_pairwise_angular_dist=max_pair_ang,
        contact_point_span_xyz=contact_span_xyz,
        contact_point_span_norm=contact_span_norm,
        contact_patch_radius=contact_radius,
    )


def print_geometry_summary(summary: SyntheticGeometrySummary) -> None:
    print("-" * 84)
    print("Control/contact geometry summary")
    print("-" * 84)
    print(f"design                         : {summary.design_name}")
    print(f"K                              : {summary.K}")
    print(f"position span xyz              : {summary.position_span_xyz}")
    print(f"position span norm             : {summary.position_span_norm:.6e}")
    print(f"min pairwise position distance : {summary.min_pairwise_position_dist:.6e}")
    print(f"mean pairwise position distance: {summary.mean_pairwise_position_dist:.6e}")
    print(f"min pairwise angular distance  : {summary.min_pairwise_angular_dist:.6e}")
    print(f"mean pairwise angular distance : {summary.mean_pairwise_angular_dist:.6e}")
    print(f"max pairwise angular distance  : {summary.max_pairwise_angular_dist:.6e}")
    print(f"contact point span xyz         : {summary.contact_point_span_xyz}")
    print(f"contact patch radius           : {summary.contact_patch_radius:.6e}")


# =========================================================
# Noise-free measurement generation
# =========================================================

def generate_clean_measurement_batch_from_truth_robust(
    X_B_true: Pose,
    U: Sequence[Pose],
    contact_points_A: np.ndarray,
    field_params: SuperquadricFieldParameters,
    stiffness_params: StiffnessParameters,
    ctrl_params: ControlPotentialParameters,
    solver_config: Optional[ImplicitEquilibriumSolverConfig] = None,
) -> tuple[np.ndarray, list[Pose], list]:
    """
    Generate noise-free measurements

        Y_clean = { y_hat_k(X_B_true) }_{k=1}^K

    by solving the implicit equilibrium state X_{A,k}^* for each k.
    """
    if solver_config is None:
        solver_config = ImplicitEquilibriumSolverConfig()

    Y_clean_list = []
    chi_A_true = []
    generation_results = []

    prev_sample_same_outer = None

    retry_configs = [
        solver_config,
        replace(
            solver_config,
            max_iter=max(solver_config.max_iter, 80),
            hessian_damping=max(solver_config.hessian_damping, 1e-5),
        ),
        replace(
            solver_config,
            max_iter=max(solver_config.max_iter, 120),
            hessian_damping=max(solver_config.hessian_damping, 1e-4),
        ),
    ]

    for k, X_U_k in enumerate(U):
        res_k = None

        for cfg_try in retry_configs:
            res_try = solve_implicit_equilibrium_and_linearization(
                X_B=X_B_true,
                X_U_k=X_U_k,
                contact_points_A=contact_points_A,
                field_params=field_params,
                stiffness_params=stiffness_params,
                ctrl_params=ctrl_params,
                solver_config=cfg_try,
                prev_sample_same_outer=prev_sample_same_outer,
            )

            res_k = res_try
            if res_try.converged:
                break

        Y_clean_list.append(res_k.y_hat.copy())
        chi_A_true.append(res_k.X_A_star)
        generation_results.append(res_k)

        prev_sample_same_outer = res_k.X_A_star

    Y_clean = np.vstack(Y_clean_list)
    return Y_clean, chi_A_true, generation_results


# =========================================================
# Main synthetic-data builder
# =========================================================
# =========================================================
# Main synthetic-data builder
# =========================================================

def make_synthetic_implicit_batch_data(
    num_meas: int = 8,
    random_seed: int = 42,
    geometry_design: str = "sparse_baseline",
    design_params_override: Optional[dict] = None,
    contact_patch_type: str = "small_asymmetric_3d",
    contact_patch_scale: float = 0.010,

    # Actual data noise. Keep fixed across geometry experiments.
    sigma_tau: float = 0.10,
    sigma_force: float = 0.70,

    # Initial error controls inference difficulty, not measurement geometry.
    init_yaw_error_deg: float = -6.0,
    init_pitch_error_deg: float = 4.0,
    init_roll_error_deg: float = 2.0,   # NEW: added x-axis right perturbation
    init_translation_error: np.ndarray = np.array([-0.005, 0.0035, -0.004]),

    # Prior confidence controls Bayesian regularization.
    kappa0: float = 18.0,
    Lambda0_diag: tuple[float, float, float] = (1800.0, 1800.0, 1800.0),
    Gamma0: Optional[np.ndarray] = None,

    solver_config_override: Optional[ImplicitEquilibriumSolverConfig] = None,
    verbose: bool = True,
) -> SyntheticImplicitBatchData:
    """
    Build a synthetic batch dataset for the implicit-equilibrium wrench model.

    Important experimental controls
    -------------------------------
    1) Measurement sparsity:
        num_meas = K.

    2) Measurement/contact geometry:
        geometry_design controls the default control batch U.
        design_params_override optionally overrides the control-geometry
        parameters used to construct U. This is used for candidate sweeps
        in Experiment 2.
        contact_patch_type and contact_patch_scale control contact_points_A.

    3) Object shape:
        field_params controls the superquadric/SDF geometry.

    4) Noise:
        sigma_tau and sigma_force control wrench noise.

    5) Inference difficulty:
        init_yaw_error_deg,
        init_pitch_error_deg,
        init_roll_error_deg,
        init_translation_error.
    """
    rng = np.random.default_rng(random_seed)

    # -----------------------------------------------------
    # A. Object geometry: fixed superquadric implicit field
    # -----------------------------------------------------
    field_params = SuperquadricFieldParameters(
        a1=0.040,
        a2=0.030,
        a3=0.050,
        eps1=1.0,
        eps2=1.0,
        sdf_eps=1e-8,
    )

    # -----------------------------------------------------
    # B. Contact stiffness profile: physical sensitivity
    # -----------------------------------------------------
    stiffness_params = StiffnessParameters(
        k_min=0.10,
        k_max=1.00,
        d0=0.05,
    )

    # -----------------------------------------------------
    # C. Control potential: virtual spring stiffness
    # -----------------------------------------------------
    ctrl_params = ControlPotentialParameters(
        K_R=8.0 * np.eye(3),
        K_p=600.0 * np.eye(3),
    )

    # -----------------------------------------------------
    # D. Inner equilibrium solver configuration
    # -----------------------------------------------------
    # solver_config = ImplicitEquilibriumSolverConfig(
    #     max_iter=60,
    #     tol_eq=1e-7,
    #     tol_step=1e-7,
    #     fd_rot_step=1e-6,
    #     fd_pos_step=1e-6,
    #     hessian_damping=1e-6,
    # )
    if solver_config_override is None:
        solver_config = ImplicitEquilibriumSolverConfig(
            max_iter=100,
            tol_eq=1e-7,
            tol_step=1e-8,
            fd_rot_step=1e-6,
            fd_pos_step=1e-6,
            hessian_damping=1e-6,
        )
    else:
        solver_config = solver_config_override

    # -----------------------------------------------------
    # E. Contact geometry in frame {A}
    # -----------------------------------------------------
    contact_points_A = build_contact_points_A(
        patch_type=contact_patch_type,
        patch_scale=contact_patch_scale,
    )

    # -----------------------------------------------------
    # F. Ground-truth object pose
    # -----------------------------------------------------
    # NEW:
    # Full three-axis ground-truth attitude:
    #
    #     R_B_true = R_z(14 deg) R_y(-6 deg) R_x(4 deg).
    #
    R_true = (
        make_rotation_z(np.deg2rad(14.0))
        @ make_rotation_y(np.deg2rad(-6.0))
        @ make_rotation_x(np.deg2rad(4.0))
    )

    p_true = np.array([0.015, -0.010, 0.020], dtype=float)
    X_B_true = Pose(R=R_true, p=p_true)

    # -----------------------------------------------------
    # G. Control geometry
    # -----------------------------------------------------
    U, design_params = build_control_batch_by_geometry_design(
        X_B_true=X_B_true,
        design=geometry_design,
        num_meas=num_meas,
        design_params_override=design_params_override,
    )

    geometry_summary = summarize_synthetic_geometry(
        U=U,
        contact_points_A=contact_points_A,
        design_name=geometry_design,
    )

    # -----------------------------------------------------
    # H. Observation noise covariance
    # -----------------------------------------------------
    Sigma_w = np.diag([
        sigma_tau**2, sigma_tau**2, sigma_tau**2,
        sigma_force**2, sigma_force**2, sigma_force**2,
    ])

    # -----------------------------------------------------
    # I. Generate clean measurements from truth
    # -----------------------------------------------------
    Y_clean, chi_A_true, generation_results = generate_clean_measurement_batch_from_truth_robust(
        X_B_true=X_B_true,
        U=U,
        contact_points_A=contact_points_A,
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        solver_config=solver_config,
    )

    # -----------------------------------------------------
    # J. Add data noise
    # -----------------------------------------------------
    Y = []
    for k in range(num_meas):
        noise_k = sample_gaussian_noise(Sigma_w, rng=rng)
        Y.append(Y_clean[k] + noise_k)
    Y = np.asarray(Y, dtype=float)

    # -----------------------------------------------------
    # K. Initial nominal pose for inference
    # -----------------------------------------------------
    # NEW:
    # Right rotational perturbation now contains z-y-x components:
    #
    #     R_bar_0
    #     = R_true
    #       R_z(init_yaw_error_deg)
    #       R_y(init_pitch_error_deg)
    #       R_x(init_roll_error_deg).
    #
    R0 = (
        X_B_true.R
        @ make_rotation_z(np.deg2rad(init_yaw_error_deg))
        @ make_rotation_y(np.deg2rad(init_pitch_error_deg))
        @ make_rotation_x(np.deg2rad(init_roll_error_deg))
    )

    p0 = X_B_true.p + np.asarray(init_translation_error, dtype=float)
    X_B_bar_0 = Pose(R=R0, p=p0)

    # -----------------------------------------------------
    # L. Initial coupled MFG prior
    # -----------------------------------------------------
    F0 = kappa0 * X_B_bar_0.R
    mu0 = X_B_bar_0.p.copy()
    Lambda0 = np.diag(np.asarray(Lambda0_diag, dtype=float))

    if Gamma0 is None:
        Gamma0 = np.zeros((3, 3), dtype=float)

    Theta_0 = MFGParameters(
        F=F0,
        mu=mu0,
        Lambda=Lambda0,
        Gamma=Gamma0,
    )

    data = SyntheticImplicitBatchData(
        X_B_true=X_B_true,
        U=list(U),
        Y_clean=Y_clean,
        Y=Y,
        chi_A_true=chi_A_true,
        generation_results=generation_results,
        Sigma_w=Sigma_w,
        sigma_tau=float(sigma_tau),
        sigma_force=float(sigma_force),
        field_params=field_params,
        stiffness_params=stiffness_params,
        ctrl_params=ctrl_params,
        solver_config=solver_config,
        contact_points_A=contact_points_A,
        X_B_bar_0=X_B_bar_0,
        Theta_0=Theta_0,
        geometry_design=geometry_design,
        geometry_summary=geometry_summary,
        design_params=design_params,
    )

    if verbose:
        print("=" * 96)
        print("Synthetic implicit batch data summary")
        print("=" * 96)

        print(f"geometry design                         : {geometry_design}")
        print(f"num_meas K                              : {num_meas}")
        print(f"random seed                             : {random_seed}")
        print(f"Y shape                                 : {data.Y.shape}")
        print(f"Y_clean shape                           : {data.Y_clean.shape}")
        print(
            "all generation solves converged         : "
            f"{sum(int(r.converged) for r in data.generation_results)}/{num_meas}"
        )

        print("-" * 96)
        print("Ground-truth object pose")
        print("-" * 96)
        print(
            "R_B_true construction                   : "
            "R_z(14 deg) @ R_y(-6 deg) @ R_x(4 deg)"
        )
        print(f"X_B_true.p                              : {data.X_B_true.p}")

        print("-" * 96)
        print("Initial nominal pose")
        print("-" * 96)
        print(
            "right rotational perturbation           : "
            f"R_z({init_yaw_error_deg:+.3f} deg) "
            f"@ R_y({init_pitch_error_deg:+.3f} deg) "
            f"@ R_x({init_roll_error_deg:+.3f} deg)"
        )
        print(f"initial translation perturbation        : {np.asarray(init_translation_error, dtype=float)}")
        print(f"X_B_bar_0.p                             : {data.X_B_bar_0.p}")
        print(
            f"initial rotation error (rad)            : "
            f"{rotation_geodesic_error(data.X_B_bar_0.R, data.X_B_true.R):.6e}"
        )
        print(
            f"initial translation error               : "
            f"{pose_translation_error(data.X_B_bar_0, data.X_B_true):.6e}"
        )

        print("-" * 96)
        print("Noise and prior setup")
        print("-" * 96)
        print(f"sigma_tau                               : {sigma_tau}")
        print(f"sigma_force                             : {sigma_force}")
        print(f"diag(Sigma_w)                           : {np.diag(data.Sigma_w)}")
        print(f"kappa0                                  : {kappa0}")
        print(f"mu0                                     : {data.Theta_0.mu}")
        print(f"diag(Lambda0)                           : {np.diag(data.Theta_0.Lambda)}")
        print(f"||Gamma0||_F                            : {np.linalg.norm(data.Theta_0.Gamma, ord='fro'):.6e}")

        print_geometry_summary(data.geometry_summary)
        print("=" * 96)

    return data


# 以下是数值试验章节的正式实验

## global experiment config

In [22]:
# ============================================================
# Final Experiment-1 global configuration
# ZYX three-axis perturbation + strict inner solve + plateau Alg2
# ============================================================

import numpy as np

# Candidate K list for nested K-ablation
EXP1_K_LIST = [6, 8, 10, 12, 16, 20, 24]
EXP1_K_MAX = max(EXP1_K_LIST)

# Multi-seed setting
EXP1_SEEDS = [11, 22, 33, 44, 55, 66, 77, 88, 99, 111]

# Final ZYX initial perturbation
EXP1_INIT_CFG = dict(
    init_yaw_error_deg=-6.0,
    init_pitch_error_deg=4.0,
    init_roll_error_deg=2.0,
    init_translation_error=np.array([-0.005, 0.0035, -0.004], dtype=float),
)

# Common data-generation setting
EXP1_DATA_CFG = dict(
    geometry_design="sparse_baseline",
    contact_patch_type="small_asymmetric_3d",
    contact_patch_scale=0.010,
    sigma_tau=0.10,
    sigma_force=0.70,
    kappa0=60.0,
    Lambda0_diag=(6000.0, 6000.0, 6000.0),
)

# Strict but practical inner equilibrium solver
EXP1_SOLVER_CFG = ImplicitEquilibriumSolverConfig(
    max_iter=100,
    tol_eq=1e-7,
    tol_step=1e-8,
    fd_rot_step=1e-6,
    fd_pos_step=1e-6,
    hessian_damping=1e-6,
)

# Algorithm 2 common setting
EXP1_ALG2_CFG = dict(
    T_max=15,
    A=(0.25, 0.10, 0.05, 0.02, 0.01, 0.005),
    eps_acc=1e-6,
    eps_R=1e-6,
    eps_p=1e-6,
    eps_info=1e-8,
    eps_g=1e-8,
    lambda_fb=1e-4,
    A_rot_reduction_factor=0.25,

    # New plateau stopping
    rho_rel_tol=5e-4,
    rho_abs_tol=1e-4,
    plateau_patience=3,
    min_outer=4,
)

# Algorithm 2 health-check setting
EXP1_ALG2_HEALTH_CFG = dict(
    require_all_inner_converged=True,

    # Keep W_AA diagnostics, but do not make PSD a hard rejection initially.
    # If this is set True, many valid contact cases may be rejected due to local curvature.
    require_psd_WAA=False,

    max_cond_WAA=1e8,
    min_min_eig_WAA=-1e-8,
    max_J_phi_norm=1e8,
    max_J_v_norm=1e8,
    max_eq_res_norm=1e-5,
    max_eq_step_norm=1e-4,

    min_eig_Lambda_post=1e-10,
    max_cond_Lambda_post=1e10,
    max_norm_H=1e12,
    max_norm_M_post=1e12,
    max_norm_a_post=1e10,
    max_norm_F_post=1e10,
    max_mode_nu_norm=1e4,
)

# Proposition 3.4 --代入3.4中陈述的准则和条件，选择最实惠的K*

## 运行proposition 3.4 --- 进行稀疏的K -selection 

## 新版

In [23]:
# ============================================================
# Proposition 3.4:
# Nested K sweep for measurement-efficient sparse-baseline selection
# ============================================================
#
# Purpose:
#   Select K_star using the measurement-efficient criterion:
#
#       s_rot(K)        = lambda_min(I_rot(K))
#       Delta_K^+(K)    = (K_next - K) / K
#       Delta_s^+(K)    = (s_rot(K_next) - s_rot(K)) / (s_rot(K) + eps)
#       eta_s^+(K)      = Delta_s^+(K) / (Delta_K^+(K) + eps)
#
# It generates the largest nested batch internally and then takes subsets.
# ============================================================


# ============================================================
# Patch: safe table printer for records
# Use this if print_records_table is not defined in the current notebook.
# ============================================================

import numpy as np


def print_records_table(
    records,
    columns,
    title=None,
    max_width=18,
    precision=4,
):
    """
    Print a list of dict records as a simple aligned text table.

    This function is self-contained and does not depend on any previous
    notebook helper.
    """
    if title is not None:
        print("\n" + title)

    if records is None or len(records) == 0:
        print("[Empty records]")
        return

    def format_value(v):
        if v is None:
            return "None"

        if isinstance(v, bool):
            return str(v)

        if isinstance(v, (int, np.integer)):
            return str(int(v))

        if isinstance(v, (float, np.floating)):
            v = float(v)
            if not np.isfinite(v):
                return str(v)
            # Scientific notation for very small or large values
            if abs(v) >= 1e3 or (abs(v) > 0 and abs(v) < 1e-3):
                return f"{v:.{precision}e}"
            return f"{v:.{precision}f}"

        return str(v)

    # Convert table to strings
    table = []
    for rec in records:
        row = []
        for col in columns:
            row.append(format_value(rec.get(col, None)))
        table.append(row)

    # Column widths
    widths = []
    for j, col in enumerate(columns):
        max_len = len(str(col))
        for row in table:
            max_len = max(max_len, len(row[j]))
        widths.append(min(max_len, max_width))

    def trim(s, w):
        s = str(s)
        if len(s) <= w:
            return s
        if w <= 3:
            return s[:w]
        return s[:w - 3] + "..."

    # Header
    header = " | ".join(trim(col, widths[j]).ljust(widths[j]) for j, col in enumerate(columns))
    sep = "-+-".join("-" * widths[j] for j in range(len(columns)))

    print(header)
    print(sep)

    for row in table:
        print(" | ".join(trim(row[j], widths[j]).ljust(widths[j]) for j in range(len(columns))))



import copy
import json
import numpy as np


# ------------------------------------------------------------
# 0. User settings for Proposition 3.4
# ------------------------------------------------------------

K_list = [6, 8, 10, 12, 16, 20, 24]
K_max = max(K_list)

# Candidate selection thresholds.
# These should match the statement of Proposition 3.4.
#
# tau_s:
#   required minimum rotational information.
#
# tau_delta:
#   maximum allowed next-step relative information gain.
#   Small value means the current K is already near an information plateau.
#
# tau_eta:
#   maximum allowed measurement-normalized marginal gain.
#   Small value means adding more measurements is not efficient.
#

tau_s = 2.50
tau_delta = 0.10
tau_eta = 0.50

eps_metric = 1e-12


# ------------------------------------------------------------
# 1. Generate the largest sparse-baseline batch once
# ------------------------------------------------------------
# This uses exactly the current Experiment 1 / baseline settings:
#   - sparse_baseline control geometry
#   - K_max measurements
#   - fixed SDF / stiffness / noise model
#   - fixed initial nominal perturbation
#   - fixed MFG prior confidence
# ------------------------------------------------------------


data_max = make_synthetic_implicit_batch_data(
    num_meas=K_max,
    random_seed=42,
    **EXP1_DATA_CFG,
    **EXP1_INIT_CFG,
    solver_config_override=EXP1_SOLVER_CFG,
    verbose=False,
)

# ------------------------------------------------------------
# 2. Helper: create nested subset data
# ------------------------------------------------------------

def make_nested_subset_data(data_full, K_subset):
    """
    Create a nested K-subset from the full synthetic batch.

    This keeps the same:
        object shape,
        stiffness model,
        control-potential parameters,
        contact patch,
        noise covariance,
        initial nominal pose,
        MFG prior,

    but uses only the first K_subset measurements.

    This realizes:
        D_6 subset D_8 subset D_10 subset ... subset D_24.
    """
    K_subset = int(K_subset)
    data_sub = copy.deepcopy(data_full)

    # Main batch fields
    data_sub.U = list(data_full.U[:K_subset])
    data_sub.Y = np.asarray(data_full.Y[:K_subset], dtype=float)

    if hasattr(data_full, "Y_clean"):
        data_sub.Y_clean = np.asarray(data_full.Y_clean[:K_subset], dtype=float)

    if hasattr(data_full, "chi_A_true"):
        data_sub.chi_A_true = list(data_full.chi_A_true[:K_subset])

    if hasattr(data_full, "generation_results"):
        data_sub.generation_results = list(data_full.generation_results[:K_subset])

    # Recompute geometry summary for the subset if the helper exists.
    # This is safer than only changing K in the old summary.
    try:
        data_sub.geometry_summary = summarize_synthetic_geometry(
            U=data_sub.U,
            contact_points_A=data_sub.contact_points_A,
            design_name=data_sub.geometry_design,
        )
    except Exception:
        # Fallback: update K if geometry_summary is a mutable object.
        if hasattr(data_sub, "geometry_summary"):
            try:
                data_sub.geometry_summary.K = K_subset
            except Exception:
                pass

    # Update bookkeeping
    if hasattr(data_sub, "design_params") and isinstance(data_sub.design_params, dict):
        data_sub.design_params = dict(data_sub.design_params)
        data_sub.design_params["K"] = K_subset
        data_sub.design_params["num_meas"] = K_subset

    return data_sub


# ------------------------------------------------------------
# 3. Helper: JSON-safe conversion
# ------------------------------------------------------------

def make_json_safe(x):
    """
    Convert numpy / None / inf values into JSON-safe objects.
    """
    if isinstance(x, dict):
        return {str(k): make_json_safe(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [make_json_safe(v) for v in x]
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, np.generic):
        return x.item()
    if isinstance(x, float):
        if np.isfinite(x):
            return float(x)
        return str(x)
    if isinstance(x, (int, str, bool)) or x is None:
        return x
    return str(x)


# ------------------------------------------------------------
# 4. Run Algorithm 1 for each nested K
# ------------------------------------------------------------

k_info_records = []

for K_test in K_list:
    print("\n" + "=" * 90)
    print(f"Nested local information test for Proposition 3.4: K={K_test}")
    print("=" * 90)

    data_test = make_nested_subset_data(data_max, K_test)

    res1_test = algorithm1_single_pass_local_mfg_posterior_update(
        Theta_prior=data_test.Theta_0,
        X_B_bar=data_test.X_B_bar_0,
        U=data_test.U,
        Y=data_test.Y,
        field_params=data_test.field_params,
        stiffness_params=data_test.stiffness_params,
        ctrl_params=data_test.ctrl_params,
        Sigma_w=data_test.Sigma_w,
        contact_points_A_shared=data_test.contact_points_A,
        solver_config=data_test.solver_config,
        chi_A_0=None,
        use_psd_mf_curvature=True,
    )

    eig_I = np.linalg.eigvalsh(res1_test.I_rot)

    eig1 = float(eig_I[0])
    eig2 = float(eig_I[1])
    eig3 = float(eig_I[2])

    s_rot = eig1

    if eig1 > eps_metric:
        cond_I_rot = float(eig3 / eig1)
    else:
        cond_I_rot = np.inf

    rho = float(res1_test.rho)
    rho_norm = float(rho / np.sqrt(6.0 * K_test))

    n_conv = int(res1_test.batch_result.n_converged)
    n_total = int(len(res1_test.batch_result.sample_results))
    all_inner_converged = (n_conv == n_total)

    record = {
        "K": int(K_test),

        # Main Proposition 3.4 quantities
        "s_rot": float(s_rot),
        "Delta_K_plus": None,
        "Delta_s_plus": None,
        "eta_s_plus": None,

        # Diagnostics
        "eig1": eig1,
        "eig2": eig2,
        "eig3": eig3,
        "cond_I_rot": cond_I_rot,
        "rho": rho,
        "rho_norm": rho_norm,
        "inner_conv": f"{n_conv}/{n_total}",
        "all_inner_converged": bool(all_inner_converged),

        # Selection flag filled later
        "selected": False,
    }

    k_info_records.append(record)


# ------------------------------------------------------------
# 5. Compute next-step marginal quantities
# ------------------------------------------------------------
# Delta_K^+(K_i)
# Delta_s^+(K_i)
# eta_s^+(K_i)
# ------------------------------------------------------------

for i in range(len(k_info_records)):
    if i < len(k_info_records) - 1:
        K_now = float(k_info_records[i]["K"])
        K_next = float(k_info_records[i + 1]["K"])

        s_now = float(k_info_records[i]["s_rot"])
        s_next = float(k_info_records[i + 1]["s_rot"])

        Delta_K_plus = (K_next - K_now) / max(K_now, eps_metric)
        Delta_s_plus = (s_next - s_now) / (abs(s_now) + eps_metric)
        eta_s_plus = Delta_s_plus / (Delta_K_plus + eps_metric)

        k_info_records[i]["K_next"] = int(K_next)
        k_info_records[i]["Delta_K_plus"] = float(Delta_K_plus)
        k_info_records[i]["Delta_s_plus"] = float(Delta_s_plus)
        k_info_records[i]["eta_s_plus"] = float(eta_s_plus)

    else:
        k_info_records[i]["K_next"] = None
        k_info_records[i]["Delta_K_plus"] = None
        k_info_records[i]["Delta_s_plus"] = None
        k_info_records[i]["eta_s_plus"] = None


# ------------------------------------------------------------
# 6. Select K_star according to Proposition 3.4
# ------------------------------------------------------------


K_star_prop34 = None
selected_record_prop34 = None



for rec in k_info_records:
    if rec["K_next"] is None:
        continue

    pass_s = rec["s_rot"] >= tau_s
    pass_delta = rec["Delta_s_plus"] <= tau_delta
    pass_eta = rec["eta_s_plus"] <= tau_eta
    pass_inner = rec["all_inner_converged"]

    rec["pass_s"] = bool(pass_s)
    rec["pass_delta"] = bool(pass_delta)
    rec["pass_eta"] = bool(pass_eta)
    rec["pass_inner"] = bool(pass_inner)

    
    if pass_s and pass_delta and pass_eta and pass_inner and K_star_prop34 is None:
        K_star_prop34 = rec["K"]
        selected_record_prop34 = rec
        rec["selected"] = True
    else:
        rec["selected"] = False

# Last row has no next-step marginal values.
if k_info_records[-1].get("K_next", None) is None:
    k_info_records[-1]["pass_s"] = None
    k_info_records[-1]["pass_delta"] = None
    k_info_records[-1]["pass_eta"] = None
    k_info_records[-1]["pass_inner"] = k_info_records[-1]["all_inner_converged"]



# ============================================================
# Additional threshold-free reference choices
# ============================================================

valid_records = [
    rec for rec in k_info_records
    if rec["all_inner_converged"]
    and rec["s_rot"] is not None
    and np.isfinite(rec["s_rot"])
]

# Information-maximizing K over the tested list
rec_max_info = max(valid_records, key=lambda r: r["s_rot"])
K_max_info = rec_max_info["K"]

# Average information per measurement
for rec in k_info_records:
    if rec["s_rot"] is not None and np.isfinite(rec["s_rot"]):
        rec["s_rot_per_K"] = rec["s_rot"] / rec["K"]
    else:
        rec["s_rot_per_K"] = None

valid_records_ratio = [
    rec for rec in valid_records
    if rec["s_rot_per_K"] is not None
    and np.isfinite(rec["s_rot_per_K"])
]

rec_best_ratio = max(valid_records_ratio, key=lambda r: r["s_rot_per_K"])
K_best_ratio = rec_best_ratio["K"]

print("\n" + "=" * 90)
print("Threshold-free reference choices")
print("=" * 90)
print(f"K_max_info   = {K_max_info}  with s_rot = {rec_max_info['s_rot']:.4f}")
print(f"K_best_ratio = {K_best_ratio}  with s_rot/K = {rec_best_ratio['s_rot_per_K']:.4f}")



# ------------------------------------------------------------
# 7. Print full diagnostic table
# ------------------------------------------------------------

print_records_table(
    k_info_records,
    [
        "K",
        "K_next",
        "s_rot",
        "s_rot_per_K",
        "Delta_K_plus",
        "Delta_s_plus",
        "eta_s_plus",
        "rho_norm",
        "inner_conv",
        "pass_s",
        "pass_delta",
        "pass_eta",
        "selected",
    ],
    title="Proposition 3.4 nested K sweep: measurement-efficient sparse-baseline selection",
    max_width=18,
    precision=4,
)


# ------------------------------------------------------------
# 8. Print compact table
# ------------------------------------------------------------

paper_records = []

for rec in k_info_records:
    paper_records.append({
        "K": rec["K"],
        "s_rot": rec["s_rot"],
        "Delta_K_plus": rec["Delta_K_plus"],
        "Delta_s_plus": rec["Delta_s_plus"],
        "eta_s_plus": rec["eta_s_plus"],
        "rho_norm": rec["rho_norm"],
        "selected": "*" if rec["selected"] else "",
    })

print_records_table(
    paper_records,
    [
        "K",
        "s_rot",
        "Delta_K_plus",
        "Delta_s_plus",
        "eta_s_plus",
        "rho_norm",
        "selected",
    ],
    title="Compact table for paper: Proposition 3.4 K selection",
    max_width=18,
    precision=4,
)


# ------------------------------------------------------------
# ------------------------------------------------------------
# 9. Print final selection result
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("Proposition 3.4 selection result")
print("=" * 90)

print(f"tau_s     = {tau_s}")
print(f"tau_delta = {tau_delta}")
print(f"tau_eta   = {tau_eta}")

if K_star_prop34 is None:
    print("\n[Warning] No candidate K satisfies all Proposition 3.4 criteria.")
    print("Please relax tau_s, tau_delta, or tau_eta, or enlarge K_list.")
else:
    print(f"\nSelected sparse-baseline measurement number: K_star_prop34 = {K_star_prop34}")
    print("Selected record:")

    for key in [
        "K",
        "s_rot",
        "Delta_K_plus",
        "Delta_s_plus",
        "eta_s_plus",
        "rho_norm",
        "inner_conv",
    ]:
        print(f"  {key:16s}: {selected_record_prop34.get(key, None)}")


# ------------------------------------------------------------
# 10. Save records as JSON
# ------------------------------------------------------------

with open("prop34_nested_k_sweep_records.json", "w") as f:
    json.dump(make_json_safe(k_info_records), f, indent=2)

print("\n[Saved] prop34_nested_k_sweep_records.json")


Nested local information test for Proposition 3.4: K=6

Nested local information test for Proposition 3.4: K=8

Nested local information test for Proposition 3.4: K=10

Nested local information test for Proposition 3.4: K=12

Nested local information test for Proposition 3.4: K=16

Nested local information test for Proposition 3.4: K=20

Nested local information test for Proposition 3.4: K=24

Threshold-free reference choices
K_max_info   = 24  with s_rot = 37.6852
K_best_ratio = 20  with s_rot/K = 1.7741

Proposition 3.4 nested K sweep: measurement-efficient sparse-baseline selection
K  | K_next | s_rot   | s_rot_per_K | Delta_K_plus | Delta_s_plus | eta_s_plus | rho_norm | inner_conv | pass_s | pass_delta | pass_eta | selected
---+--------+---------+-------------+--------------+--------------+------------+----------+------------+--------+------------+----------+---------
6  | 8      | 1.7077  | 0.2846      | 0.3333       | 3.3936       | 10.1807    | 2.8667   | 6/6        | False  |

In [24]:
print("\n" + "=" * 80)
print("Check final ZYX Experiment-1 configuration")
print("=" * 80)

print("Expected init yaw   :", EXP1_INIT_CFG["init_yaw_error_deg"])
print("Expected init pitch :", EXP1_INIT_CFG["init_pitch_error_deg"])
print("Expected init roll  :", EXP1_INIT_CFG["init_roll_error_deg"])
print("Expected init trans :", EXP1_INIT_CFG["init_translation_error"])

print("\nActual X_B_true.p   :", data_max.X_B_true.p)
print("Actual X_B_bar_0.p  :", data_max.X_B_bar_0.p)
print("Actual trans error  :", data_max.X_B_bar_0.p - data_max.X_B_true.p)

expected_p0 = data_max.X_B_true.p + EXP1_INIT_CFG["init_translation_error"]
print("Expected X_B_bar_0.p:", expected_p0)

print("\nSolver config:")
print("max_iter :", data_max.solver_config.max_iter)
print("tol_eq   :", data_max.solver_config.tol_eq)
print("tol_step :", data_max.solver_config.tol_step)

assert np.allclose(
    data_max.X_B_bar_0.p,
    expected_p0,
), "Translation perturbation does not match EXP1_INIT_CFG."

assert data_max.solver_config.max_iter == EXP1_SOLVER_CFG.max_iter, \
    "solver_config_override was not applied."

assert np.isclose(data_max.solver_config.tol_step, EXP1_SOLVER_CFG.tol_step), \
    "solver_config tol_step does not match EXP1_SOLVER_CFG."

print("\n[OK] Final ZYX Experiment-1 configuration is active.")


Check final ZYX Experiment-1 configuration
Expected init yaw   : -6.0
Expected init pitch : 4.0
Expected init roll  : 2.0
Expected init trans : [-0.005   0.0035 -0.004 ]

Actual X_B_true.p   : [ 0.015 -0.01   0.02 ]
Actual X_B_bar_0.p  : [ 0.01   -0.0065  0.016 ]
Actual trans error  : [-0.005   0.0035 -0.004 ]
Expected X_B_bar_0.p: [ 0.01   -0.0065  0.016 ]

Solver config:
max_iter : 100
tol_eq   : 1e-07
tol_step : 1e-08

[OK] Final ZYX Experiment-1 configuration is active.


In [25]:
# ============================================================
# Final K choices from Proposition 3.4
# ============================================================

K_eff = K_star_prop34        # measurement-efficient sparse baseline
K_high = K_best_ratio        # high-information reference by max s_rot/K
K_max_info_ref = K_max_info  # maximum-information reference

print("\nFinal K choices for Experiment 1:")
print(f"K_eff          = {K_eff}  # sparse-efficient operating point")
print(f"K_high         = {K_high}  # best s_rot/K high-information reference")
print(f"K_max_info_ref = {K_max_info_ref}  # maximum s_rot reference")


Final K choices for Experiment 1:
K_eff          = 10  # sparse-efficient operating point
K_high         = 20  # best s_rot/K high-information reference
K_max_info_ref = 24  # maximum s_rot reference


# Experiement 2: Different  Noise- level 

## 2C : Large Initial Error with Falsely High Confidence

In [27]:
def exp2C_hat(w):
    w = np.asarray(w, dtype=float).reshape(3)
    return np.array([
        [0.0, -w[2],  w[1]],
        [w[2],  0.0, -w[0]],
        [-w[1], w[0], 0.0],
    ])


def exp2C_vee(W):
    return np.array([W[2, 1], W[0, 2], W[1, 0]], dtype=float)


def exp2C_so3_exp(phi):
    phi = np.asarray(phi, dtype=float).reshape(3)
    theta = np.linalg.norm(phi)

    if theta < 1e-12:
        return np.eye(3) + exp2C_hat(phi)

    K = exp2C_hat(phi / theta)
    return (
        np.eye(3)
        + np.sin(theta) * K
        + (1.0 - np.cos(theta)) * (K @ K)
    )


def exp2C_so3_log(R):
    R = np.asarray(R, dtype=float).reshape(3, 3)
    cos_theta = (np.trace(R) - 1.0) / 2.0
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    theta = np.arccos(cos_theta)

    if theta < 1e-12:
        return exp2C_vee(0.5 * (R - R.T))

    return exp2C_vee(theta / (2.0 * np.sin(theta)) * (R - R.T))



def exp2C_Rx(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([
        [1.0, 0.0, 0.0],
        [0.0, c, -s],
        [0.0, s,  c],
    ])

def exp2C_Ry(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([
        [ c, 0.0,  s],
        [0.0, 1.0, 0.0],
        [-s, 0.0,  c],
    ])


def exp2C_Rz(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([
        [ c, -s, 0.0],
        [ s,  c, 0.0],
        [0.0, 0.0, 1.0],
    ])

In [28]:
# ============================================================
# Experiment 2C: part A 
# Large initial error with falsely high confidence
# All variables are prefixed by exp2C_.
# ============================================================

import numpy as np
import json
import time
import inspect
from dataclasses import replace
import matplotlib.pyplot as plt


# ============================================================
# 0. Rerun control
# ============================================================

# Only run the two cases used for the main Experiment 2C result

EXP2C_CASES = [
    {
        "case": "medium",
        "beta": 1.5,
        "c_kappa": 1.0,
        "c_Lambda": 1.0,
    },
    {
        "case": "false_confidence",
        "beta": 2.0,
        "c_kappa": 2.0,
        "c_Lambda": 2.0,
    },
]


FORCE_RERUN_EXP2C = True

# Multi-seed setting
EXP2C_SEEDS = [42, 43, 44, 45, 46]
#EXP2C_SEEDS = [42]

# To save memory, keep only records by default.
# Set True only if you need to inspect data/res1/res2 later.
EXP2C_STORE_RAW_OUTPUTS = False

if FORCE_RERUN_EXP2C:
    exp2C_records = []
    exp2C_raw_outputs = []
    print("[Reset] exp2C_records and exp2C_raw_outputs have been cleared.")
else:
    if "exp2C_records" not in globals():
        exp2C_records = []
    if "exp2C_raw_outputs" not in globals():
        exp2C_raw_outputs = []


# ============================================================
# 1. Required function checks
# ============================================================

exp2C_required_functions = [
    "make_synthetic_implicit_batch_data",
    "make_nested_subset_data",
    "algorithm1_single_pass_local_mfg_posterior_update",
    "algorithm2_safeguarded_multi_pass_mfg_posterior_refinement",
    "recover_pose_from_theta",
    "rotation_geodesic_error",
]

exp2C_missing = [name for name in exp2C_required_functions if name not in globals()]

if len(exp2C_missing) > 0:
    raise NameError(
        "The following required functions are not defined:\n"
        + "\n".join(f"  - {name}" for name in exp2C_missing)
        + "\nPlease run the corresponding definition cells first."
    )


# ============================================================
# 2. Utilities
# ============================================================

def exp2C_make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): exp2C_make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [exp2C_make_json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [exp2C_make_json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, float):
        if np.isfinite(obj):
            return float(obj)
        return str(obj)
    if isinstance(obj, (int, str, bool)) or obj is None:
        return obj
    return str(obj)


def exp2C_translation_error(X_est, X_true):
    return float(np.linalg.norm(np.asarray(X_est.p) - np.asarray(X_true.p)))


def exp2C_rotation_error(X_est, X_true):
    return float(rotation_geodesic_error(X_est.R, X_true.R))


def exp2C_pose_error_summary(X_est, X_true):
    return exp2C_rotation_error(X_est, X_true), exp2C_translation_error(X_est, X_true)


def exp2C_relative_reduction(before, after):
    """
    Relative reduction in percent:
        100 * (before - after) / before.
    Positive means Algorithm 2 improves over Algorithm 1.
    """
    before = float(before)
    after = float(after)

    if not np.isfinite(before) or not np.isfinite(after):
        return np.nan
    if abs(before) < 1e-12:
        return np.nan

    return 100.0 * (before - after) / before


def exp2C_filter_kwargs(func, kwargs):
    sig = inspect.signature(func)
    allowed = set(sig.parameters.keys())
    return {k: v for k, v in kwargs.items() if k in allowed}


def exp2C_format_value(x, precision=4):
    if x is None:
        return "None"
    if isinstance(x, bool):
        return str(x)
    if isinstance(x, (int, np.integer)):
        return str(int(x))
    if isinstance(x, (float, np.floating)):
        x = float(x)
        if not np.isfinite(x):
            return str(x)
        if abs(x) >= 1e3 or (abs(x) > 0 and abs(x) < 1e-3):
            return f"{x:.{precision}e}"
        return f"{x:.{precision}f}"
    return str(x)


def exp2C_print_records_table(records, columns, title=None, max_width=18, precision=4):
    if title is not None:
        print("\n" + title)

    if records is None or len(records) == 0:
        print("(empty)")
        return

    rows = []
    for rec in records:
        rows.append([
            exp2C_format_value(rec.get(col, None), precision=precision)
            for col in columns
        ])

    widths = []
    for j, col in enumerate(columns):
        width_j = len(str(col))
        for row in rows:
            width_j = max(width_j, len(row[j]))
        widths.append(min(width_j, max_width))

    def crop(s, w):
        s = str(s)
        if len(s) <= w:
            return s
        if w <= 3:
            return s[:w]
        return s[:w - 3] + "..."

    header = " | ".join(
        crop(str(col), widths[j]).ljust(widths[j])
        for j, col in enumerate(columns)
    )
    sep = "-+-".join("-" * widths[j] for j in range(len(columns)))

    print(header)
    print(sep)

    for row in rows:
        print(
            " | ".join(
                crop(row[j], widths[j]).ljust(widths[j])
                for j in range(len(columns))
            )
        )


def exp2C_group_mean_std(records, key, case_name):
    vals = [
        float(r[key])
        for r in records
        if str(r["case"]) == str(case_name) and np.isfinite(float(r[key]))
    ]

    if len(vals) == 0:
        return np.nan, np.nan

    vals = np.asarray(vals, dtype=float)
    mean = float(np.mean(vals))
    std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    return mean, std


# ============================================================
# 3. Experiment 2C fixed protocol
# ============================================================

# Same selected sparse-baseline measurement number as Experiment 1
EXP2C_K = int(K_eff) if "K_eff" in globals() else 10
EXP2C_K_MAX = int(EXP1_K_MAX) if "EXP1_K_MAX" in globals() else 24

# Baseline measurement noise
EXP2C_SIGMA_TAU_0 = 0.10
EXP2C_SIGMA_FORCE_0 = 0.70

# Baseline ZYX right perturbation from the final 3D Experiment-1 setting:
#     R_base = R_z(-6 deg) R_y(4 deg) R_x(2 deg).
EXP2C_BASE_YAW_ERROR_DEG = (
    float(EXP1_INIT_CFG["init_yaw_error_deg"])
    if "EXP1_INIT_CFG" in globals() else -6.0
)
EXP2C_BASE_PITCH_ERROR_DEG = (
    float(EXP1_INIT_CFG["init_pitch_error_deg"])
    if "EXP1_INIT_CFG" in globals() else 4.0
)
EXP2C_BASE_ROLL_ERROR_DEG = (
    float(EXP1_INIT_CFG["init_roll_error_deg"])
    if "EXP1_INIT_CFG" in globals() else 2.0
)

EXP2C_BASE_R_ERROR = (
    exp2C_Rz(np.deg2rad(EXP2C_BASE_YAW_ERROR_DEG))
    @ exp2C_Ry(np.deg2rad(EXP2C_BASE_PITCH_ERROR_DEG))
    @ exp2C_Rx(np.deg2rad(EXP2C_BASE_ROLL_ERROR_DEG))
)
EXP2C_BASE_PHI_ERROR = exp2C_so3_log(EXP2C_BASE_R_ERROR)

EXP2C_BASE_TRANS_ERROR = (
    np.asarray(EXP1_INIT_CFG["init_translation_error"], dtype=float)
    if "EXP1_INIT_CFG" in globals()
    else np.array([-0.005, 0.0035, -0.004], dtype=float)
)

# Baseline prior confidence
EXP2C_KAPPA0_BASE = 60.0
EXP2C_LAMBDA0_BASE_DIAG = np.array([6000.0, 6000.0, 6000.0], dtype=float)

# Initialization and prior-confidence cases
# EXP2C_CASES = [
#     dict(case="mild", beta=1.0, c_kappa=1.0, c_Lambda=1.0),
#     dict(case="medium", beta=1.5, c_kappa=1.0, c_Lambda=1.0),
#     dict(case="large", beta=2.0, c_kappa=1.0, c_Lambda=1.0),
#     dict(case="false_confidence", beta=2.0, c_kappa=2.0, c_Lambda=2.0),
# ]


# Same data protocol as Experiment 1, except that the initial nominal pose
# and prior confidence are overwritten case by case after generating the
# nested measurement batch.
if "EXP1_DATA_CFG" in globals():
    EXP2C_BASE_DATA_KWARGS = dict(**EXP1_DATA_CFG)
else:
    EXP2C_BASE_DATA_KWARGS = dict(
        geometry_design="sparse_baseline",
        contact_patch_type="small_asymmetric_3d",
        contact_patch_scale=0.010,
        sigma_tau=EXP2C_SIGMA_TAU_0,
        sigma_force=EXP2C_SIGMA_FORCE_0,
        kappa0=EXP2C_KAPPA0_BASE,
        Lambda0_diag=tuple(EXP2C_LAMBDA0_BASE_DIAG),
    )

# Ensure the final 3D initial convention is active during data construction.
# This initial pose will be overwritten below, but keeping it here makes
# diagnostic outputs consistent.
if "EXP1_INIT_CFG" in globals():
    EXP2C_BASE_DATA_KWARGS.update(**EXP1_INIT_CFG)
else:
    EXP2C_BASE_DATA_KWARGS.update(dict(
        init_yaw_error_deg=-6.0,
        init_pitch_error_deg=4.0,
        init_roll_error_deg=2.0,
        init_translation_error=np.array([-0.005, 0.0035, -0.004], dtype=float),
    ))

EXP2C_BASE_DATA_KWARGS.update(dict(
    sigma_tau=EXP2C_SIGMA_TAU_0,
    sigma_force=EXP2C_SIGMA_FORCE_0,
    kappa0=EXP2C_KAPPA0_BASE,
    Lambda0_diag=tuple(EXP2C_LAMBDA0_BASE_DIAG),
    solver_config_override=EXP1_SOLVER_CFG if "EXP1_SOLVER_CFG" in globals() else None,
    verbose=False,
))

# Same Algorithm-2 protocol as the final experiments, but use T_max=20 for
# the main prior/initialization mismatch experiment.
if "EXP1_ALG2_CFG" in globals() and "EXP1_ALG2_HEALTH_CFG" in globals():
    EXP2C_ALG2_KWARGS = dict(
        **EXP1_ALG2_CFG,
        **EXP1_ALG2_HEALTH_CFG,
    )
else:
    EXP2C_ALG2_KWARGS = dict(
        A=(0.25, 0.10, 0.05, 0.02, 0.01, 0.005),
        eps_acc=1e-6,
        eps_R=1e-6,
        eps_p=1e-6,
        eps_info=1e-8,
        eps_g=1e-8,
        lambda_fb=1e-4,
        A_rot_reduction_factor=0.25,
        require_all_inner_converged=True,
        require_psd_WAA=False,
        max_cond_WAA=1e8,
        min_min_eig_WAA=-1e-8,
        max_J_phi_norm=1e8,
        max_J_v_norm=1e8,
        max_eq_res_norm=1e-5,
        max_eq_step_norm=1e-4,
        min_eig_Lambda_post=1e-10,
        max_cond_Lambda_post=1e10,
        max_norm_H=1e12,
        max_norm_M_post=1e12,
        max_norm_a_post=1e10,
        max_norm_F_post=1e10,
        max_mode_nu_norm=1e4,
        rho_rel_tol=5e-4,
        rho_abs_tol=1e-4,
        plateau_patience=3,
        min_outer=4,
    )

EXP2C_ALG2_KWARGS.update(dict(
    T_max=20,
))


# ============================================================
# 4. Algorithm wrappers
# ============================================================

def exp2C_run_alg1(data):
    """
    Run Algorithm 1 using the same settings as Table IV.
    Measurement noise is correctly specified and equal to the baseline Sigma_0.
    """
    return algorithm1_single_pass_local_mfg_posterior_update(
        Theta_prior=data.Theta_0,
        X_B_bar=data.X_B_bar_0,
        U=data.U,
        Y=data.Y,
        field_params=data.field_params,
        stiffness_params=data.stiffness_params,
        ctrl_params=data.ctrl_params,
        Sigma_w=data.Sigma_w,
        contact_points_A_shared=data.contact_points_A,
        solver_config=data.solver_config,
        chi_A_0=None,
        use_psd_mf_curvature=False,
    )


def exp2C_run_alg2(data):
    """
    Run Algorithm 2 using the same settings as Table IV.
    """
    alg2_kwargs = exp2C_filter_kwargs(
        algorithm2_safeguarded_multi_pass_mfg_posterior_refinement,
        EXP2C_ALG2_KWARGS,
    )

    return algorithm2_safeguarded_multi_pass_mfg_posterior_refinement(
        X_B_bar_0=data.X_B_bar_0,
        Theta_0=data.Theta_0,
        U=data.U,
        Y=data.Y,
        field_params=data.field_params,
        stiffness_params=data.stiffness_params,
        ctrl_params=data.ctrl_params,
        Sigma_w=data.Sigma_w,
        contact_points_A_shared=data.contact_points_A,
        contact_points_A_per_sample=None,
        solver_config=data.solver_config,
        chi_A_0=None,
        **alg2_kwargs,
    )


# ============================================================
# 5. One trial of Experiment 2C
# ============================================================

def exp2C_run_one_trial(case_cfg, seed):
    """
    Run one initialization/prior-confidence mismatch trial.

    Initial perturbation:
        phi_error   = beta * phi_base,
        R_error     = Exp(phi_error),
        trans_error = beta * [-0.006, 0.004, -0.005].

    Prior confidence:
        kappa0      = c_kappa * 60,
        Lambda0     = c_Lambda * diag(6000, 6000, 6000).

    The measurement noise is fixed at the baseline level:
        Sigma_data = Sigma_model = Sigma_0.
    """

    case_name = str(case_cfg.get("case", case_cfg.get("case_name", "unknown")))
    beta = float(case_cfg["beta"])
    c_kappa = float(case_cfg["c_kappa"])
    c_Lambda = float(case_cfg["c_Lambda"])
    seed = int(seed)

    # --------------------------------------------------------
    # Lie algebra scaled initial perturbation
    # --------------------------------------------------------
    init_phi_error = beta * EXP2C_BASE_PHI_ERROR
    init_R_error = exp2C_so3_exp(init_phi_error)
    init_translation_error = beta * EXP2C_BASE_TRANS_ERROR

    # --------------------------------------------------------
    # Prior confidence scaling
    # --------------------------------------------------------
    kappa0 = c_kappa * EXP2C_KAPPA0_BASE
    Lambda0_diag = c_Lambda * np.asarray(EXP2C_LAMBDA0_BASE_DIAG, dtype=float)

    # Robustly record baseline noise values
    sigma_tau_value = float(
        EXP2C_BASE_DATA_KWARGS.get(
            "sigma_tau",
            globals().get("EXP2C_SIGMA_TAU_0", np.nan),
        )
    )
    sigma_force_value = float(
        EXP2C_BASE_DATA_KWARGS.get(
            "sigma_force",
            globals().get("EXP2C_SIGMA_FORCE_0", np.nan),
        )
    )

    print("\n" + "=" * 92)
    print(f"Experiment 2C trial: case={case_name}, seed={seed}")
    print("=" * 92)
    print(f"beta                  = {beta:.3f}")
    print(f"c_kappa               = {c_kappa:.3f}")
    print(f"c_Lambda              = {c_Lambda:.3f}")
    print(f"||phi_base||          = {np.linalg.norm(EXP2C_BASE_PHI_ERROR):.6e}")
    print(f"||init_phi_error||    = {np.linalg.norm(init_phi_error):.6e}")
    print(f"init_phi_error        = {init_phi_error}")
    print(f"init_translation_err  = {init_translation_error}")
    print(f"kappa0                = {kappa0:.6f}")
    print(f"Lambda0_diag          = {Lambda0_diag}")
    print(f"sigma_tau             = {sigma_tau_value:.6f}")
    print(f"sigma_force           = {sigma_force_value:.6f}")

    t0 = time.time()

    # --------------------------------------------------------
    # Generate the full nested batch, take K_eff, and then overwrite
    # the initial nominal pose and prior confidence for this case.
    # --------------------------------------------------------
    print("[1/3] Generating synthetic data ...", flush=True)
    t_data = time.time()

    data_full = make_synthetic_implicit_batch_data(
        num_meas=EXP2C_K_MAX,
        random_seed=seed,
        **EXP2C_BASE_DATA_KWARGS,
    )
    data = make_nested_subset_data(data_full, EXP2C_K)

    X_B_bar_0 = Pose(
        R=data.X_B_true.R @ init_R_error,
        p=data.X_B_true.p + init_translation_error,
    )

    Theta_0 = MFGParameters(
        F=kappa0 * X_B_bar_0.R,
        mu=X_B_bar_0.p.copy(),
        Lambda=np.diag(np.asarray(Lambda0_diag, dtype=float)),
        Gamma=np.zeros((3, 3), dtype=float),
    )

    data = replace(
        data,
        X_B_bar_0=X_B_bar_0,
        Theta_0=Theta_0,
    )

    print(
        f"[1/3] Data generated in {time.time() - t_data:.1f} s | "
        f"inner solves = {sum(r.converged for r in data.generation_results)}/{len(data.generation_results)}",
        flush=True,
    )

    # --------------------------------------------------------
    # Algorithm 1
    # --------------------------------------------------------
    print("[2/3] Running Algorithm 1 ...", flush=True)
    t1 = time.time()
    res1 = exp2C_run_alg1(data)
    X_alg1 = recover_pose_from_theta(res1.Theta_post)
    print(
        f"[2/3] Algorithm 1 done in {time.time() - t1:.1f} s | "
        f"rho = {res1.rho:.6e}, s_rot = {res1.s_rot:.6e}",
        flush=True,
    )

    # --------------------------------------------------------
    # Algorithm 2
    # --------------------------------------------------------
    print("[3/3] Running Algorithm 2 ...", flush=True)
    t2 = time.time()
    res2 = exp2C_run_alg2(data)
    print(
        f"[3/3] Algorithm 2 done in {time.time() - t2:.1f} s | "
        f"rho_star = {res2.rho_star:.6e}, "
        f"n_outer = {res2.n_outer_iterations}, "
        f"stop = {res2.stop_reason}",
        flush=True,
    )

    # --------------------------------------------------------
    # Pose errors
    # --------------------------------------------------------
    init_eR, init_ep = exp2C_pose_error_summary(data.X_B_bar_0, data.X_B_true)
    alg1_eR, alg1_ep = exp2C_pose_error_summary(X_alg1, data.X_B_true)
    alg2_eR, alg2_ep = exp2C_pose_error_summary(res2.X_B_star, data.X_B_true)

    rho_alg1 = float(getattr(res1, "rho", np.nan))
    rho_alg2 = float(getattr(res2, "rho_star", np.nan))

    s_rot_alg1 = float(getattr(res1, "s_rot", np.nan))
    s_rot_alg2 = float(getattr(res2, "s_rot_star", np.nan))

    rot_improv_pct = exp2C_relative_reduction(alg1_eR, alg2_eR)
    trans_improv_pct = exp2C_relative_reduction(alg1_ep, alg2_ep)
    rho_improv_pct = exp2C_relative_reduction(rho_alg1, rho_alg2)

    init_to_alg2_rot_reduction = exp2C_relative_reduction(init_eR, alg2_eR)
    init_to_alg2_trans_reduction = exp2C_relative_reduction(init_ep, alg2_ep)

    basic_success = bool(
        np.isfinite(alg2_eR)
        and np.isfinite(alg2_ep)
        and np.isfinite(rho_alg2)
    )

    alg2_better_than_alg1 = bool(
        alg2_eR <= alg1_eR
        and alg2_ep <= alg1_ep
        and rho_alg2 <= rho_alg1
    )

    alg2_better_than_initial = bool(
        alg2_eR <= init_eR
        and alg2_ep <= init_ep
    )

    s_rot_gain_ratio = (
        float(s_rot_alg2 / s_rot_alg1)
        if np.isfinite(s_rot_alg1) and abs(s_rot_alg1) > 1e-12
        else np.nan
    )

    # --------------------------------------------------------
    # Record
    # --------------------------------------------------------
    record = {
        "case": case_name,
        "seed": seed,
        "K": int(EXP2C_K),

        "beta": float(beta),
        "c_kappa": float(c_kappa),
        "c_Lambda": float(c_Lambda),

        # Lie algebra initialization records
        "init_phi_norm": float(np.linalg.norm(init_phi_error)),
        "base_phi_norm": float(np.linalg.norm(EXP2C_BASE_PHI_ERROR)),
        "init_phi_error": init_phi_error.tolist(),
        "init_translation_error": init_translation_error.tolist(),
        "init_trans_scale": float(beta),

        # Prior and noise records
        "kappa0": float(kappa0),
        "Lambda0_diag": Lambda0_diag.tolist(),
        "Lambda0_scale": float(c_Lambda),
        "sigma_tau": sigma_tau_value,
        "sigma_force": sigma_force_value,

        # Initial errors
        "init_eR": float(init_eR),
        "init_ep": float(init_ep),

        # Algorithm 1
        "alg1_eR": float(alg1_eR),
        "alg1_ep": float(alg1_ep),
        "rho_alg1": float(rho_alg1),
        "s_rot_alg1": float(s_rot_alg1),

        # Algorithm 2
        "alg2_eR": float(alg2_eR),
        "alg2_ep": float(alg2_ep),
        "rho_alg2": float(rho_alg2),
        "s_rot_alg2": float(s_rot_alg2),

        # Relative improvements
        "rot_improv_pct": float(rot_improv_pct),
        "trans_improv_pct": float(trans_improv_pct),
        "rho_improv_pct": float(rho_improv_pct),

        "init_to_alg2_rot_reduction": float(init_to_alg2_rot_reduction),
        "init_to_alg2_trans_reduction": float(init_to_alg2_trans_reduction),

        "s_rot_gain_ratio": s_rot_gain_ratio,

        # Algorithm 2 status
        "alg2_n_iter": int(getattr(res2, "n_outer_iterations", -1)),
        "alg2_converged": bool(getattr(res2, "converged", False)),
        "alg2_stop_reason": str(getattr(res2, "stop_reason", None)),

        # Success flags
        "basic_success": basic_success,
        "alg2_better_than_alg1": alg2_better_than_alg1,
        "alg2_better_than_initial": alg2_better_than_initial,

        "elapsed_sec": float(time.time() - t0),
    }

    # --------------------------------------------------------
    # Print summary
    # --------------------------------------------------------
    print("\nTrial summary")
    print("-" * 80)
    print(f"Initial rotation error       : {init_eR:.6e}")
    print(f"Algorithm 1 rotation error   : {alg1_eR:.6e}")
    print(f"Algorithm 2 rotation error   : {alg2_eR:.6e}")
    print(f"Alg2 vs Alg1 rotation gain   : {rot_improv_pct:.2f}%")
    print(f"Alg2 vs init rotation gain   : {init_to_alg2_rot_reduction:.2f}%")
    print(f"Initial translation error    : {init_ep:.6e}")
    print(f"Algorithm 1 translation error: {alg1_ep:.6e}")
    print(f"Algorithm 2 translation error: {alg2_ep:.6e}")
    print(f"Alg2 vs Alg1 trans gain      : {trans_improv_pct:.2f}%")
    print(f"Alg2 vs init trans gain      : {init_to_alg2_trans_reduction:.2f}%")
    print(f"rho Algorithm 1              : {rho_alg1:.6e}")
    print(f"rho Algorithm 2              : {rho_alg2:.6e}")
    print(f"rho improvement              : {rho_improv_pct:.2f}%")
    print(f"s_rot Algorithm 1            : {s_rot_alg1:.6e}")
    print(f"s_rot Algorithm 2            : {s_rot_alg2:.6e}")
    print(f"s_rot gain ratio             : {s_rot_gain_ratio:.3f}")
    print(f"Algorithm 2 stop reason      : {record['alg2_stop_reason']}")
    print(f"Algorithm 2 better than Alg1 : {alg2_better_than_alg1}")
    print(f"Algorithm 2 better than init : {alg2_better_than_initial}")
    print(f"[Done] Total trial time      : {time.time() - t0:.1f} s")
    print("-" * 80)

    output = {
        "data": data,
        "res1": res1,
        "res2": res2,
        "record": record,
    }

    return output


# ============================================================


[Reset] exp2C_records and exp2C_raw_outputs have been cleared.


In [29]:
# ============================================================
# Experiment 2C smoke test:
# Check ZYX baseline perturbation and case-wise prior overwrite
# ============================================================

import numpy as np

print("=" * 80)
print("Experiment 2C smoke test: final ZYX baseline perturbation")
print("=" * 80)
print("base yaw/pitch/roll (deg):",
      EXP2C_BASE_YAW_ERROR_DEG,
      EXP2C_BASE_PITCH_ERROR_DEG,
      EXP2C_BASE_ROLL_ERROR_DEG)
print("phi_base =", EXP2C_BASE_PHI_ERROR)
print("||phi_base|| =", np.linalg.norm(EXP2C_BASE_PHI_ERROR))
print("v_base =", EXP2C_BASE_TRANS_ERROR)
print("||v_base|| =", np.linalg.norm(EXP2C_BASE_TRANS_ERROR))
print("K_eff =", EXP2C_K, "| K_max =", EXP2C_K_MAX)
print("Alg2 T_max =", EXP2C_ALG2_KWARGS.get("T_max"))

# Use the first seed and compare beta=1 and beta=2 construction.
seed = EXP2C_SEEDS[0]

def make_exp2C_test_data(beta, c_kappa=1.0, c_Lambda=1.0):
    init_phi_error = beta * EXP2C_BASE_PHI_ERROR
    init_R_error = exp2C_so3_exp(init_phi_error)
    init_translation_error = beta * EXP2C_BASE_TRANS_ERROR

    kappa0 = c_kappa * EXP2C_KAPPA0_BASE
    Lambda0_diag = c_Lambda * np.asarray(EXP2C_LAMBDA0_BASE_DIAG, dtype=float)

    data_full = make_synthetic_implicit_batch_data(
        num_meas=EXP2C_K_MAX,
        random_seed=seed,
        **EXP2C_BASE_DATA_KWARGS,
    )
    data = make_nested_subset_data(data_full, EXP2C_K)

    X_B_bar_0 = Pose(
        R=data.X_B_true.R @ init_R_error,
        p=data.X_B_true.p + init_translation_error,
    )
    Theta_0 = MFGParameters(
        F=kappa0 * X_B_bar_0.R,
        mu=X_B_bar_0.p.copy(),
        Lambda=np.diag(np.asarray(Lambda0_diag, dtype=float)),
        Gamma=np.zeros((3, 3), dtype=float),
    )

    return replace(data, X_B_bar_0=X_B_bar_0, Theta_0=Theta_0)

print("\nGenerating beta=1.0 test data...")
data_beta1 = make_exp2C_test_data(beta=1.0)

print("Generating beta=2.0 test data...")
data_beta2 = make_exp2C_test_data(beta=2.0)

Y_clean_diff = np.max(np.abs(data_beta1.Y_clean - data_beta2.Y_clean))
Y_diff = np.max(np.abs(data_beta1.Y - data_beta2.Y))

U_pos_diff = max(
    np.linalg.norm(data_beta1.U[k].p - data_beta2.U[k].p)
    for k in range(len(data_beta1.U))
)
U_rot_diff = max(
    rotation_geodesic_error(data_beta1.U[k].R, data_beta2.U[k].R)
    for k in range(len(data_beta1.U))
)

init_R_err_beta1 = rotation_geodesic_error(data_beta1.X_B_bar_0.R, data_beta1.X_B_true.R)
init_R_err_beta2 = rotation_geodesic_error(data_beta2.X_B_bar_0.R, data_beta2.X_B_true.R)

init_p_err_beta1 = exp2C_translation_error(data_beta1.X_B_bar_0, data_beta1.X_B_true)
init_p_err_beta2 = exp2C_translation_error(data_beta2.X_B_bar_0, data_beta2.X_B_true)

print("\n" + "=" * 80)
print("Experiment 2C data-generation consistency check")
print("=" * 80)
print(f"max |Y_clean(beta=1) - Y_clean(beta=2)| : {Y_clean_diff:.6e}")
print(f"max |Y(beta=1)       - Y(beta=2)|       : {Y_diff:.6e}")
print(f"max U position difference              : {U_pos_diff:.6e}")
print(f"max U rotation difference              : {U_rot_diff:.6e}")
print("-" * 80)
print(f"initial rotation error beta=1           : {init_R_err_beta1:.6e}")
print(f"initial rotation error beta=2           : {init_R_err_beta2:.6e}")
print(f"ratio rotation error beta2/beta1        : {init_R_err_beta2/init_R_err_beta1:.6f}")
print(f"initial translation error beta=1        : {init_p_err_beta1:.6e}")
print(f"initial translation error beta=2        : {init_p_err_beta2:.6e}")
print(f"ratio translation error beta2/beta1     : {init_p_err_beta2/init_p_err_beta1:.6f}")
print("=" * 80)

assert Y_clean_diff < 1e-12, "Y_clean changed across beta. This should not happen."
assert Y_diff < 1e-12, "Noisy Y changed across beta. This should not happen for the same seed."
assert U_pos_diff < 1e-12, "Control positions U changed across beta."
assert U_rot_diff < 1e-12, "Control rotations U changed across beta."

print("\n[PASS] Experiment 2C data generation is consistent.")
print("Only the initial nominal pose and prior parameters change across beta.")


Experiment 2C smoke test: final ZYX baseline perturbation
base yaw/pitch/roll (deg): -6.0 4.0 2.0
phi_base = [ 0.03851595  0.06791602 -0.10588505]
||phi_base|| = 0.1315587660703086
v_base = [-0.005   0.0035 -0.004 ]
||v_base|| = 0.007297259759663213
K_eff = 10 | K_max = 24
Alg2 T_max = 20

Generating beta=1.0 test data...
Generating beta=2.0 test data...

Experiment 2C data-generation consistency check
max |Y_clean(beta=1) - Y_clean(beta=2)| : 0.000000e+00
max |Y(beta=1)       - Y(beta=2)|       : 0.000000e+00
max U position difference              : 0.000000e+00
max U rotation difference              : 2.358056e-16
--------------------------------------------------------------------------------
initial rotation error beta=1           : 1.315588e-01
initial rotation error beta=2           : 2.631175e-01
ratio rotation error beta2/beta1        : 2.000000
initial translation error beta=1        : 7.297260e-03
initial translation error beta=2        : 1.459452e-02
ratio translation error 

In [ ]:
# ============================================================
# 6. Run Experiment 2C
# ============================================================

existing_pairs = set((str(r["case"]), int(r["seed"])) for r in exp2C_records)

for case_cfg in EXP2C_CASES:
    case_name = str(case_cfg["case"])

    for seed in EXP2C_SEEDS:
        pair = (case_name, int(seed))

        if (not FORCE_RERUN_EXP2C) and pair in existing_pairs:
            print(f"[Skip] case={case_name}, seed={seed} already completed.")
            continue

        out = exp2C_run_one_trial(case_cfg=case_cfg, seed=seed)

        if EXP2C_STORE_RAW_OUTPUTS:
            exp2C_raw_outputs.append(out)

        exp2C_records.append(out["record"])

        with open("exp2C_initialization_prior_mismatch_records.json", "w") as f:
            json.dump(exp2C_make_json_safe(exp2C_records), f, indent=2)

        print("[Saved] Partial exp2C_initialization_prior_mismatch_records.json")


# ============================================================
# 7. Print trial-level table
# ============================================================

exp2C_print_records_table(
    exp2C_records,
    [
        "case",
        "seed",
        "beta",
        "c_kappa",
        "c_Lambda",
        "init_eR",
        "alg1_eR",
        "alg2_eR",
        "rot_improv_pct",
        "init_ep",
        "alg1_ep",
        "alg2_ep",
        "trans_improv_pct",
        "rho_alg1",
        "rho_alg2",
        "rho_improv_pct",
        "alg2_stop_reason",
    ],
    title="Experiment 2C trial-level results: initialization and prior-confidence mismatch",
    max_width=18,
    precision=4,
)


# ============================================================
# 8. Aggregate table by case
# ============================================================

exp2C_summary_records = []

for case_cfg in EXP2C_CASES:
    case_name = str(case_cfg["case"])

    init_eR_mean, init_eR_std = exp2C_group_mean_std(exp2C_records, "init_eR", case_name)
    alg1_eR_mean, alg1_eR_std = exp2C_group_mean_std(exp2C_records, "alg1_eR", case_name)
    alg2_eR_mean, alg2_eR_std = exp2C_group_mean_std(exp2C_records, "alg2_eR", case_name)

    init_ep_mean, init_ep_std = exp2C_group_mean_std(exp2C_records, "init_ep", case_name)
    alg1_ep_mean, alg1_ep_std = exp2C_group_mean_std(exp2C_records, "alg1_ep", case_name)
    alg2_ep_mean, alg2_ep_std = exp2C_group_mean_std(exp2C_records, "alg2_ep", case_name)

    rho1_mean, rho1_std = exp2C_group_mean_std(exp2C_records, "rho_alg1", case_name)
    rho2_mean, rho2_std = exp2C_group_mean_std(exp2C_records, "rho_alg2", case_name)

    rot_imp_mean, rot_imp_std = exp2C_group_mean_std(exp2C_records, "rot_improv_pct", case_name)
    trans_imp_mean, trans_imp_std = exp2C_group_mean_std(exp2C_records, "trans_improv_pct", case_name)
    rho_imp_mean, rho_imp_std = exp2C_group_mean_std(exp2C_records, "rho_improv_pct", case_name)

    init_rot_red_mean, init_rot_red_std = exp2C_group_mean_std(
        exp2C_records,
        "init_to_alg2_rot_reduction",
        case_name,
    )

    init_trans_red_mean, init_trans_red_std = exp2C_group_mean_std(
        exp2C_records,
        "init_to_alg2_trans_reduction",
        case_name,
    )

    srot1_mean, srot1_std = exp2C_group_mean_std(exp2C_records, "s_rot_alg1", case_name)
    srot2_mean, srot2_std = exp2C_group_mean_std(exp2C_records, "s_rot_alg2", case_name)
    srot_gain_mean, srot_gain_std = exp2C_group_mean_std(exp2C_records, "s_rot_gain_ratio", case_name)

    subset = [
        r for r in exp2C_records
        if str(r["case"]) == case_name
    ]

    n_trials = len(subset)

    success_rate = (
        100.0 * np.mean([r["basic_success"] for r in subset])
        if n_trials > 0 else np.nan
    )

    better_rate_alg1 = (
        100.0 * np.mean([r["alg2_better_than_alg1"] for r in subset])
        if n_trials > 0 else np.nan
    )

    better_rate_init = (
        100.0 * np.mean([r["alg2_better_than_initial"] for r in subset])
        if n_trials > 0 else np.nan
    )

    exp2C_summary_records.append({
        "case": case_name,
        "n": n_trials,

        "beta": float(case_cfg["beta"]),
        "c_kappa": float(case_cfg["c_kappa"]),
        "c_Lambda": float(case_cfg["c_Lambda"]),

        "init_eR_mean": init_eR_mean,
        "init_eR_std": init_eR_std,
        "Alg1_eR_mean": alg1_eR_mean,
        "Alg1_eR_std": alg1_eR_std,
        "Alg2_eR_mean": alg2_eR_mean,
        "Alg2_eR_std": alg2_eR_std,

        "init_ep_mean": init_ep_mean,
        "init_ep_std": init_ep_std,
        "Alg1_ep_mean": alg1_ep_mean,
        "Alg1_ep_std": alg1_ep_std,
        "Alg2_ep_mean": alg2_ep_mean,
        "Alg2_ep_std": alg2_ep_std,

        "rho1_mean": rho1_mean,
        "rho1_std": rho1_std,
        "rho2_mean": rho2_mean,
        "rho2_std": rho2_std,

        "rot_imp_mean": rot_imp_mean,
        "rot_imp_std": rot_imp_std,
        "trans_imp_mean": trans_imp_mean,
        "trans_imp_std": trans_imp_std,
        "rho_imp_mean": rho_imp_mean,
        "rho_imp_std": rho_imp_std,

        "init_rot_red_mean": init_rot_red_mean,
        "init_trans_red_mean": init_trans_red_mean,

        "srot1_mean": srot1_mean,
        "srot2_mean": srot2_mean,
        "srot_gain_mean": srot_gain_mean,

        "success_rate": success_rate,
        "better_rate_alg1": better_rate_alg1,
        "better_rate_init": better_rate_init,
    })


exp2C_print_records_table(
    exp2C_summary_records,
    [
        "case",
        "n",
        "beta",
        "c_kappa",
        "c_Lambda",
        "init_eR_mean",
        "Alg1_eR_mean",
        "Alg2_eR_mean",
        "rot_imp_mean",
        "init_ep_mean",
        "Alg1_ep_mean",
        "Alg2_ep_mean",
        "trans_imp_mean",
        "rho1_mean",
        "rho2_mean",
        "rho_imp_mean",
        "srot_gain_mean",
        "better_rate_alg1",
        "better_rate_init",
    ],
    title="Experiment 2C summary by initialization/prior-confidence case",
    max_width=18,
    precision=4,
)


with open("exp2C_initialization_prior_mismatch_summary.json", "w") as f:
    json.dump(exp2C_make_json_safe(exp2C_summary_records), f, indent=2)

print("\n[Saved] exp2C_initialization_prior_mismatch_records.json")
print("[Saved] exp2C_initialization_prior_mismatch_summary.json")


# ============================================================
# 9. Optional plotting
# ============================================================

SAVE_EXP2C_FIGURES = True

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "stix"

case_labels = [r["case"].replace("_", "\n") for r in exp2C_summary_records]
x = np.arange(len(case_labels))

init_eR_mean = np.array([r["init_eR_mean"] for r in exp2C_summary_records], dtype=float)
init_eR_std = np.array([r["init_eR_std"] for r in exp2C_summary_records], dtype=float)
alg1_eR_mean = np.array([r["Alg1_eR_mean"] for r in exp2C_summary_records], dtype=float)
alg1_eR_std = np.array([r["Alg1_eR_std"] for r in exp2C_summary_records], dtype=float)
alg2_eR_mean = np.array([r["Alg2_eR_mean"] for r in exp2C_summary_records], dtype=float)
alg2_eR_std = np.array([r["Alg2_eR_std"] for r in exp2C_summary_records], dtype=float)

init_ep_mean = np.array([r["init_ep_mean"] for r in exp2C_summary_records], dtype=float)
init_ep_std = np.array([r["init_ep_std"] for r in exp2C_summary_records], dtype=float)
alg1_ep_mean = np.array([r["Alg1_ep_mean"] for r in exp2C_summary_records], dtype=float)
alg1_ep_std = np.array([r["Alg1_ep_std"] for r in exp2C_summary_records], dtype=float)
alg2_ep_mean = np.array([r["Alg2_ep_mean"] for r in exp2C_summary_records], dtype=float)
alg2_ep_std = np.array([r["Alg2_ep_std"] for r in exp2C_summary_records], dtype=float)

rho1_mean = np.array([r["rho1_mean"] for r in exp2C_summary_records], dtype=float)
rho1_std = np.array([r["rho1_std"] for r in exp2C_summary_records], dtype=float)
rho2_mean = np.array([r["rho2_mean"] for r in exp2C_summary_records], dtype=float)
rho2_std = np.array([r["rho2_std"] for r in exp2C_summary_records], dtype=float)

c_init = "#9e9e9e"
c_alg1 = "#5a5a5a"
c_alg2 = "#1f4e79"

bar_w = 0.24

fig, axes = plt.subplots(1, 3, figsize=(7.16, 3.20))

# Rotation error
ax = axes[0]
ax.bar(
    x - bar_w,
    init_eR_mean,
    width=bar_w,
    yerr=init_eR_std if len(EXP2C_SEEDS) > 1 else None,
    capsize=2.5,
    label="Initial",
    color=c_init,
)
ax.bar(
    x,
    alg1_eR_mean,
    width=bar_w,
    yerr=alg1_eR_std if len(EXP2C_SEEDS) > 1 else None,
    capsize=2.5,
    label="Algorithm 1",
    color=c_alg1,
)
ax.bar(
    x + bar_w,
    alg2_eR_mean,
    width=bar_w,
    yerr=alg2_eR_std if len(EXP2C_SEEDS) > 1 else None,
    capsize=2.5,
    label="Algorithm 2",
    color=c_alg2,
)
ax.set_xticks(x)
ax.set_xticklabels(case_labels, fontsize=7)
ax.set_ylabel(r"Rotation error $e_R$ (rad)")
ax.grid(True, axis="y", alpha=0.25, linestyle="--")
ax.legend(fontsize=6.5, frameon=True)

# Translation error
ax = axes[1]
ax.bar(
    x - bar_w,
    init_ep_mean,
    width=bar_w,
    yerr=init_ep_std if len(EXP2C_SEEDS) > 1 else None,
    capsize=2.5,
    label="Initial",
    color=c_init,
)
ax.bar(
    x,
    alg1_ep_mean,
    width=bar_w,
    yerr=alg1_ep_std if len(EXP2C_SEEDS) > 1 else None,
    capsize=2.5,
    label="Algorithm 1",
    color=c_alg1,
)
ax.bar(
    x + bar_w,
    alg2_ep_mean,
    width=bar_w,
    yerr=alg2_ep_std if len(EXP2C_SEEDS) > 1 else None,
    capsize=2.5,
    label="Algorithm 2",
    color=c_alg2,
)
ax.set_xticks(x)
ax.set_xticklabels(case_labels, fontsize=7)
ax.set_ylabel(r"Translation error $e_p$")
ax.grid(True, axis="y", alpha=0.25, linestyle="--")

# Residual merit
ax = axes[2]
ax.bar(
    x - bar_w / 2,
    rho1_mean,
    width=bar_w,
    yerr=rho1_std if len(EXP2C_SEEDS) > 1 else None,
    capsize=2.5,
    label="Algorithm 1",
    color=c_alg1,
)
ax.bar(
    x + bar_w / 2,
    rho2_mean,
    width=bar_w,
    yerr=rho2_std if len(EXP2C_SEEDS) > 1 else None,
    capsize=2.5,
    label="Algorithm 2",
    color=c_alg2,
)
ax.set_xticks(x)
ax.set_xticklabels(case_labels, fontsize=7)
ax.set_ylabel(r"Residual merit $\rho$")
ax.grid(True, axis="y", alpha=0.25, linestyle="--")

fig.suptitle(
    r"Experiment 2C: robustness to initialization and prior-confidence mismatch",
    fontsize=10,
    fontweight="bold",
)

plt.tight_layout(rect=[0, 0, 1, 0.90])

if SAVE_EXP2C_FIGURES:
    fig.savefig("exp2C_initialization_prior_mismatch.pdf", bbox_inches="tight")
    fig.savefig("exp2C_initialization_prior_mismatch.png", dpi=300, bbox_inches="tight")

plt.show()

print("\n[Done] Experiment 2C completed.")


Experiment 2C trial: case=medium, seed=42
beta                  = 1.500
c_kappa               = 1.000
c_Lambda              = 1.000
||phi_base||          = 1.315588e-01
||init_phi_error||    = 1.973381e-01
init_phi_error        = [ 0.05777393  0.10187404 -0.15882758]
init_translation_err  = [-0.0075   0.00525 -0.006  ]
kappa0                = 60.000000
Lambda0_diag          = [6000. 6000. 6000.]
sigma_tau             = 0.100000
sigma_force           = 0.700000
[1/3] Generating synthetic data ...
[1/3] Data generated in 6.7 s | inner solves = 10/10
[2/3] Running Algorithm 1 ...
[2/3] Algorithm 1 done in 2.5 s | rho = 1.936297e+01, s_rot = 9.425410e-01
[3/3] Running Algorithm 2 ...
[3/3] Algorithm 2 done in 11111.0 s | rho_star = 7.522480e+00, n_outer = 20, stop = reached T_max

Trial summary
--------------------------------------------------------------------------------
Initial rotation error       : 1.973381e-01
Algorithm 1 rotation error   : 3.120537e-01
Algorithm 2 rotation error  

In [35]:
import copy

def find_case_cfg(label="false_confidence"):
    candidates = []

    for name, obj in globals().items():
        if name.startswith("__"):
            continue

        # 情况 1: 一个大字典，里面有 false_confidence 这一项
        if isinstance(obj, dict):
            if label in obj:
                candidates.append((f"{name}['{label}']", obj[label]))

            # 情况 2: 这个字典本身就是 case_cfg
            if label in str(obj)[:5000]:
                candidates.append((name, obj))

        # 情况 3: list/tuple 里面存了一组 case_cfg
        if isinstance(obj, (list, tuple)):
            for i, item in enumerate(obj):
                if isinstance(item, dict) and label in str(item)[:5000]:
                    candidates.append((f"{name}[{i}]", item))

    print("Found candidates:")
    for i, (name, cfg) in enumerate(candidates):
        print(f"[{i}] {name}")
        if isinstance(cfg, dict):
            print("    keys:", list(cfg.keys()))

    if len(candidates) == 0:
        raise RuntimeError("Cannot find false_confidence case_cfg.")

    return copy.deepcopy(candidates[0][1])


case_cfg_false_confidence = find_case_cfg("false_confidence")

all_records_new = []

for seed in [45, 46]:
    print("=" * 90)
    print(f"Continue exp2C_run_one_trial: case=false_confidence, seed={seed}")
    print("=" * 90)

    rec = exp2C_run_one_trial(case_cfg_false_confidence, seed)
    all_records_new.append(rec)

print("\n[Done] Continued missing Experiment 2C seeds.")

Found candidates:
[0] EXP2C_CASES[1]
    keys: ['case', 'beta', 'c_kappa', 'c_Lambda']
Continue exp2C_run_one_trial: case=false_confidence, seed=45

Experiment 2C trial: case=false_confidence, seed=45
beta                  = 2.000
c_kappa               = 2.000
c_Lambda              = 2.000
||phi_base||          = 1.315588e-01
||init_phi_error||    = 2.631175e-01
init_phi_error        = [ 0.07703191  0.13583205 -0.2117701 ]
init_translation_err  = [-0.01   0.007 -0.008]
kappa0                = 120.000000
Lambda0_diag          = [12000. 12000. 12000.]
sigma_tau             = 0.100000
sigma_force           = 0.700000
[1/3] Generating synthetic data ...
[1/3] Data generated in 6.5 s | inner solves = 10/10
[2/3] Running Algorithm 1 ...
[2/3] Algorithm 1 done in 2.5 s | rho = 1.766461e+01, s_rot = 2.989145e-01
[3/3] Running Algorithm 2 ...
[3/3] Algorithm 2 done in 698.0 s | rho_star = 6.684605e+00, n_outer = 20, stop = reached T_max

Trial summary
-------------------------------------------